# SynTrustBench v0.4.1 — gate-focused prospective experiment

**Claim under test.** When synthetic clinical data is scored on several dimensions, averaging the scores lets a serious failure in one dimension hide behind good scores in the others. A gate that requires every dimension to pass catches those failures, but only if each dimension is first calibrated against real-versus-real comparisons, so the gate does not also reject real patients.

This **unexecuted** notebook runs the complete campaign: `prepare → smoke → null → calibrate → clean → controls → injections → judges → analysis → report`, then a completion audit. It contains no prewritten results.

**What changed from v0.4 (declared amendment).**
- The gate decides on **fidelity, utility and privacy** only, the dimensions with established real-referenced tests. **Equity and robustness are still computed and reported as diagnostics** and never change the verdict; the v0.4 audit showed their rules reject honest real data, and no published benchmark provides a validated decision rule for them.
- Calibration uses **100 fresh null runs** (50 fit, 50 held-out audit; new seeds, disjoint from v0.4) and a **full-conformal** max-statistic rule with exact marginal level under exchangeability.
- A **prespecified admission rule** decides whether each dimension may gate, tested on the held-out audit.
- Judges: the gate versus mean, geometric (Belgodere et al.), harmonic and native SDMetrics, all on identical evidence. **Primary comparison at matched cost:** every judge is calibrated on the same fit runs to the same honest-data rejection rate, then scored on known-bad tables **and** on held-out honest data. The v0.4 τ* threshold is kept as a secondary analysis.

Scientific authority: [SynTrustBench v0.4.1 one-page spec](sources/SynTrustBench_v041_spec.md). Frozen official v0.3 is installed at commit `3dd0f7ce088994d84f4925ca7e0949d7e0ecd3d8`, never patched, and reported as the uncalibrated five-dimension baseline. Generators: Gaussian Copula, CTGAN, TVAE (SDV 1.37.4) and DP-AIM (ε = 1), three seeds each; optional DPMST and TabDDPM are off by default. Default plans contain 542 full jobs or 69 smoke jobs.

## One configuration cell

Edit only the next cell. Set the raw UCI CSV path, or provide both the frozen split directory and its manifest. Leave `dry_run=True` to inspect the complete job plan. For a small end-to-end campaign set `smoke_mode=True`, `dry_run=False` and keep all stages enabled. For paper work use `smoke_mode=False`; the full smoke preflight then runs separately using development data only.

The dictionary is converted to validated frozen dataclasses. Paths and execution switches do not change the scientific campaign identifier; smoke settings and all scientific parameters do. Dataset and input hashes, numerical versions and implementation hashes are checked separately on resume.

In [ ]:
# Only this cell is intended for editing. A fresh kernel is recommended.
SETTINGS = {'data': {'raw_csv': 'data/diabetic_data.csv',
          'frozen_dir': 'inputs/frozen',
          'frozen_manifest': 'inputs/frozen/split_manifest.json',
          'continuous': ('time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',
                         'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'),
          'categorical': ('race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id',
                          'admission_source_id', 'diag_1', 'max_glu_serum', 'A1Cresult', 'insulin', 'change',
                          'diabetesMed'),
          'target': 'readmitted_30d',
          'protected': ('race', 'gender'),
          'split_seed': 49173,
          'split_fractions': (0.4, 0.4, 0.2)},
 'generators': {'names': ('GaussianCopula', 'CTGAN', 'TVAE'),
                'seeds': (17, 29, 43),
                'epochs': 300,
                'enforce_min_max_values': True,
                'enforce_rounding': True,
                'allow_cuda': False,
                'enable_dpmst': False,
                'enable_tabddpm': False,
                'epsilon': 1.0,
                'delta': None,
                'dp_domains': (('time_in_hospital', 1, 14, 1, 14), ('num_lab_procedures', 0, 200, 10, 1000),
                               ('num_procedures', 0, 20, 1, 100), ('num_medications', 0, 100, 5, 500),
                               ('number_outpatient', 0, 20, 1, 250), ('number_emergency', 0, 20, 1, 250),
                               ('number_inpatient', 0, 20, 1, 250), ('number_diagnoses', 0, 20, 1, 100)),
                               # width-1 bins for zero-inflated counts: width-5 bins decoded uniformly erased the real zero mass
                'dp_categories': (('race', ('Caucasian', 'Asian', 'AfricanAmerican', 'Hispanic', 'Other')),
                                  ('gender', ('Female', 'Male', 'Unknown/Invalid')),
                                  ('age',
                                   ('[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)',
                                    '[70-80)', '[80-90)', '[90-100)')),
                                  ('admission_type_id', ('1', '2', '3', '4', '5', '6', '7', '8', '9')),
                                  ('discharge_disposition_id',
                                   ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
                                    '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27',
                                    '28', '29', '30')),
                                  ('admission_source_id',
                                   ('1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
                                    '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26')),
                                  ('diag_1',
                                   ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14',
                                    '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27',
                                    '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40',
                                    '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53',
                                    '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66',
                                    '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79',
                                    '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92',
                                    '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104',
                                    '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115',
                                    '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126',
                                    '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137',
                                    '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148',
                                    '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159',
                                    '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170',
                                    '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181',
                                    '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192',
                                    '193', '194', '195', '196', '197', '198', '199', '200', '201', '202', '203',
                                    '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214',
                                    '215', '216', '217', '218', '219', '220', '221', '222', '223', '224', '225',
                                    '226', '227', '228', '229', '230', '231', '232', '233', '234', '235', '236',
                                    '237', '238', '239', '240', '241', '242', '243', '244', '245', '246', '247',
                                    '248', '249', '250', '251', '252', '253', '254', '255', '256', '257', '258',
                                    '259', '260', '261', '262', '263', '264', '265', '266', '267', '268', '269',
                                    '270', '271', '272', '273', '274', '275', '276', '277', '278', '279', '280',
                                    '281', '282', '283', '284', '285', '286', '287', '288', '289', '290', '291',
                                    '292', '293', '294', '295', '296', '297', '298', '299', '300', '301', '302',
                                    '303', '304', '305', '306', '307', '308', '309', '310', '311', '312', '313',
                                    '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324',
                                    '325', '326', '327', '328', '329', '330', '331', '332', '333', '334', '335',
                                    '336', '337', '338', '339', '340', '341', '342', '343', '344', '345', '346',
                                    '347', '348', '349', '350', '351', '352', '353', '354', '355', '356', '357',
                                    '358', '359', '360', '361', '362', '363', '364', '365', '366', '367', '368',
                                    '369', '370', '371', '372', '373', '374', '375', '376', '377', '378', '379',
                                    '380', '381', '382', '383', '384', '385', '386', '387', '388', '389', '390',
                                    '391', '392', '393', '394', '395', '396', '397', '398', '399', '400', '401',
                                    '402', '403', '404', '405', '406', '407', '408', '409', '410', '411', '412',
                                    '413', '414', '415', '416', '417', '418', '419', '420', '421', '422', '423',
                                    '424', '425', '426', '427', '428', '429', '430', '431', '432', '433', '434',
                                    '435', '436', '437', '438', '439', '440', '441', '442', '443', '444', '445',
                                    '446', '447', '448', '449', '450', '451', '452', '453', '454', '455', '456',
                                    '457', '458', '459', '460', '461', '462', '463', '464', '465', '466', '467',
                                    '468', '469', '470', '471', '472', '473', '474', '475', '476', '477', '478',
                                    '479', '480', '481', '482', '483', '484', '485', '486', '487', '488', '489',
                                    '490', '491', '492', '493', '494', '495', '496', '497', '498', '499', '500',
                                    '501', '502', '503', '504', '505', '506', '507', '508', '509', '510', '511',
                                    '512', '513', '514', '515', '516', '517', '518', '519', '520', '521', '522',
                                    '523', '524', '525', '526', '527', '528', '529', '530', '531', '532', '533',
                                    '534', '535', '536', '537', '538', '539', '540', '541', '542', '543', '544',
                                    '545', '546', '547', '548', '549', '550', '551', '552', '553', '554', '555',
                                    '556', '557', '558', '559', '560', '561', '562', '563', '564', '565', '566',
                                    '567', '568', '569', '570', '571', '572', '573', '574', '575', '576', '577',
                                    '578', '579', '580', '581', '582', '583', '584', '585', '586', '587', '588',
                                    '589', '590', '591', '592', '593', '594', '595', '596', '597', '598', '599',
                                    '600', '601', '602', '603', '604', '605', '606', '607', '608', '609', '610',
                                    '611', '612', '613', '614', '615', '616', '617', '618', '619', '620', '621',
                                    '622', '623', '624', '625', '626', '627', '628', '629', '630', '631', '632',
                                    '633', '634', '635', '636', '637', '638', '639', '640', '641', '642', '643',
                                    '644', '645', '646', '647', '648', '649', '650', '651', '652', '653', '654',
                                    '655', '656', '657', '658', '659', '660', '661', '662', '663', '664', '665',
                                    '666', '667', '668', '669', '670', '671', '672', '673', '674', '675', '676',
                                    '677', '678', '679', '680', '681', '682', '683', '684', '685', '686', '687',
                                    '688', '689', '690', '691', '692', '693', '694', '695', '696', '697', '698',
                                    '699', '700', '701', '702', '703', '704', '705', '706', '707', '708', '709',
                                    '710', '711', '712', '713', '714', '715', '716', '717', '718', '719', '720',
                                    '721', '722', '723', '724', '725', '726', '727', '728', '729', '730', '731',
                                    '732', '733', '734', '735', '736', '737', '738', '739', '740', '741', '742',
                                    '743', '744', '745', '746', '747', '748', '749', '750', '751', '752', '753',
                                    '754', '755', '756', '757', '758', '759', '760', '761', '762', '763', '764',
                                    '765', '766', '767', '768', '769', '770', '771', '772', '773', '774', '775',
                                    '776', '777', '778', '779', '780', '781', '782', '783', '784', '785', '786',
                                    '787', '788', '789', '790', '791', '792', '793', '794', '795', '796', '797',
                                    '798', '799', '800', '801', '802', '803', '804', '805', '806', '807', '808',
                                    '809', '810', '811', '812', '813', '814', '815', '816', '817', '818', '819',
                                    '820', '821', '822', '823', '824', '825', '826', '827', '828', '829', '830',
                                    '831', '832', '833', '834', '835', '836', '837', '838', '839', '840', '841',
                                    '842', '843', '844', '845', '846', '847', '848', '849', '850', '851', '852',
                                    '853', '854', '855', '856', '857', '858', '859', '860', '861', '862', '863',
                                    '864', '865', '866', '867', '868', '869', '870', '871', '872', '873', '874',
                                    '875', '876', '877', '878', '879', '880', '881', '882', '883', '884', '885',
                                    '886', '887', '888', '889', '890', '891', '892', '893', '894', '895', '896',
                                    '897', '898', '899', '900', '901', '902', '903', '904', '905', '906', '907',
                                    '908', '909', '910', '911', '912', '913', '914', '915', '916', '917', '918',
                                    '919', '920', '921', '922', '923', '924', '925', '926', '927', '928', '929',
                                    '930', '931', '932', '933', '934', '935', '936', '937', '938', '939', '940',
                                    '941', '942', '943', '944', '945', '946', '947', '948', '949', '950', '951',
                                    '952', '953', '954', '955', '956', '957', '958', '959', '960', '961', '962',
                                    '963', '964', '965', '966', '967', '968', '969', '970', '971', '972', '973',
                                    '974', '975', '976', '977', '978', '979', '980', '981', '982', '983', '984',
                                    '985', '986', '987', '988', '989', '990', '991', '992', '993', '994', '995',
                                    '996', '997', '998', '999', 'V00', 'V01', 'V02', 'V03', 'V04', 'V05', 'V06',
                                    'V07', 'V08', 'V09', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17',
                                    'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28',
                                    'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39',
                                    'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50',
                                    'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61',
                                    'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72',
                                    'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83',
                                    'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94',
                                    'V95', 'V96', 'V97', 'V98', 'V99', 'E000', 'E001', 'E002', 'E003', 'E004',
                                    'E005', 'E006', 'E007', 'E008', 'E009', 'E010', 'E011', 'E012', 'E013', 'E014',
                                    'E015', 'E016', 'E017', 'E018', 'E019', 'E020', 'E021', 'E022', 'E023', 'E024',
                                    'E025', 'E026', 'E027', 'E028', 'E029', 'E030', 'E031', 'E032', 'E033', 'E034',
                                    'E035', 'E036', 'E037', 'E038', 'E039', 'E040', 'E041', 'E042', 'E043', 'E044',
                                    'E045', 'E046', 'E047', 'E048', 'E049', 'E050', 'E051', 'E052', 'E053', 'E054',
                                    'E055', 'E056', 'E057', 'E058', 'E059', 'E060', 'E061', 'E062', 'E063', 'E064',
                                    'E065', 'E066', 'E067', 'E068', 'E069', 'E070', 'E071', 'E072', 'E073', 'E074',
                                    'E075', 'E076', 'E077', 'E078', 'E079', 'E080', 'E081', 'E082', 'E083', 'E084',
                                    'E085', 'E086', 'E087', 'E088', 'E089', 'E090', 'E091', 'E092', 'E093', 'E094',
                                    'E095', 'E096', 'E097', 'E098', 'E099', 'E100', 'E101', 'E102', 'E103', 'E104',
                                    'E105', 'E106', 'E107', 'E108', 'E109', 'E110', 'E111', 'E112', 'E113', 'E114',
                                    'E115', 'E116', 'E117', 'E118', 'E119', 'E120', 'E121', 'E122', 'E123', 'E124',
                                    'E125', 'E126', 'E127', 'E128', 'E129', 'E130', 'E131', 'E132', 'E133', 'E134',
                                    'E135', 'E136', 'E137', 'E138', 'E139', 'E140', 'E141', 'E142', 'E143', 'E144',
                                    'E145', 'E146', 'E147', 'E148', 'E149', 'E150', 'E151', 'E152', 'E153', 'E154',
                                    'E155', 'E156', 'E157', 'E158', 'E159', 'E160', 'E161', 'E162', 'E163', 'E164',
                                    'E165', 'E166', 'E167', 'E168', 'E169', 'E170', 'E171', 'E172', 'E173', 'E174',
                                    'E175', 'E176', 'E177', 'E178', 'E179', 'E180', 'E181', 'E182', 'E183', 'E184',
                                    'E185', 'E186', 'E187', 'E188', 'E189', 'E190', 'E191', 'E192', 'E193', 'E194',
                                    'E195', 'E196', 'E197', 'E198', 'E199', 'E200', 'E201', 'E202', 'E203', 'E204',
                                    'E205', 'E206', 'E207', 'E208', 'E209', 'E210', 'E211', 'E212', 'E213', 'E214',
                                    'E215', 'E216', 'E217', 'E218', 'E219', 'E220', 'E221', 'E222', 'E223', 'E224',
                                    'E225', 'E226', 'E227', 'E228', 'E229', 'E230', 'E231', 'E232', 'E233', 'E234',
                                    'E235', 'E236', 'E237', 'E238', 'E239', 'E240', 'E241', 'E242', 'E243', 'E244',
                                    'E245', 'E246', 'E247', 'E248', 'E249', 'E250', 'E251', 'E252', 'E253', 'E254',
                                    'E255', 'E256', 'E257', 'E258', 'E259', 'E260', 'E261', 'E262', 'E263', 'E264',
                                    'E265', 'E266', 'E267', 'E268', 'E269', 'E270', 'E271', 'E272', 'E273', 'E274',
                                    'E275', 'E276', 'E277', 'E278', 'E279', 'E280', 'E281', 'E282', 'E283', 'E284',
                                    'E285', 'E286', 'E287', 'E288', 'E289', 'E290', 'E291', 'E292', 'E293', 'E294',
                                    'E295', 'E296', 'E297', 'E298', 'E299', 'E300', 'E301', 'E302', 'E303', 'E304',
                                    'E305', 'E306', 'E307', 'E308', 'E309', 'E310', 'E311', 'E312', 'E313', 'E314',
                                    'E315', 'E316', 'E317', 'E318', 'E319', 'E320', 'E321', 'E322', 'E323', 'E324',
                                    'E325', 'E326', 'E327', 'E328', 'E329', 'E330', 'E331', 'E332', 'E333', 'E334',
                                    'E335', 'E336', 'E337', 'E338', 'E339', 'E340', 'E341', 'E342', 'E343', 'E344',
                                    'E345', 'E346', 'E347', 'E348', 'E349', 'E350', 'E351', 'E352', 'E353', 'E354',
                                    'E355', 'E356', 'E357', 'E358', 'E359', 'E360', 'E361', 'E362', 'E363', 'E364',
                                    'E365', 'E366', 'E367', 'E368', 'E369', 'E370', 'E371', 'E372', 'E373', 'E374',
                                    'E375', 'E376', 'E377', 'E378', 'E379', 'E380', 'E381', 'E382', 'E383', 'E384',
                                    'E385', 'E386', 'E387', 'E388', 'E389', 'E390', 'E391', 'E392', 'E393', 'E394',
                                    'E395', 'E396', 'E397', 'E398', 'E399', 'E400', 'E401', 'E402', 'E403', 'E404',
                                    'E405', 'E406', 'E407', 'E408', 'E409', 'E410', 'E411', 'E412', 'E413', 'E414',
                                    'E415', 'E416', 'E417', 'E418', 'E419', 'E420', 'E421', 'E422', 'E423', 'E424',
                                    'E425', 'E426', 'E427', 'E428', 'E429', 'E430', 'E431', 'E432', 'E433', 'E434',
                                    'E435', 'E436', 'E437', 'E438', 'E439', 'E440', 'E441', 'E442', 'E443', 'E444',
                                    'E445', 'E446', 'E447', 'E448', 'E449', 'E450', 'E451', 'E452', 'E453', 'E454',
                                    'E455', 'E456', 'E457', 'E458', 'E459', 'E460', 'E461', 'E462', 'E463', 'E464',
                                    'E465', 'E466', 'E467', 'E468', 'E469', 'E470', 'E471', 'E472', 'E473', 'E474',
                                    'E475', 'E476', 'E477', 'E478', 'E479', 'E480', 'E481', 'E482', 'E483', 'E484',
                                    'E485', 'E486', 'E487', 'E488', 'E489', 'E490', 'E491', 'E492', 'E493', 'E494',
                                    'E495', 'E496', 'E497', 'E498', 'E499', 'E500', 'E501', 'E502', 'E503', 'E504',
                                    'E505', 'E506', 'E507', 'E508', 'E509', 'E510', 'E511', 'E512', 'E513', 'E514',
                                    'E515', 'E516', 'E517', 'E518', 'E519', 'E520', 'E521', 'E522', 'E523', 'E524',
                                    'E525', 'E526', 'E527', 'E528', 'E529', 'E530', 'E531', 'E532', 'E533', 'E534',
                                    'E535', 'E536', 'E537', 'E538', 'E539', 'E540', 'E541', 'E542', 'E543', 'E544',
                                    'E545', 'E546', 'E547', 'E548', 'E549', 'E550', 'E551', 'E552', 'E553', 'E554',
                                    'E555', 'E556', 'E557', 'E558', 'E559', 'E560', 'E561', 'E562', 'E563', 'E564',
                                    'E565', 'E566', 'E567', 'E568', 'E569', 'E570', 'E571', 'E572', 'E573', 'E574',
                                    'E575', 'E576', 'E577', 'E578', 'E579', 'E580', 'E581', 'E582', 'E583', 'E584',
                                    'E585', 'E586', 'E587', 'E588', 'E589', 'E590', 'E591', 'E592', 'E593', 'E594',
                                    'E595', 'E596', 'E597', 'E598', 'E599', 'E600', 'E601', 'E602', 'E603', 'E604',
                                    'E605', 'E606', 'E607', 'E608', 'E609', 'E610', 'E611', 'E612', 'E613', 'E614',
                                    'E615', 'E616', 'E617', 'E618', 'E619', 'E620', 'E621', 'E622', 'E623', 'E624',
                                    'E625', 'E626', 'E627', 'E628', 'E629', 'E630', 'E631', 'E632', 'E633', 'E634',
                                    'E635', 'E636', 'E637', 'E638', 'E639', 'E640', 'E641', 'E642', 'E643', 'E644',
                                    'E645', 'E646', 'E647', 'E648', 'E649', 'E650', 'E651', 'E652', 'E653', 'E654',
                                    'E655', 'E656', 'E657', 'E658', 'E659', 'E660', 'E661', 'E662', 'E663', 'E664',
                                    'E665', 'E666', 'E667', 'E668', 'E669', 'E670', 'E671', 'E672', 'E673', 'E674',
                                    'E675', 'E676', 'E677', 'E678', 'E679', 'E680', 'E681', 'E682', 'E683', 'E684',
                                    'E685', 'E686', 'E687', 'E688', 'E689', 'E690', 'E691', 'E692', 'E693', 'E694',
                                    'E695', 'E696', 'E697', 'E698', 'E699', 'E700', 'E701', 'E702', 'E703', 'E704',
                                    'E705', 'E706', 'E707', 'E708', 'E709', 'E710', 'E711', 'E712', 'E713', 'E714',
                                    'E715', 'E716', 'E717', 'E718', 'E719', 'E720', 'E721', 'E722', 'E723', 'E724',
                                    'E725', 'E726', 'E727', 'E728', 'E729', 'E730', 'E731', 'E732', 'E733', 'E734',
                                    'E735', 'E736', 'E737', 'E738', 'E739', 'E740', 'E741', 'E742', 'E743', 'E744',
                                    'E745', 'E746', 'E747', 'E748', 'E749', 'E750', 'E751', 'E752', 'E753', 'E754',
                                    'E755', 'E756', 'E757', 'E758', 'E759', 'E760', 'E761', 'E762', 'E763', 'E764',
                                    'E765', 'E766', 'E767', 'E768', 'E769', 'E770', 'E771', 'E772', 'E773', 'E774',
                                    'E775', 'E776', 'E777', 'E778', 'E779', 'E780', 'E781', 'E782', 'E783', 'E784',
                                    'E785', 'E786', 'E787', 'E788', 'E789', 'E790', 'E791', 'E792', 'E793', 'E794',
                                    'E795', 'E796', 'E797', 'E798', 'E799', 'E800', 'E801', 'E802', 'E803', 'E804',
                                    'E805', 'E806', 'E807', 'E808', 'E809', 'E810', 'E811', 'E812', 'E813', 'E814',
                                    'E815', 'E816', 'E817', 'E818', 'E819', 'E820', 'E821', 'E822', 'E823', 'E824',
                                    'E825', 'E826', 'E827', 'E828', 'E829', 'E830', 'E831', 'E832', 'E833', 'E834',
                                    'E835', 'E836', 'E837', 'E838', 'E839', 'E840', 'E841', 'E842', 'E843', 'E844',
                                    'E845', 'E846', 'E847', 'E848', 'E849', 'E850', 'E851', 'E852', 'E853', 'E854',
                                    'E855', 'E856', 'E857', 'E858', 'E859', 'E860', 'E861', 'E862', 'E863', 'E864',
                                    'E865', 'E866', 'E867', 'E868', 'E869', 'E870', 'E871', 'E872', 'E873', 'E874',
                                    'E875', 'E876', 'E877', 'E878', 'E879', 'E880', 'E881', 'E882', 'E883', 'E884',
                                    'E885', 'E886', 'E887', 'E888', 'E889', 'E890', 'E891', 'E892', 'E893', 'E894',
                                    'E895', 'E896', 'E897', 'E898', 'E899', 'E900', 'E901', 'E902', 'E903', 'E904',
                                    'E905', 'E906', 'E907', 'E908', 'E909', 'E910', 'E911', 'E912', 'E913', 'E914',
                                    'E915', 'E916', 'E917', 'E918', 'E919', 'E920', 'E921', 'E922', 'E923', 'E924',
                                    'E925', 'E926', 'E927', 'E928', 'E929', 'E930', 'E931', 'E932', 'E933', 'E934',
                                    'E935', 'E936', 'E937', 'E938', 'E939', 'E940', 'E941', 'E942', 'E943', 'E944',
                                    'E945', 'E946', 'E947', 'E948', 'E949', 'E950', 'E951', 'E952', 'E953', 'E954',
                                    'E955', 'E956', 'E957', 'E958', 'E959', 'E960', 'E961', 'E962', 'E963', 'E964',
                                    'E965', 'E966', 'E967', 'E968', 'E969', 'E970', 'E971', 'E972', 'E973', 'E974',
                                    'E975', 'E976', 'E977', 'E978', 'E979', 'E980', 'E981', 'E982', 'E983', 'E984',
                                    'E985', 'E986', 'E987', 'E988', 'E989', 'E990', 'E991', 'E992', 'E993', 'E994',
                                    'E995', 'E996', 'E997', 'E998', 'E999')),
                                  ('max_glu_serum', ('None', 'Norm', '>200', '>300')),
                                  ('A1Cresult', ('None', 'Norm', '>7', '>8')),
                                  ('insulin', ('No', 'Steady', 'Up', 'Down')), ('change', ('No', 'Ch')),
                                  ('diabetesMed', ('No', 'Yes')), ('readmitted_30d', ('0', '1'))),
                'dp_aim_rounds': None,
                'dp_smoke_rounds': 32,
                'dp_max_model_size_mb': 80.0,
                'dp_max_cells': 10000,
                'tabddpm_python': 'python3.10',
                'tabddpm_build_environment': True,
                'tabddpm_epochs': 1000,
                'tabddpm_timesteps': 1000,
                'tabddpm_timeout_seconds': 7200},
 'injections': {'doses': (('exact_copy', (0.001, 0.01, 0.05)), ('subgroup_suppression', (0.1, 0.5, 0.9)),
                          ('feature_shuffle', (0.1, 0.5, 0.9)), ('mcar', (0.05, 0.15, 0.3))),
                # v0.4.1: intended dimensions are gating dimensions only.
                'intended': (('exact_copy', ('privacy',)), ('subgroup_suppression', ('fidelity',)),
                             ('feature_shuffle', ('fidelity', 'utility')),
                             ('mcar', ('fidelity', 'utility'))),
                'subgroup_column': 'race',
                'subgroup_value': 'AfricanAmerican',
                'shuffle_feature': 'number_inpatient',
                'missing_features': ('time_in_hospital', 'num_lab_procedures', 'num_medications',
                                     'number_inpatient')},
 'calibration': {'repetitions': 100,          # v0.4.1: 50 fit + 50 held-out audit runs
                 'fit_runs': 50,
                 'assignment_seed': 81631,
                 'target_rates': (0.025, 0.05, 0.1),
                 'primary_rate': 0.05,
                 'scale_floor': 1e-08,
                 'design': 'development_scaled',
                 'studentization': 'pooled_with_candidate',   # full conformal
                 'quantile': 'full_conformal',
                 'admission_test_alpha': 0.05,  # prespecified admission rule for gating dimensions
                 'null_seed_salt': 'v0.4.1',    # fresh null draws, disjoint in seed from v0.4
                 'smoke_repetitions': 5,
                 'smoke_fit_runs': 3},
 'judges': {'threshold_offsets': (0.0, -0.05, -0.1),
            'score_mapping': 'exp(-max(0,max_directional_z)/temperature)',
            'temperature': 3.0},
 'execution': {'output_root': 'runs/main_v5',
               'checkout_dir': 'syntrustbench_v04_dependencies/frozen_v030',
               'stages': ('prepare', 'smoke', 'null', 'calibrate', 'clean', 'controls', 'injections', 'judges',
                          'analysis', 'report'),
               'dry_run': False,
               'workers': 4,
               'threads_per_worker': 2,
               'retry_failed': False,
               'install_dependencies': False,
               'smoke_mode': False,
               'smoke_rows': 200,
               'smoke_bootstrap': 20,
               'smoke_epochs': 1},
 'bootstrap_iterations': 200,
 'confidence_level': 0.95,
 'evaluation_seed': 7103,
 'min_subgroup_n': 30,
 'min_subgroup_outcomes': 10,
 'privacy_max_rows': 10000,
 'privacy_binomial_alpha': 0.05,
 'model_seeds': (7103, 7104, 7105, 7106, 7107),
 'missingness_rates': (0.05, 0.1, 0.2),
 'training_fractions': (0.25, 0.5, 0.75, 1.0),
 'missingness_repeats': 3,
 'positive_replicates': (17, 29, 43)}


## Install the pinned environment

**Inputs:** configuration, Python 3.11+ and network access for first installation. **Outputs:** pinned packages and a detached frozen v0.3 checkout. **Success criterion:** installation succeeds; full installed-source and environment-lock verification occurs after the library cells.

Apple Silicon uses CPU. CUDA is optional and is selected only when explicitly enabled and `torch.cuda.is_available()` is true. Installation may require a kernel restart if numerical modules were already imported. Dry-run performs setup but fits no models. All subsequent execution uses these pins; package drift cannot silently resume a campaign.

In [ ]:
import hashlib
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.11 or later (3.12 recommended).")
PINNED_PACKAGES = {'numpy': '2.3.4', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit-learn': '1.7.2', 'torch': '2.7.1', 'sdv': '1.37.4', 'ctgan': '0.13.0', 'rdt': '2.1.0', 'copulas': '0.15.0', 'sdmetrics': '0.33.0', 'pyyaml': '6.0.2', 'matplotlib': '3.10.3', 'smartnoise-synth': '1.0.8', 'mbi': '1.1.0', 'jax': '0.6.2', 'jaxlib': '0.6.2', 'optax': '0.2.5', 'chex': '0.1.90', 'opendp': '0.14.2', 'smartnoise-sql': '1.0.10', 'networkx': '3.4.2', 'disjoint-set': '0.9.0', 'cloudpickle': '3.1.2'}
FROZEN_COMMIT = "3dd0f7ce088994d84f4925ca7e0949d7e0ecd3d8"
checkout = Path(SETTINGS["execution"]["checkout_dir"]).resolve()
if SETTINGS["execution"]["install_dependencies"]:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *[f"{k}=={v}" for k, v in PINNED_PACKAGES.items()]], check=True)
    if not checkout.exists():
        checkout.parent.mkdir(parents=True, exist_ok=True)
        # The public repository of the frozen v0.3 release is withheld during double-blind review;
        # set FROZEN_V030_REPOSITORY to its URL to rerun the campaign.
        repository = os.environ.get("FROZEN_V030_REPOSITORY")
        if not repository:
            raise RuntimeError("Set FROZEN_V030_REPOSITORY to the public URL of the frozen v0.3 release.")
        subprocess.run(["git", "clone", repository, str(checkout)], check=True)
    subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", FROZEN_COMMIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(checkout)], check=True)
    importlib.invalidate_caches()
print("Dependency setup complete; exact source verification follows.")


## Library bootstrap

The library is additive and separate from the frozen checkout. Each following component cell writes inspectable source and ends with a short assertion check. The expensive instrument checks run in `smoke`, not during definition. The supplied reference code is preserved, including its fixed 95% intervals, top-250 categorical pooling, model settings and shared-RNG calculations.

In [ ]:
LIBRARY_DIGEST = 'c46914cd323948669320735f14ac180c4f03e7c0cedaaf2faf43d65bfcfc5090'
library_root = Path(SETTINGS["execution"]["output_root"]).resolve() / "notebook_library" / LIBRARY_DIGEST[:16]
package_dir = library_root / "syntrustbench_campaign"
package_dir.mkdir(parents=True, exist_ok=True)
def _write_module(name, source):
    path = package_dir / name
    if path.exists() and path.read_text() != source:
        raise RuntimeError(f"Existing embedded source was modified: {path}; choose a new output_root.")
    path.write_text(source)
_write_module("__init__.py", '"""SynTrustBench v0.4.1 gate-focused campaign. Components are in focused modules."""\n')
sys.path.insert(0, str(library_root))
assert package_dir.is_dir()


### Frozen configuration

In [ ]:
_write_module('config.py', r'''"""Frozen scientific configuration and semantic random seeds."""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
import hashlib
import json
import math
from pathlib import Path

DIMENSIONS = ('fidelity', 'utility', 'privacy', 'equity', 'robustness')
# v0.4.1: only dimensions with established, real-referenced tests decide the verdict.
# Equity and robustness are still computed and reported as non-gating diagnostics.
GATE_DIMENSIONS = ('fidelity', 'utility', 'privacy')
DIAGNOSTIC_DIMENSIONS = ('equity', 'robustness')
PROTOCOL = 'v0.4.1-gate'
STAGES = ('prepare', 'smoke', 'null', 'calibrate', 'clean', 'controls',
          'injections', 'judges', 'analysis', 'report')
COMMIT = '3dd0f7ce088994d84f4925ca7e0949d7e0ecd3d8'
CONTINUOUS = ('time_in_hospital', 'num_lab_procedures', 'num_procedures',
              'num_medications', 'number_outpatient', 'number_emergency',
              'number_inpatient', 'number_diagnoses')
CATEGORICAL = ('race', 'gender', 'age', 'admission_type_id',
               'discharge_disposition_id', 'admission_source_id', 'diag_1',
               'max_glu_serum', 'A1Cresult', 'insulin', 'change', 'diabetesMed')
TARGET = 'readmitted_30d'


def seed_for(*parts: object) -> int:
    """A stable JSON tuple SHA-256, interpreted as an unsigned 32-bit seed."""
    return int.from_bytes(hashlib.sha256(json.dumps(parts, separators=(',', ':'),
                          ensure_ascii=True).encode()).digest()[:4], 'big')


@dataclass(frozen=True)
class DataConfig:
    raw_csv: str = 'data/diabetic_data.csv'
    frozen_dir: str | None = None
    frozen_manifest: str | None = None
    continuous: tuple[str, ...] = CONTINUOUS
    categorical: tuple[str, ...] = CATEGORICAL
    target: str = TARGET
    protected: tuple[str, ...] = ('race', 'gender')
    split_seed: int = 49173
    split_fractions: tuple[float, ...] = (.4, .4, .2)

    def __post_init__(self):
        if self.split_fractions != (.4, .4, .2):
            raise ValueError('This protocol fixes the patient split at 40/40/20.')
        if bool(self.frozen_dir) != bool(self.frozen_manifest):
            raise ValueError('Frozen splits require a manifest with hashes and patient audit.')
        if len(set(self.columns)) != len(self.columns):
            raise ValueError('Analytic columns must be unique.')

    @property
    def columns(self) -> tuple[str, ...]:
        return self.continuous + self.categorical + (self.target,)


@dataclass(frozen=True)
class GeneratorConfig:
    names: tuple[str, ...] = ('GaussianCopula', 'CTGAN', 'TVAE')
    seeds: tuple[int, ...] = (17, 29, 43)
    epochs: int = 300
    enforce_min_max_values: bool = True
    enforce_rounding: bool = True
    allow_cuda: bool = False
    enable_dpmst: bool = False
    enable_tabddpm: bool = False
    epsilon: float = 1.0
    delta: float | None = None
    # Public, analyst-declared integer domains: column, lower, cap, width, overflow upper.
    dp_domains: tuple[tuple[str, int, int, int, int], ...] = (
        ('time_in_hospital', 1, 14, 1, 14),
        ('num_lab_procedures', 0, 200, 10, 1000),
        ('num_procedures', 0, 20, 1, 100),
        ('num_medications', 0, 100, 5, 500),
        ('number_outpatient', 0, 50, 5, 250),
        ('number_emergency', 0, 50, 5, 250),
        ('number_inpatient', 0, 50, 5, 250),
        ('number_diagnoses', 0, 30, 2, 100))
    dp_categories: tuple[tuple[str, tuple[str, ...]], ...] = (
        ('race', ('Caucasian', 'Asian', 'AfricanAmerican', 'Hispanic', 'Other')),
        ('gender', ('Female', 'Male', 'Unknown/Invalid')),
        ('age', tuple(f'[{i}-{i+10})' for i in range(0, 100, 10))),
        ('admission_type_id', tuple(str(i) for i in range(1, 10))),
        ('discharge_disposition_id', tuple(str(i) for i in range(1, 31))),
        ('admission_source_id', tuple(str(i) for i in range(1, 27))),
        ('diag_1', tuple(str(i) for i in range(1000))
                   + tuple(f'V{i:02d}' for i in range(100))
                   + tuple(f'E{i:03d}' for i in range(1000))),
        ('max_glu_serum', ('None', 'Norm', '>200', '>300')),
        ('A1Cresult', ('None', 'Norm', '>7', '>8')),
        ('insulin', ('No', 'Steady', 'Up', 'Down')),
        ('change', ('No', 'Ch')), ('diabetesMed', ('No', 'Yes')),
        ('readmitted_30d', ('0', '1')))
    dp_aim_rounds: int | None = None
    dp_smoke_rounds: int = 32
    dp_max_model_size_mb: float = 80.
    dp_max_cells: int = 10000
    tabddpm_python: str = 'python3.10'
    tabddpm_build_environment: bool = True
    tabddpm_epochs: int = 1000
    tabddpm_timesteps: int = 1000
    tabddpm_timeout_seconds: int = 7200

    @property
    def enabled_names(self) -> tuple[str, ...]:
        return self.names + ('DPAIM',) + (('DPMST',) if self.enable_dpmst else ()) + (('TabDDPM',) if self.enable_tabddpm else ())

    def __post_init__(self):
        if self.epochs < 1 or len(set(self.seeds)) != len(self.seeds):
            raise ValueError('Positive epochs and unique seeds are required.')
        if set(self.names) != {'GaussianCopula', 'CTGAN', 'TVAE'}:
            raise ValueError('Only the three prespecified generators are supported.')
        if not math.isfinite(self.epsilon) or self.epsilon <= 0 or (self.delta is not None and not 0 < self.delta < 1):
            raise ValueError('DP requires positive epsilon and 0 < delta < 1.')
        if len(dict(self.dp_categories)) != len(self.dp_categories):
            raise ValueError('Duplicate public categorical domains.')
        for _, levels in self.dp_categories:
            if not levels or len(set(levels)) != len(levels) or {'__OTHER__', '__MISSING__'} & set(levels):
                raise ValueError('Public categories must be unique; reserved levels are appended automatically.')
        for _, lower, cap, width, overflow in self.dp_domains:
            if lower < 0 or cap < lower or width < 1 or overflow < cap:
                raise ValueError('Invalid public integer domain.')
        if self.dp_smoke_rounds < 32 or (self.dp_aim_rounds is not None and self.dp_aim_rounds < 32):
            raise ValueError('AIM rounds must cover its initial marginal privacy allocation (>=32).')
        if min(self.dp_max_model_size_mb, self.dp_max_cells, self.tabddpm_epochs,
               self.tabddpm_timesteps, self.tabddpm_timeout_seconds) <= 0:
            raise ValueError('Generator compute limits must be positive.')


@dataclass(frozen=True)
class InjectionConfig:
    doses: tuple[tuple[str, tuple[float, ...]], ...] = (
        ('exact_copy', (.001, .01, .05)),
        ('subgroup_suppression', (.10, .50, .90)),
        ('feature_shuffle', (.10, .50, .90)), ('mcar', (.05, .15, .30)))
    # Intended gating dimensions only: the gate is what is being tested.
    intended: tuple[tuple[str, tuple[str, ...]], ...] = (
        ('exact_copy', ('privacy',)), ('subgroup_suppression', ('fidelity',)),
        ('feature_shuffle', ('fidelity', 'utility')),
        ('mcar', ('fidelity', 'utility')))
    subgroup_column: str = 'race'
    subgroup_value: str = 'AfricanAmerican'
    shuffle_feature: str = 'number_inpatient'
    missing_features: tuple[str, ...] = (
        'time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_inpatient')

    def __post_init__(self):
        if set(dict(self.doses)) != set(dict(self.intended)):
            raise ValueError('Every injection needs an intended dimension.')
        if any(not dims or not set(dims) <= set(GATE_DIMENSIONS) for _, dims in self.intended):
            raise ValueError('Intended dimensions must be gating dimensions.')
        for _, doses in self.doses:
            if tuple(sorted(set(doses))) != doses or any(not 0 < d <= 1 for d in doses):
                raise ValueError('Doses must be unique, increasing and in (0, 1].')


@dataclass(frozen=True)
class CalibrationConfig:
    repetitions: int = 100
    fit_runs: int = 50
    assignment_seed: int = 81631
    target_rates: tuple[float, ...] = (.025, .05, .10)
    primary_rate: float = .05
    scale_floor: float = 1e-8
    design: str = 'development_scaled'
    # Full conformal: the candidate is pooled with the fit runs for standardization and
    # ranked against them; exact marginal level r/(n+1) under exchangeability.
    studentization: str = 'pooled_with_candidate'
    quantile: str = 'full_conformal'
    # Prespecified admission rule: a dimension may gate only if every audit run is
    # evaluable and an exact one-sided binomial test finds no excess of honest-data
    # non-acceptance over primary_rate at admission_test_alpha.
    admission_test_alpha: float = .05
    null_seed_salt: str = 'v0.4.1'
    smoke_repetitions: int = 5
    smoke_fit_runs: int = 3

    def __post_init__(self):
        if not 3 <= self.fit_runs < self.repetitions or self.scale_floor <= 0:
            raise ValueError('Need at least three fit runs, held-out audit runs and positive scale.')
        if not 3 <= self.smoke_fit_runs < self.smoke_repetitions:
            raise ValueError('Smoke calibration needs at least three fit runs and one audit run.')
        if self.primary_rate not in self.target_rates or any(not 0 < a < 1 for a in self.target_rates):
            raise ValueError('Invalid target rates.')
        if self.design != 'development_scaled':
            raise ValueError('Only the specified development-only null design is implemented.')
        if (self.studentization, self.quantile) != ('pooled_with_candidate', 'full_conformal'):
            raise ValueError('Only the full-conformal max-statistic rule is implemented.')
        if not self.null_seed_salt:
            raise ValueError('A null seed salt is required.')
        if not 0 < self.admission_test_alpha < 1:
            raise ValueError('Invalid admission test level.')


@dataclass(frozen=True)
class JudgeConfig:
    # No weighted judge: with the equal default weights used in published practice a
    # weighted sum equals the mean, and any unequal weights would be an unprespecified choice.
    threshold_offsets: tuple[float, ...] = (0., -.05, -.10)
    score_mapping: str = 'exp(-max(0,max_directional_z)/temperature)'
    temperature: float = 3.

    def __post_init__(self):
        if self.temperature <= 0 or self.score_mapping != 'exp(-max(0,max_directional_z)/temperature)':
            raise ValueError('Unknown or invalid frozen score mapping.')
        if self.threshold_offsets[0] != 0 or any(o > 0 for o in self.threshold_offsets):
            raise ValueError('Offsets start at zero and only relax the threshold.')


@dataclass(frozen=True)
class ExecutionConfig:
    output_root: str = 'syntrustbench_v04_runs'
    checkout_dir: str = 'syntrustbench_v04_dependencies/frozen_v030'
    stages: tuple[str, ...] = STAGES
    dry_run: bool = True
    workers: int = 1
    threads_per_worker: int = 1
    retry_failed: bool = False
    install_dependencies: bool = False
    smoke_mode: bool = False
    smoke_rows: int = 200
    smoke_bootstrap: int = 20
    smoke_epochs: int = 1

    def __post_init__(self):
        if not set(self.stages) <= set(STAGES) or self.workers < 1 or self.threads_per_worker < 1:
            raise ValueError('Invalid stage or worker configuration.')
        if not 50 <= self.smoke_rows <= 500 or self.smoke_bootstrap < 20:
            raise ValueError('Smoke uses 50–500 rows and at least 20 bootstrap iterations.')


@dataclass(frozen=True)
class ExperimentConfig:
    data: DataConfig = field(default_factory=DataConfig)
    generators: GeneratorConfig = field(default_factory=GeneratorConfig)
    injections: InjectionConfig = field(default_factory=InjectionConfig)
    calibration: CalibrationConfig = field(default_factory=CalibrationConfig)
    judges: JudgeConfig = field(default_factory=JudgeConfig)
    execution: ExecutionConfig = field(default_factory=ExecutionConfig)
    bootstrap_iterations: int = 200
    confidence_level: float = .95
    evaluation_seed: int = 7103
    min_subgroup_n: int = 30
    min_subgroup_outcomes: int = 10
    privacy_max_rows: int = 10000
    privacy_binomial_alpha: float = .05
    model_seeds: tuple[int, ...] = (7103, 7104, 7105, 7106, 7107)
    missingness_rates: tuple[float, ...] = (.05, .10, .20)
    training_fractions: tuple[float, ...] = (.25, .50, .75, 1.)
    missingness_repeats: int = 3
    positive_replicates: tuple[int, ...] = (17, 29, 43)

    def __post_init__(self):
        if self.bootstrap_iterations < 20 or self.confidence_level != .95:
            raise ValueError('Reference implementation uses fixed 95% CIs; need >=20 resamples.')
        if self.privacy_max_rows < 10 or self.min_subgroup_n < 2 or self.min_subgroup_outcomes < 1:
            raise ValueError('Invalid row limits.')
        if self.missingness_repeats < 2 or len(self.model_seeds) < 2:
            raise ValueError('Robustness needs repeated stressors.')

    def scientific(self) -> dict:
        value = asdict(self)
        value['protocol'] = PROTOCOL
        value['gate_dimensions'] = list(GATE_DIMENSIONS)
        execution = value.pop('execution')
        value['smoke_settings'] = {k: execution[k] for k in
                                  ('smoke_mode', 'smoke_rows', 'smoke_bootstrap', 'smoke_epochs')}
        for path in ('raw_csv', 'frozen_dir', 'frozen_manifest'):
            value['data'].pop(path)
        return value

    @property
    def fingerprint(self) -> str:
        return hashlib.sha256(json.dumps(self.scientific(), sort_keys=True).encode()).hexdigest()

    @property
    def root(self) -> Path:
        label = 'smoke' if self.execution.smoke_mode else 'full'
        return Path(self.execution.output_root).resolve() / f'{label}_{self.fingerprint[:16]}'

    def benchmark(self, *, smoke: bool = False) -> dict:
        small = smoke or self.execution.smoke_mode
        d = self.data
        return {
            'dataset': {'name': 'UCI Diabetes earliest encounter', 'modality': 'tabular', 'outcome': d.target},
            'columns': {'continuous': list(d.continuous), 'categorical': list(d.categorical), 'binary': [d.target]},
            'utility_task': {'target': d.target, 'type': 'binary_classification', 'positive_label': '1',
                             'features': list(d.continuous + d.categorical)},
            'protected_attributes': list(d.protected),
            'clinical_constraints': [f'{c} >= 0' for c in d.continuous] + ['time_in_hospital >= 1', 'time_in_hospital <= 14'],
            'generator': {'name': 'v04_campaign', 'seed': self.evaluation_seed},
            'uncertainty': {'bootstrap_iterations': self.execution.smoke_bootstrap if small else self.bootstrap_iterations,
                            'confidence_level': .95, 'random_seed': self.evaluation_seed},
            'subgroup_analysis': {'minimum_n': self.min_subgroup_n,
                                  'minimum_outcomes_per_class': self.min_subgroup_outcomes},
            'privacy': {'distance_max_rows': min(500, self.privacy_max_rows) if small else self.privacy_max_rows,
                        'attack_max_rows': min(500, self.privacy_max_rows) if small else self.privacy_max_rows},
            'robustness': {'downstream_model_seeds': list(self.model_seeds[:2] if small else self.model_seeds),
                           'missingness_rates': list(self.missingness_rates),
                           'training_size_fractions': list(self.training_fractions)},
        }


def self_test() -> None:
    assert seed_for('a', 1) == seed_for('a', 1) != seed_for('a', 2)
    assert ExperimentConfig().benchmark()['uncertainty']['bootstrap_iterations'] == 200
    assert set(GATE_DIMENSIONS) | set(DIAGNOSTIC_DIMENSIONS) == set(DIMENSIONS)
    assert not set(GATE_DIMENSIONS) & set(DIAGNOSTIC_DIMENSIONS)
    c = CalibrationConfig()
    assert (c.repetitions, c.fit_runs) == (100, 50)


def from_mapping(settings: dict) -> ExperimentConfig:
    """Convert the single notebook configuration cell to frozen dataclasses."""
    def freeze(value):
        if isinstance(value, list): return tuple(freeze(v) for v in value)
        if isinstance(value, dict): return {k:freeze(v) for k,v in value.items()}
        return value
    values = freeze(settings)
    constructors = {'data':DataConfig, 'generators':GeneratorConfig, 'injections':InjectionConfig,
                    'calibration':CalibrationConfig, 'judges':JudgeConfig, 'execution':ExecutionConfig}
    return ExperimentConfig(**{k:constructors[k](**v) if k in constructors else v for k,v in values.items()})
''')

from syntrustbench_campaign import config

config.self_test()


### Typed evaluator results and non-compensatory gate

In [ ]:
_write_module('results.py', r'''"""Typed evaluation contract; metric payloads retain their original meaning."""
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Literal

import pandas as pd

Status = Literal["Pass", "Conditional", "Fail", "NotEvaluated"]


@dataclass(frozen=True)
class DimensionResult:
    """A dimension verdict plus unmodified metrics and extension metadata.

    Frozen fields prevent reassignment; nested legacy dictionaries remain plain
    dictionaries. No normalization, rounding, missing-value replacement, or
    threshold remapping occurs here.
    """

    dimension: str
    status: Status
    metrics: dict[str, dict[str, Any]]
    weaknesses: list[str]
    warnings: tuple[str, ...] = ()
    details: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> DimensionResult:
        """Wrap a legacy result, keeping every metric and additional field."""
        core = {"dimension", "status", "metrics", "weaknesses", "warnings"}
        return cls(
            dimension=payload["dimension"], status=payload["status"],
            metrics=payload["metrics"], weaknesses=payload["weaknesses"],
            warnings=tuple(payload.get("warnings", ())),
            details={key: value for key, value in payload.items() if key not in core},
        )

    def to_dict(self, *, include_warnings: bool = False) -> dict[str, Any]:
        """Return the exact legacy schema by default; warnings are opt-in."""
        payload = {
            "dimension": self.dimension, "status": self.status,
            "metrics": self.metrics, "weaknesses": self.weaknesses,
            **self.details,
        }
        if include_warnings or self.warnings:
            payload["warnings"] = list(self.warnings)
        return payload


class DimensionEvaluator(ABC):
    """Shared interface for candidate-table evaluation."""

    @abstractmethod
    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        """Return a dimension's evidence and verdict without aggregating others."""
        raise NotImplementedError
''')

from syntrustbench_campaign import results

_write_module('core.py', r'''"""Atomic artifact handling and a transactional, hash-verifying job ledger."""
from __future__ import annotations

import contextlib
from datetime import datetime, timezone
import fcntl
import hashlib
import json
import os
from pathlib import Path
import time
import traceback
import uuid
from typing import Any, Callable

import numpy as np
import pandas as pd


class UnavailableAnalysis(RuntimeError):
    """Expected missing evidence, never a numerical/programming-error substitute."""


def utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def file_hash(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def clean_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [clean_json(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.generic):
        return value.item()
    return value


def write_json(path: str | Path, value: Any) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    with tmp.open('w') as handle:
        json.dump(clean_json(value), handle, indent=2, allow_nan=False)
        handle.write('\n'); handle.flush(); os.fsync(handle.fileno())
    os.replace(tmp, path)
    return str(path)


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text())


def save_frame(frame: pd.DataFrame, path: str | Path) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, float_format='%.17g', lineterminator='\n')
    return str(path)


def available_number(value: Any) -> bool:
    return isinstance(value, (int, float, np.number)) and bool(np.isfinite(value))


class Gate:
    """Non-compensatory precedence, including at the overall gate."""

    @staticmethod
    def aggregate(statuses) -> str:
        values = list(statuses)
        if not values:
            return 'NotEvaluated'
        if not set(values) <= {'Pass', 'Conditional', 'Fail', 'NotEvaluated'}:
            raise ValueError('Unknown evidence status.')
        return next(s for s in ('Fail', 'NotEvaluated', 'Conditional', 'Pass') if s in values)


class JobLedger:
    def __init__(self, root: Path, fingerprint: str):
        self.root, self.fingerprint = Path(root), fingerprint
        (self.root / 'states').mkdir(parents=True, exist_ok=True)

    def state_path(self, key: str) -> Path:
        return self.root / 'states' / (key.replace('/', '__') + '.json')

    def state(self, key: str) -> dict:
        path = self.state_path(key)
        return read_json(path) if path.exists() else {'key': key, 'state': 'planned'}

    def append(self, filename: str, record: dict) -> None:
        data = (json.dumps(clean_json({'time': utc(), **record}), allow_nan=False) + '\n').encode()
        descriptor = os.open(self.root / filename, os.O_CREAT | os.O_WRONLY | os.O_APPEND, 0o600)
        try:
            os.write(descriptor, data)
        finally:
            os.close(descriptor)

    def set(self, key: str, state: str, **details) -> dict:
        record = {'key': key, 'state': state, 'fingerprint': self.fingerprint,
                  'updated_at': utc(), **details}
        write_json(self.state_path(key), record)
        self.append('events.jsonl', record)
        return record

    def result(self, key: str) -> dict:
        record = self.state(key)
        if record['state'] != 'completed':
            raise UnavailableAnalysis(f'{key}: {record["state"]}; {record.get("reason", "completed dependency required")}')
        if record['fingerprint'] != self.fingerprint:
            raise RuntimeError(f'Campaign fingerprint changed: {key}')
        for path, expected in record['artifact_hashes'].items():
            if not Path(path).is_file() or file_hash(path) != expected:
                raise RuntimeError(f'Artifact changed or missing: {path}')
        for path, expected in record.get('input_hashes', {}).items():
            if not Path(path).is_file() or file_hash(path) != expected:
                raise RuntimeError(f'Input changed or missing: {path}')
        return record['result']


class JobRunner:
    def __init__(self, ledger: JobLedger, retry_failed: bool = False):
        self.ledger, self.retry_failed = ledger, retry_failed

    def run(self, key: str, operation: Callable[[Path], dict], inputs=()) -> dict | None:
        """Each job owns an attempt directory; retries never overwrite old evidence."""
        lock_path = self.ledger.state_path(key).with_suffix('.lock')
        with lock_path.open('a') as lock:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            previous = self.ledger.state(key)
            hashes = {str(Path(p).resolve()): file_hash(p) for p in inputs}
            if previous['state'] == 'unavailable' and not self.retry_failed:
                if previous.get('input_hashes', {}) != hashes:
                    raise RuntimeError(f'Unavailable job inputs changed: {key}; enable retry_failed to retry.')
                for path, expected in previous.get('artifact_hashes', {}).items():
                    if not Path(path).is_file() or file_hash(path) != expected:
                        raise RuntimeError(f'Unavailable artifact changed or missing: {path}')
                return None
            if previous['state'] == 'completed':
                if previous.get('input_hashes', {}) != hashes:
                    changed = set(hashes) | set(previous.get('input_hashes', {}))
                    changed = [p for p in changed if hashes.get(p) != previous.get('input_hashes', {}).get(p)]
                    refresh = (self.retry_failed and key in ('judges/all','analysis/tables','report/artifacts')
                               and all(Path(p).parent == self.ledger.root/'states' for p in changed))
                    if not refresh:
                        raise RuntimeError(f'Job inputs changed: {key}; use a new campaign.')
                    for path, expected in previous['artifact_hashes'].items():
                        if not Path(path).is_file() or file_hash(path) != expected:
                            raise RuntimeError(f'Artifact changed or missing: {path}')
                else:
                    return self.ledger.result(key)
            if previous['state'] in ('failed', 'running') and not self.retry_failed:
                raise RuntimeError(f'{key} was {previous["state"]}; inspect ledger before enabling retry_failed.')
            attempt = self.ledger.root / 'jobs' / key.replace('/', '__') / uuid.uuid4().hex
            attempt.mkdir(parents=True)
            self.ledger.set(key, 'running', attempt=str(attempt), input_hashes=hashes)
            started = time.perf_counter()
            try:
                with (attempt / 'job.log').open('w') as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                    result = operation(attempt)
                write_json(attempt / 'result.json', result)
                artifacts = {str(p): file_hash(p) for p in sorted(attempt.rglob('*')) if p.is_file()}
                self.ledger.set(key, 'completed', result=result, artifact_hashes=artifacts,
                                input_hashes=hashes, seconds=time.perf_counter() - started)
                return result
            except UnavailableAnalysis as exc:
                write_json(attempt / 'unavailable.json', {'status': 'Unavailable', 'reason': str(exc)})
                artifacts = {str(p): file_hash(p) for p in sorted(attempt.rglob('*')) if p.is_file()}
                self.ledger.set(key, 'unavailable', reason=str(exc), attempt=str(attempt), input_hashes=hashes,
                                artifact_hashes=artifacts, seconds=time.perf_counter()-started)
                return None
            except BaseException:
                failure = traceback.format_exc()
                (attempt / 'failure_traceback.txt').write_text(failure)
                self.ledger.set(key, 'failed', attempt=str(attempt), input_hashes=hashes,
                                traceback_path=str(attempt / 'failure_traceback.txt'))
                self.ledger.append('failures.jsonl', {'key': key, 'traceback': failure})
                raise


def self_test() -> None:
    assert Gate.aggregate(['Pass', 'NotEvaluated']) == 'NotEvaluated'
    assert Gate.aggregate(['Fail', 'NotEvaluated']) == 'Fail'
    assert Gate.aggregate([]) == 'NotEvaluated'
''')

from syntrustbench_campaign import core

core.self_test()
assert results.DimensionResult.from_dict({"dimension":"test","status":"Pass","metrics":{},"weaknesses":[]}).status == "Pass"


### Reference Fidelity, Utility and cached real baselines — preserved component

In [ ]:
_write_module('reference.py', r'''"""Reference-calibrated Fidelity and Utility (experimental v0.4 add-on).

This module does not modify the frozen v0.3 dimensions. It adds three things
that v0.3 lacks, each tied to an established method:

1. A held-out real reference.  Fidelity is measured against real rows the
   generator never saw (as Synthcity's Benchmarks do with ``X.test()``), and
   every fidelity statistic is paired with the same statistic computed for an
   independent *real* sample of the same population (the "real-vs-real floor").
   A synthetic table is judged by its excess over what sampling alone produces.
2. Mass-weighted category coverage.  v0.3 counts category *levels*, so a level
   seen once weighs as much as a common one and a second real sample fails the
   0.95 rule.  Here coverage is the share of reference *patients* whose level
   appears in the synthetic table.
3. A joint-distribution test.  v0.3 checks marginals and pairwise associations
   only.  A cross-fitted classifier two-sample test (Lopez-Paz & Oquab, 2017;
   Synthcity ``detection_*``) and the propensity-score MSE (Snoke et al., 2018)
   are added, again relative to the real-vs-real floor.

Utility keeps the v0.3 primary rule (random-forest chance-corrected retention)
so official verdicts stay comparable, and adds: a real-reference retention
(positive control inside the dimension), a probe family (random forest,
logistic regression, gradient boosting, as in Synthcity's performance metrics),
and calibration (calibration-in-the-large and slope; Van Calster et al., 2019),
each relative to a model trained on real data.

All margins below are provisional, are recorded in the output, and are meant
to be replaced by empirically calibrated cut-offs from repeated real-vs-real
runs. Detection can raise Conditional but never Fail until that calibration
exists, because with tens of thousands of rows almost any difference is
statistically detectable.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from types import MappingProxyType

from .results import DimensionEvaluator, DimensionResult

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from syntrustbench.constraints import constraint_violations
from syntrustbench.fidelity import _association_error, _jensen_shannon, _normalized_wasserstein
from syntrustbench.stats import (
    FeatureEncoder,
    aggregate_status,
    categorical_values,
    metric,
    percentile_interval,
    safe_auroc,
)
from syntrustbench.config import load_config
from syntrustbench.validation import _canonical_keys, _coerce_frame, _map_target

PROVISIONAL = MappingProxyType({
    # Fidelity
    "coverage_margin": 0.05,          # tolerated loss of reference patient mass beyond the floor
    "detection_margin": 0.05,         # AUC excess over the floor that raises Conditional
    "wasserstein_excess": 0.20,       # v0.3 thresholds, now applied to excess over floor
    "jsd_excess": 0.10,
    "association_excess": 0.15,
    "constraint_rate_fail": 0.01,     # unchanged from v0.3
    # Utility
    "retention_fail": 0.80,           # unchanged from v0.3
    "retention_conditional": 0.90,
    "calibration_ratio_band": 1.25,   # relative O/E and slope must stay within [1/1.25, 1.25]
})
MAX_TREE_LEVELS = 250  # HistGradientBoosting native categoricals allow at most 255 levels


# --------------------------------------------------------------------------- encoders
@dataclass
class _TreeEncoder:
    """Integer codes for native-categorical gradient boosting.

    Levels outside the ``MAX_TREE_LEVELS`` most frequent levels of the fitting
    table are pooled into one ``__OTHER__`` code (stated limitation: rare-level
    detail is invisible to this encoder; mass-weighted coverage covers it).
    """

    continuous: list[str]
    categorical: list[str]
    levels: dict[str, list[str]] = field(default_factory=dict)

    def fit(self, frame: pd.DataFrame) -> "_TreeEncoder":
        self.levels = {
            c: categorical_values(frame[c]).value_counts().index[:MAX_TREE_LEVELS].tolist()
            for c in self.categorical
        }
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        out = frame[self.continuous].apply(pd.to_numeric, errors="coerce").copy()
        for c in self.categorical:
            values = categorical_values(frame[c])
            values = values.where(values.isin(self.levels[c]), "__OTHER__")
            out[c] = pd.Categorical(values, categories=self.levels[c] + ["__OTHER__"]).codes
        return out.reset_index(drop=True)

    @property
    def categorical_mask(self) -> list[bool]:
        return [False] * len(self.continuous) + [True] * len(self.categorical)


def _hgb(seed: int, mask: list[bool]) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        max_iter=200, categorical_features=mask, random_state=seed
    )


# --------------------------------------------------------------------------- fidelity pieces
def _mass_coverage(reference: pd.Series, candidate: pd.Series) -> np.ndarray:
    """Per-reference-row indicator: is this patient's level present in the candidate table?"""
    present = set(categorical_values(candidate))
    return categorical_values(reference).isin(present).to_numpy(dtype=float)


def _two_sample_detection(
    reference: pd.DataFrame,
    candidate: pd.DataFrame,
    encoder: _TreeEncoder,
    seed: int,
    folds: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Cross-fitted classifier two-sample test; returns labels and out-of-fold scores."""
    features = pd.concat(
        [encoder.transform(reference), encoder.transform(candidate)], ignore_index=True
    )
    labels = np.r_[np.zeros(len(reference)), np.ones(len(candidate))]
    scores = np.zeros(len(labels))
    splitter = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for train_rows, test_rows in splitter.split(features, labels):
        model = _hgb(seed, encoder.categorical_mask).fit(
            features.iloc[train_rows], labels[train_rows]
        )
        scores[test_rows] = model.predict_proba(features.iloc[test_rows])[:, 1]
    return labels, scores


def _pmse(labels: np.ndarray, scores: np.ndarray) -> float:
    share = float(labels.mean())
    return float(np.mean((scores - share) ** 2))


def _distance_block(reference, candidate, continuous, categorical) -> dict[str, float]:
    wass = [_normalized_wasserstein(reference[c], candidate[c]) for c in continuous]
    wass = [w for w in wass if np.isfinite(w)]
    jsd = [_jensen_shannon(reference[c], candidate[c])[0] for c in categorical]
    per_column_wass = {
        c: _normalized_wasserstein(reference[c], candidate[c]) for c in continuous
    }
    return {
        "mean_normalized_wasserstein": float(np.mean(wass)) if wass else float("nan"),
        "worst_column_normalized_wasserstein": float(np.nanmax(list(per_column_wass.values())))
        if per_column_wass else float("nan"),
        "worst_column": max(per_column_wass, key=lambda c: np.nan_to_num(per_column_wass[c], nan=-1))
        if per_column_wass else None,
        "mean_jensen_shannon_divergence": float(np.mean(jsd)) if jsd else float("nan"),
        "association_matrix_error": _association_error(
            reference, candidate, continuous, categorical
        ),
    }


# --------------------------------------------------------------------------- utility pieces
def _fit_predict(probe: str, train: pd.DataFrame, test: pd.DataFrame, target: str,
                 onehot: FeatureEncoder, tree: _TreeEncoder, seed: int) -> np.ndarray:
    y = train[target].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return np.full(len(test), float(y.mean()))
    if probe == "random_forest":  # identical to the v0.3 primary probe
        model = RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=seed
        )
        return model.fit(onehot.transform(train), y).predict_proba(onehot.transform(test))[:, 1]
    if probe == "logistic_regression":
        model = LogisticRegression(C=1.0, max_iter=3000)
        return model.fit(onehot.transform(train), y).predict_proba(onehot.transform(test))[:, 1]
    if probe == "gradient_boosting":
        model = _hgb(seed, tree.categorical_mask)
        return model.fit(tree.transform(train), y).predict_proba(tree.transform(test))[:, 1]
    raise ValueError(probe)


def _calibration(y: np.ndarray, p: np.ndarray) -> tuple[float, float]:
    """Calibration-in-the-large (observed/expected) and calibration slope."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    if len(np.unique(y)) < 2 or np.allclose(p, p[0]):
        return float("nan"), float("nan")
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=2000).fit(logit, y).coef_[0][0]
    return float(y.mean() / p.mean()), float(slope)


def _retention(y, base, cand):
    real, syn = safe_auroc(y, base), safe_auroc(y, cand)
    return (syn - 0.5) / (real - 0.5) if np.isfinite(real) and real > 0.5 else float("nan")


def _retention_verdict(point, low, high, base_low) -> str:
    if base_low is None or not np.isfinite(base_low) or base_low <= 0.5:
        return "NotEvaluated"
    if high is not None and high < PROVISIONAL["retention_fail"]:
        return "Fail"
    if low is None or high is None:
        return "NotEvaluated"
    return "Pass" if low >= PROVISIONAL["retention_conditional"] else "Conditional"


# --------------------------------------------------------------------------- main context
class ReferenceContext:
    """Everything that depends only on real data, computed once per cohort.

    ``real_train`` is the generator's training data, ``real_reference`` an
    independent real sample of the same population and size (e.g. the
    calibration pool), ``real_test`` the untouched utility test set.
    """

    PROBES = ("random_forest", "logistic_regression", "gradient_boosting")

    def __init__(self, real_train, real_reference, real_test, config, *,
                 iterations: int | None = None, seed: int | None = None):
        # Reuse frozen v0.3 validation helpers: declared columns, types, target mapping.
        # v0.3 stops on any exact row shared between two real tables, as a leakage proxy.
        # Here patient disjointness of every role is already asserted upstream by hashed
        # patient IDs (RoleAssigner), so an exact match is two different patients with
        # identical recorded values: an expected real-real coincidence (the quantity the
        # v0.4 privacy null models). Stopping, dropping or re-drawing such runs would
        # select the null sample, so the counts are recorded and evaluation continues.
        benchmark = load_config(config)

        def _real(frame, label):
            if not isinstance(frame, pd.DataFrame):
                frame = pd.read_csv(frame)
            frame, _ = _coerce_frame(frame.copy(), benchmark, label)
            frame, _ = _map_target(frame, benchmark, label, require_two_classes=True)
            return frame.reset_index(drop=True)

        self.config = benchmark
        self.train = _real(real_train, "real_train")
        self.test = _real(real_test, "real_test")
        self.reference = _real(real_reference, "real_reference")
        train_keys = set(_canonical_keys(self.train, benchmark).tolist())
        self.exact_row_coincidences = {
            name: int(_canonical_keys(frame, benchmark).isin(train_keys).sum())
            for name, frame in (("reference", self.reference), ("test", self.test))}
        self.iterations = iterations or self.config.bootstrap_iterations
        self.seed = self.config.random_seed if seed is None else seed
        self.rng = np.random.default_rng(self.seed)
        cfg = self.config
        self.continuous = cfg.continuous
        self.categorical = cfg.categorical + cfg.binary
        self.target = cfg.target
        self.tree_fidelity = _TreeEncoder(self.continuous, self.categorical).fit(self.reference)
        self.onehot = FeatureEncoder(cfg.continuous_features, cfg.categorical_features).fit(self.train)
        self.tree_utility = _TreeEncoder(cfg.continuous_features, cfg.categorical_features).fit(self.train)
        self.y_test = self.test[self.target].astype(int).to_numpy()

        # Fidelity floor: the generator's own training data versus the reference.
        self.floor = self._fidelity_statistics(self.train)
        # Utility baselines: train on real training data and on the real reference.
        self.trtr = {p: self._predict(p, self.train) for p in self.PROBES}
        self.reference_pred = {p: self._predict(p, self.reference) for p in self.PROBES}
        self.trtr_auroc_ci = {
            p: percentile_interval(
                safe_auroc(self.y_test, self.trtr[p]),
                lambda rows, p=p: safe_auroc(self.y_test[rows], self.trtr[p][rows]),
                len(self.y_test), self.rng, self.iterations, 0.95)
            for p in self.PROBES
        }
        lr = "logistic_regression"
        self.reference_calibration = _calibration(self.y_test, self.trtr[lr])

    # ---------------------------------------------------------------- helpers
    def _predict(self, probe, train_frame):
        return _fit_predict(probe, train_frame, self.test, self.target,
                            self.onehot, self.tree_utility, self.seed)

    def _fidelity_statistics(self, candidate: pd.DataFrame) -> dict:
        coverage_rows = {c: _mass_coverage(self.reference[c], candidate[c]) for c in self.categorical}
        coverage = {c: float(v.mean()) for c, v in coverage_rows.items()}
        labels, scores = _two_sample_detection(self.reference, candidate, self.tree_fidelity, self.seed)
        auc = safe_auroc(labels, scores)
        auc_ci = percentile_interval(
            auc, lambda rows: safe_auroc(labels[rows], scores[rows]),
            len(labels), self.rng, self.iterations, 0.95)
        stats = _distance_block(self.reference, candidate, self.continuous, self.categorical)
        stats.update({
            "coverage": coverage, "coverage_rows": coverage_rows,
            "detection_auc": auc, "detection_auc_ci": auc_ci,
            "pmse": _pmse(labels, scores),
        })
        return stats

    # Backward-compatible dict API; use the evaluators for typed results.
    def fidelity(self, synthetic: pd.DataFrame) -> dict:
        """Evaluate fidelity, consuming the shared RNG exactly as before."""
        return ReferenceFidelity(self).evaluate(synthetic).to_dict()

    def utility(self, synthetic: pd.DataFrame) -> dict:
        """Evaluate utility, consuming the shared RNG exactly as before."""
        return ReferenceUtility(self).evaluate(synthetic).to_dict()


@dataclass(frozen=True)
class ReferenceFidelity(DimensionEvaluator):
    """Original fidelity calculations over one shared real-data context.

    Metrics retain the original units and higher_is_worse metadata.
    Evaluator construction consumes no randomness and fits no models.
    Calls advance context.rng: preserve table and dimension ordering.
    """

    context: ReferenceContext

    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        """Score a candidate without changing the supplied statistical rules."""
        ctx = self.context
        P = PROVISIONAL
        cand = ctx._fidelity_statistics(synthetic)
        floor = ctx.floor
        metrics: dict[str, dict] = {}
        weaknesses: list[str] = []

        # Mass-weighted coverage, judged as loss relative to the real floor.
        deficits = {}
        for c in ctx.categorical:
            rows_syn, rows_floor = cand["coverage_rows"][c], floor["coverage_rows"][c]
            point = float(rows_floor.mean() - rows_syn.mean())
            low, high = percentile_interval(
                point, lambda r, a=rows_floor, b=rows_syn: float(a[r].mean() - b[r].mean()),
                len(rows_syn), ctx.rng, ctx.iterations, 0.95)
            deficits[c] = (point, low, high)
        worst = max(deficits, key=lambda c: deficits[c][0])
        point, low, high = deficits[worst]
        coverage_fail = low is not None and low > P["coverage_margin"]
        metrics["mass_weighted_coverage_deficit"] = metric(
            point, ci_low=low, ci_high=high, interval_type="paired reference-row bootstrap (95%)",
            higher_is_worse=True, flag=coverage_fail,
            threshold=f"lower 95% bound > {P['coverage_margin']:.2f} -> Fail",
            note=(f"Worst column '{worst}': share of reference patients whose category is "
                  f"absent from the synthetic table, minus the same share for the real floor "
                  f"(synthetic {cand['coverage'][worst]:.4f}; floor {floor['coverage'][worst]:.4f})."))
        if coverage_fail:
            weaknesses.append(f"{worst}: synthetic omits categories held by "
                              f"{point:.1%} more reference patients than a real sample does")

        # Joint-distribution test (cannot Fail before null calibration).
        auc, (alow, ahigh) = cand["detection_auc"], cand["detection_auc_ci"]
        fhigh = floor["detection_auc_ci"][1]
        detected = (alow is not None and fhigh is not None
                    and alow > fhigh + P["detection_margin"])
        metrics["detection_auc"] = metric(
            auc, ci_low=alow, ci_high=ahigh,
            interval_type="bootstrap over cross-fitted predictions (95%)",
            higher_is_worse=True, flag=detected,
            threshold=(f"lower bound > floor upper bound + {P['detection_margin']:.2f} "
                       "-> Conditional (Fail requires null calibration)"),
            note=(f"Cross-fitted gradient-boosting classifier two-sample test against the "
                  f"held-out real reference; real-vs-real floor AUC {floor['detection_auc']:.3f}."))
        floor_pmse = floor["pmse"]
        metrics["pmse_ratio_to_real_floor"] = metric(
            cand["pmse"] / floor_pmse if floor_pmse > 0 else float("nan"), higher_is_worse=True,
            note="Propensity-score MSE divided by the real-vs-real pMSE (empirical null; 1 = like real).")

        # Low-order distances, now as excess over the floor (v0.3 thresholds).
        for name, key in [("mean_normalized_wasserstein", "wasserstein_excess"),
                          ("mean_jensen_shannon_divergence", "jsd_excess"),
                          ("association_matrix_error", "association_excess")]:
            excess = cand[name] - floor[name]
            metrics[f"{name}_excess"] = metric(
                excess, higher_is_worse=True, flag=bool(excess > P[key]),
                threshold=f"> {P[key]:.2f} -> Conditional",
                note=f"Synthetic {cand[name]:.4f} minus real floor {floor[name]:.4f}.")
        metrics["worst_column_normalized_wasserstein"] = metric(
            cand["worst_column_normalized_wasserstein"], higher_is_worse=True,
            note=(f"Worst continuous column '{cand['worst_column']}' (reported so a column "
                  "average cannot hide one broken variable); floor "
                  f"{floor['worst_column_normalized_wasserstein']:.4f}."))

        # Clinical constraints: unchanged v0.3 rule.
        rate, per_rule, _ = constraint_violations(synthetic, ctx.config.clinical_constraints)
        constraint_fail = rate is not None and rate > P["constraint_rate_fail"]
        metrics["clinical_constraint_violation_rate"] = metric(
            rate, higher_is_worse=True, flag=constraint_fail,
            status="Available" if rate is not None else "NotEvaluated",
            threshold=f"> {P['constraint_rate_fail']:.2f} -> Fail", note=str(per_rule))

        conditional = detected or any(
            metrics[f"{n}_excess"]["flag"] for n in
            ["mean_normalized_wasserstein", "mean_jensen_shannon_divergence", "association_matrix_error"])
        status = aggregate_status([
            "Fail" if (coverage_fail or constraint_fail) else "Pass",
            "NotEvaluated" if rate is None else "Pass",
            "Conditional" if conditional else "Pass",
        ])
        return DimensionResult.from_dict({"dimension": "fidelity_reference_calibrated", "status": status,
                "metrics": metrics, "weaknesses": weaknesses,
                "provisional_margins": {k: v for k, v in P.items() if not k.startswith(("retention", "calibration"))},
                "per_column_coverage": cand["coverage"],
                "floor_per_column_coverage": floor["coverage"]})


@dataclass(frozen=True)
class ReferenceUtility(DimensionEvaluator):
    """Original utility calculations over one shared real-data context.

    Metrics retain the original units and higher_is_worse metadata.
    Evaluator construction consumes no randomness and fits no models.
    Calls advance context.rng: preserve table and dimension ordering.
    """

    context: ReferenceContext

    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        """Score a candidate without changing the supplied statistical rules."""
        ctx = self.context
        P = PROVISIONAL
        y = ctx.y_test
        metrics: dict[str, dict] = {}
        weaknesses: list[str] = []
        outcome_levels = synthetic[ctx.target].astype(int).nunique()
        verdicts = {}
        for probe in ctx.PROBES:
            tstr = ctx._predict(probe, synthetic)
            base_low = ctx.trtr_auroc_ci[probe][0]
            for label, cand in [("synthetic", tstr), ("real_reference", ctx.reference_pred[probe])]:
                point = _retention(y, ctx.trtr[probe], cand)
                low, high = (None, None)
                if np.isfinite(point):
                    low, high = percentile_interval(
                        point, lambda r, c=cand: _retention(y[r], ctx.trtr[probe][r], c[r]),
                        len(y), ctx.rng, ctx.iterations, 0.95)
                verdict = _retention_verdict(point, low, high, base_low)
                if label == "synthetic":
                    verdicts[probe] = verdict
                    if probe == "logistic_regression":
                        oe, slope = _calibration(y, cand)
                metrics[f"{label}_retention_{probe}"] = metric(
                    point if np.isfinite(point) else None, ci_low=low, ci_high=high,
                    interval_type="paired held-out real-test bootstrap (95%)",
                    higher_is_worse=False, status=verdict if label == "synthetic" else "Available",
                    note=("Chance-corrected AUROC retention versus the same probe trained on "
                          "real training data." + (" Positive control: an independent real "
                          "sample; should be near 1." if label == "real_reference" else "")))
        primary = verdicts["random_forest"]
        disagree = sorted(p for p, v in verdicts.items() if v != primary)
        # Only a confident failure elsewhere (upper bound below the Fail line) downgrades a
        # primary Pass; a merely wider interval does not (it would flag real data, as the
        # half-size positive control showed for logistic regression).
        probe_conditional = primary == "Pass" and any(verdicts[p] == "Fail" for p in disagree)
        metrics["probe_agreement"] = metric(
            ", ".join(f"{p}={v}" for p, v in verdicts.items()), higher_is_worse=False,
            flag=bool(disagree),
            note="Primary verdict is the v0.3 random forest; a primary Pass becomes "
                 "Conditional only if another probe confidently Fails (probe-dependent utility).")

        ref_oe, ref_slope = ctx.reference_calibration
        rel_oe = oe / ref_oe if np.isfinite(oe) and ref_oe else float("nan")
        rel_slope = slope / ref_slope if np.isfinite(slope) and ref_slope else float("nan")
        band = P["calibration_ratio_band"]
        outside = lambda v: bool(np.isfinite(v) and not (1 / band <= v <= band))
        miscalibrated = outside(rel_oe) or outside(rel_slope)
        metrics["relative_calibration_in_the_large"] = metric(
            rel_oe if np.isfinite(rel_oe) else None, higher_is_worse=False, flag=outside(rel_oe),
            threshold=f"outside [{1/band:.2f}, {band:.2f}] -> Conditional",
            note=f"Observed/expected events of the synthetic-trained logistic probe on real test data, "
                 f"divided by the real-trained value ({ref_oe:.3f}).")
        metrics["relative_calibration_slope"] = metric(
            rel_slope if np.isfinite(rel_slope) else None, higher_is_worse=False,
            flag=outside(rel_slope),
            threshold=f"outside [{1/band:.2f}, {band:.2f}] -> Conditional",
            note=f"Calibration slope of the synthetic-trained logistic probe divided by the "
                 f"real-trained slope ({ref_slope:.3f}).")
        if outcome_levels < 2:
            weaknesses.append("Synthetic outcome collapsed to a single class; no model can be trained.")
        status = aggregate_status([
            "Fail" if outcome_levels < 2 else "Pass",
            primary,
            "Conditional" if (probe_conditional or miscalibrated) else "Pass",
        ])
        return DimensionResult.from_dict({"dimension": "utility_reference_calibrated", "status": status,
                "metrics": metrics, "weaknesses": weaknesses,
                "probe_verdicts": verdicts,
                "provisional_margins": {k: v for k, v in P.items() if k.startswith(("retention", "calibration"))}})
''')

from syntrustbench_campaign import reference

assert reference.PROVISIONAL["coverage_margin"] == .05
assert reference.ReferenceContext.PROBES == ("random_forest","logistic_regression","gradient_boosting")


### Cohort construction and patient-disjoint roles

In [ ]:
_write_module('data.py', r'''"""Earliest-encounter cohort and audited patient-disjoint real-data roles."""
from __future__ import annotations

from dataclasses import dataclass
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from .config import ExperimentConfig, seed_for
from .core import UnavailableAnalysis, file_hash, read_json, save_frame, write_json


def read_analytic(path: str | Path, cfg: ExperimentConfig) -> pd.DataFrame:
    frame = pd.read_csv(path, keep_default_na=False, na_values=[''],
                        dtype={c: 'object' for c in cfg.data.categorical}, float_precision='round_trip')
    if list(frame.columns) != list(cfg.data.columns):
        raise ValueError(f'Unexpected analytic schema: {path}')
    frame[cfg.data.target] = pd.to_numeric(frame[cfg.data.target], errors='raise').astype(int)
    if not set(frame[cfg.data.target].unique()) <= {0, 1}:
        raise ValueError('Outcome must be integer binary data.')
    return frame


def stratified_indices(y: pd.Series, size: int, seed: int) -> np.ndarray:
    if size > len(y) or size < 1:
        raise ValueError('Invalid sample size.')
    if size == len(y):
        return np.random.default_rng(seed).permutation(len(y))
    rows, _ = train_test_split(np.arange(len(y)), train_size=size, stratify=y, random_state=seed)
    return np.asarray(rows)


def split_three(frame: pd.DataFrame, cfg: ExperimentConfig, seed: int) -> list[pd.DataFrame]:
    """Reuse the previous notebook's stratified 40/40/20 algorithm."""
    dev, test = train_test_split(frame, test_size=.2, stratify=frame[cfg.data.target], random_state=seed)
    train, pool = train_test_split(dev, test_size=.5, stratify=dev[cfg.data.target],
                                  random_state=seed_for(seed, 'half'))
    return [x.reset_index(drop=True) for x in (train, pool, test)]


@dataclass
class CohortBuilder:
    cfg: ExperimentConfig

    def _raw(self) -> tuple[dict, dict, list]:
        d = self.cfg.data
        raw = pd.read_csv(d.raw_csv, keep_default_na=False, na_values=['?', ''], dtype='object')
        required = set(d.continuous + d.categorical + ('patient_nbr', 'encounter_id', 'readmitted'))
        if not required <= set(raw):
            raise ValueError(f'Missing columns: {required - set(raw)}')
        if raw[['patient_nbr', 'encounter_id', 'readmitted']].isna().any().any():
            raise ValueError('Patient, encounter and outcome fields must be complete.')
        if not set(raw.readmitted) <= {'<30', '>30', 'NO'}:
            raise ValueError('Unknown readmission label.')
        raw['encounter_id'] = pd.to_numeric(raw.encounter_id, errors='raise')
        if not raw.encounter_id.is_unique:
            raise ValueError('Duplicate encounter IDs need source review.')
        cohort = raw.sort_values('encounter_id', kind='stable').drop_duplicates('patient_nbr').copy()
        cohort[d.target] = (cohort.readmitted == '<30').astype(int)
        for column in d.continuous:
            cohort[column] = pd.to_numeric(cohort[column], errors='raise')
        cohort['_patient_hash'] = cohort.patient_nbr.map(lambda s: hashlib.sha256(str(s).encode()).hexdigest())
        frames = dict(zip(('real_train', 'calibration_pool', 'real_test'), split_three(cohort, self.cfg, d.split_seed)))
        ids = {name: f['_patient_hash'].tolist() for name, f in frames.items()}
        flow = [{'section': 'flow', 'label': 'raw encounters', 'rows': len(raw)},
                {'section': 'flow', 'label': 'later encounters removed', 'rows': len(raw)-len(cohort)},
                {'section': 'flow', 'label': 'earliest encounters', 'rows': len(cohort)}]
        return {k: v[list(d.columns)].copy() for k, v in frames.items()}, ids, flow

    def _frozen(self) -> tuple[dict, dict, list]:
        d = self.cfg.data
        manifest = read_json(d.frozen_manifest)
        hashes = manifest.get('split_hashes', manifest.get('hashes', {}))
        frames = {}
        for name in ('real_train', 'calibration_pool', 'real_test'):
            path = Path(d.frozen_dir) / f'{name}.csv'
            if name not in hashes or file_hash(path) != hashes[name]:
                raise ValueError(f'Frozen split hash missing/mismatched: {name}')
            frames[name] = read_analytic(path, self.cfg)
        audit_path = Path(manifest.get('patient_audit', Path(d.frozen_dir).resolve() / 'patient_role_audit.csv'))
        if not audit_path.is_absolute():
            audit_path = Path(d.frozen_manifest).resolve().parent / audit_path
        if not audit_path.is_file():
            raise UnavailableAnalysis('Frozen splits need row-aligned hashed patient_role_audit.csv; no resplitting fallback.')
        if manifest.get('patient_audit_sha256') and file_hash(audit_path) != manifest['patient_audit_sha256']:
            raise ValueError('Patient audit hash mismatch.')
        audit = pd.read_csv(audit_path)
        ids = {}
        for name, frame in frames.items():
            rows = audit[audit.role == name].sort_values('analytic_row')
            if rows.analytic_row.tolist() != list(range(len(frame))):
                raise ValueError(f'Patient audit row alignment invalid: {name}')
            ids[name] = rows.patient_id_sha256.astype(str).tolist()
        flow = [{'section': 'flow', 'label': 'frozen earliest-encounter cohort',
                 'rows': sum(map(len, frames.values())), 'note': 'Upstream raw encounter flow not reconstructed.'}]
        return frames, ids, flow

    def build(self, folder: Path) -> dict:
        frames, ids, rows = self._frozen() if self.cfg.data.frozen_dir else self._raw()
        flattened = sum(ids.values(), [])
        if len(set(flattened)) != len(flattened):
            raise ValueError('Repeated patient hashes within or across frozen roles.')
        paths, patient_rows, missing, groups = {}, [], [], []
        for name, frame in frames.items():
            if self.cfg.execution.smoke_mode:
                indices = stratified_indices(frame[self.cfg.data.target], min(len(frame), self.cfg.execution.smoke_rows), seed_for('smoke', name))
                ids[name] = [ids[name][i] for i in indices]
                frame = frame.iloc[indices].reset_index(drop=True)
            paths[name] = save_frame(frame, folder / f'{name}.csv')
            patient_rows.extend({'role': name, 'analytic_row': i, 'patient_id_sha256': p} for i, p in enumerate(ids[name]))
            rows.append({'section': 'split', 'label': name, 'rows': len(frame),
                         'positive': int(frame[self.cfg.data.target].sum()),
                         'prevalence': float(frame[self.cfg.data.target].mean())})
            missing.extend({'role': name, 'column': c, 'missing': int(frame[c].isna().sum()),
                            'rows': len(frame), 'fraction': float(frame[c].isna().mean())} for c in frame)
            for attribute in self.cfg.data.protected:
                groups.extend({'role': name, 'attribute': attribute, 'level': str(level),
                               'rows': len(group), 'positive': int(group[self.cfg.data.target].sum())}
                              for level, group in frame.groupby(attribute, dropna=False))
        audit = save_frame(pd.DataFrame(patient_rows), folder / 'patient_role_audit.csv')
        for name, value in [('cohort_split_summary', rows), ('missingness', missing), ('subgroup_counts', groups)]:
            save_frame(pd.DataFrame(value), folder / f'{name}.csv')
        result = {'paths': paths, 'split_hashes': {k: file_hash(p) for k, p in paths.items()},
                  'patient_audit': audit, 'patient_audit_sha256': file_hash(audit), 'patient_overlap': 0,
                  'folder': str(folder), 'schema': list(self.cfg.data.columns),
                  'cohort_rule': 'minimum numeric encounter_id per patient; readmitted == <30',
                  'smoke_mode': self.cfg.execution.smoke_mode}
        write_json(folder / 'split_manifest.json', result)
        return result


@dataclass
class RoleAssigner:
    cfg: ExperimentConfig

    def assign(self, frames: list[pd.DataFrame], ids: list[list[str]], folder: Path, key: str) -> dict:
        train, pool, test = frames
        ti, pi, ei = ids
        a, b = train_test_split(np.arange(len(pool)), test_size=.5,
                                stratify=pool[self.cfg.data.target], random_state=seed_for('roles', key, 'reference'))
        n = min(len(a), len(b), len(train))
        a, b = a[:n], b[:n]
        floor = stratified_indices(train[self.cfg.data.target], n, seed_for('roles', key, 'floor'))
        tables = {'train': train, 'reference': pool.iloc[a], 'positive': pool.iloc[b],
                  'test': test, 'floor': train.iloc[floor]}
        roles = {'train': ti, 'reference': [pi[i] for i in a], 'positive': [pi[i] for i in b],
                 'test': ei, 'floor': [ti[i] for i in floor]}
        independent = [roles[k] for k in ('train', 'reference', 'positive', 'test')]
        if len(set(sum(independent, []))) != sum(map(len, independent)):
            raise ValueError('Patient overlap between independent roles.')
        paths = {k: save_frame(v.reset_index(drop=True), folder / f'{k}.csv') for k, v in tables.items()}
        audit = save_frame(pd.DataFrame([{'role': k, 'analytic_row': i, 'patient_id_sha256': p}
                            for k, values in roles.items() for i, p in enumerate(values)]), folder / 'role_patients.csv')
        result = {'paths': paths, 'patient_audit': audit, 'n': n, 'key': key,
                  'role_counts': {k: len(v) for k, v in tables.items()}, 'independent_role_overlap': 0,
                  'floor_subset_of_train': True, 'omitted_pool_rows': len(pool)-2*n,
                  'main_test_used': key == 'main'}
        write_json(folder / 'roles.json', result)
        return result

    def main(self, cohort: dict, folder: Path) -> dict:
        names = ('real_train', 'calibration_pool', 'real_test')
        audit = pd.read_csv(cohort['patient_audit'])
        frames = [read_analytic(cohort['paths'][n], self.cfg) for n in names]
        ids = [audit[audit.role == n].sort_values('analytic_row').patient_id_sha256.tolist() for n in names]
        return self.assign(frames, ids, folder, 'main')

    def null(self, cohort: dict, repeat: int | str, folder: Path) -> dict:
        # E is deliberately never read here.
        names = ('real_train', 'calibration_pool')
        audit = pd.read_csv(cohort['patient_audit'])
        development = pd.concat([read_analytic(cohort['paths'][n], self.cfg) for n in names], ignore_index=True)
        development['_patient_hash'] = sum([audit[audit.role == n].sort_values('analytic_row').patient_id_sha256.tolist()
                                           for n in names], [])
        # The salt gives v0.4.1 fresh null draws, disjoint in seed from the v0.4 runs whose
        # audit motivated the calibration change, so the new audit is not circular.
        salt = self.cfg.calibration.null_seed_salt
        parts = split_three(development, self.cfg, seed_for('null', self.cfg.calibration.assignment_seed, salt, repeat))
        ids = [p['_patient_hash'].tolist() for p in parts]
        frames = [p[list(self.cfg.data.columns)] for p in parts]
        result = self.assign(frames, ids, folder, f'null-{salt}-{repeat}')
        result.update(main_test_used=False, scope='development-scaled; development patients reused across runs')
        return result


def self_test() -> None:
    y = pd.Series([0, 1] * 30)
    assert len(set(stratified_indices(y, 20, 7))) == 20
''')

from syntrustbench_campaign import data

data.self_test()


### Generator strategies

In [ ]:
_write_module('generators.py', r'''"""Pinned SDV adapters with explicit fit/sample seeds and CPU-safe device choice."""
from __future__ import annotations

from abc import ABC, abstractmethod
import inspect
import random
import time
import numpy as np
import pandas as pd

from .config import ExperimentConfig


def seed_all(seed: int) -> None:
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class Generator(ABC):
    def __init__(self, cfg: ExperimentConfig, seed: int, smoke: bool = False):
        self.cfg, self.seed, self.smoke = cfg, seed, smoke
        self.model = None
        self.timing = {}
        self.provenance = {'formal_privacy': {'status': 'NotApplicable', 'reason': 'Not a DP generator.'}}
        self.artifact_dir = None

    def save(self, path) -> None:
        self.model.save(str(path))

    @abstractmethod
    def synthesizer_class(self):
        raise NotImplementedError

    def fit(self, train: pd.DataFrame) -> Generator:
        from sdv.metadata import SingleTableMetadata
        import torch
        seed_all(self.seed)
        metadata = SingleTableMetadata()
        for c in self.cfg.data.columns:
            metadata.add_column(column_name=c, sdtype='numerical' if c in self.cfg.data.continuous else 'categorical')
        cls = self.synthesizer_class()
        options = {'enforce_min_max_values': self.cfg.generators.enforce_min_max_values,
                   'enforce_rounding': self.cfg.generators.enforce_rounding}
        gpu = self.cfg.generators.allow_cuda and torch.cuda.is_available()
        if cls.__name__ != 'GaussianCopulaSynthesizer':
            gpu_parameter = 'enable_gpu' if 'enable_gpu' in inspect.signature(cls).parameters else 'cuda'
            options[gpu_parameter] = bool(gpu)  # Apple Silicon stays CPU by default.
            options['epochs'] = self.cfg.execution.smoke_epochs if (self.smoke or self.cfg.execution.smoke_mode) else self.cfg.generators.epochs
        self.model = cls(metadata, **options)
        start = time.perf_counter(); self.model.fit(train)
        self.timing.update(fit_seconds=time.perf_counter()-start, device='CUDA' if gpu else 'CPU', parameters=options)
        self.schema = list(train.columns)
        return self

    def sample(self, n: int, seed: int) -> pd.DataFrame:
        if self.model is None:
            raise RuntimeError('Fit before sampling.')
        seed_all(seed); self.model.reset_sampling()
        if not hasattr(self.model, '_set_random_state'):
            raise RuntimeError('Pinned SDV sampling API changed.')
        self.model._set_random_state(seed)
        start = time.perf_counter()
        frame = self.model.sample(num_rows=n)[self.schema]
        self.timing.update(sample_seconds=time.perf_counter()-start, sample_seed=seed)
        if len(frame) != n:
            raise ValueError('Generator returned the wrong row count.')
        return frame


class GaussianCopulaGenerator(Generator):
    def synthesizer_class(self):
        from sdv.single_table import GaussianCopulaSynthesizer
        return GaussianCopulaSynthesizer


class CTGANGenerator(Generator):
    def synthesizer_class(self):
        from sdv.single_table import CTGANSynthesizer
        return CTGANSynthesizer


class TVAEGenerator(Generator):
    def synthesizer_class(self):
        from sdv.single_table import TVAESynthesizer
        return TVAESynthesizer


def generator(name: str, cfg: ExperimentConfig, seed: int, smoke: bool = False) -> Generator:
    from .dp_generators import DPAIMGenerator, DPMSTGenerator
    from .tabddpm import TabDDPMGenerator
    return {'GaussianCopula': GaussianCopulaGenerator, 'CTGAN': CTGANGenerator,
            'TVAE': TVAEGenerator, 'DPAIM': DPAIMGenerator, 'DPMST': DPMSTGenerator,
            'TabDDPM': TabDDPMGenerator}[name](cfg, seed, smoke)


def self_test() -> None:
    assert len({GaussianCopulaGenerator, CTGANGenerator, TVAEGenerator}) == 3
''')

from syntrustbench_campaign import generators

generators.self_test()


### DP generator extension: public preprocessing and separate privacy accounting

DPAIM is always included. `generators.enable_dpmst` and `generators.enable_tabddpm` default to `False`. The original SDV adapters, Fidelity/Utility calculations, and frozen v0.3 evaluator are retained. Injections still use only positive and Gaussian Copula baselines.

**Inputs:** configured public integer bounds, overflow intervals and complete categorical domains. **Outputs:** fixed-bin manifests, SmartNoise fits and seeded postprocessed samples. **Success criterion:** no private minima, maxima, quantiles or observed level sets determine preprocessing; output columns/row counts and clinical constraints hold.

SmartNoise 1.0.8 supplies AIM/MST. Epsilon defaults to 1.0; delta defaults to `1/(10*n_train)` and is recorded per fit. Numeric missing values decode to the public lower bound; overflow values use a fixed finite overflow interval. Diagnosis codes use a fixed ICD-9 prefix domain, with details in JUDGEMENT_CALLS.md. These transformations affect only DP generator inputs/outputs.

Fresh private fit randomness is retained. Seeds 17/29/43 label independent fits; sampling and bin decoding from an existing model are seed-deterministic. Resumption reuses the hashed fit/sample rather than recreating its private random tape. `generator_formal_privacy.csv` reports the nominal mechanism guarantee separately from empirical privacy tests. The overall campaign, raw inputs and evaluation artifacts are not a DP release.

In [ ]:
_write_module('dp_generators.py', r'''"""SmartNoise AIM/MST adapters with entirely public discrete domains.

Fit noise is fresh and private. Public seeds control postprocessing only.
The model's nominal privacy accounting does not cover campaign diagnostics.
"""
from __future__ import annotations

import contextlib
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd

from .config import ExperimentConfig
from .generators import Generator


class PublicDiscretizer:
    """Data-independent bins; integer intervals inclusive at both ends."""

    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.bins = {}
        for col, lower, cap, width, overflow in cfg.generators.dp_domains:
            intervals = [(lo, min(lo + width - 1, cap)) for lo in range(lower, cap + 1, width)]
            if overflow > cap:
                intervals.append((cap + 1, overflow))
            self.bins[col] = intervals
        self.categories = {c: levels + (('__MISSING__', '__OTHER__') if c != cfg.data.target else ())
                           for c, levels in cfg.generators.dp_categories}
        if set(self.bins) != set(cfg.data.continuous) or set(self.categories) != set(cfg.data.categorical + (cfg.data.target,)):
            raise ValueError('Every analytic column needs an explicit public DP domain.')
        if self.categories[cfg.data.target] != ('0','1'):
            raise ValueError('The public DP outcome domain must be exactly 0/1.')
        if self.bins['time_in_hospital'][0][0] != 1 or self.bins['time_in_hospital'][-1][1] != 14:
            raise ValueError('The public hospital-stay domain must enforce 1–14 days.')
        self.cards = [len(self.bins[c]) + 1 if c in self.bins else len(self.categories[c]) for c in cfg.data.columns]

    def manifest(self) -> dict:
        value = {'integer_intervals_inclusive': self.bins, 'categories': self.categories,
                 'cardinalities': dict(zip(self.cfg.data.columns, self.cards)),
                 'missing_numeric_bin': 'extra final bin; decoded to fixed public lower bound',
                 'overflow': 'above cap uses final interval; above overflow upper also uses that interval',
                 'diagnosis': 'public ICD-9 prefix before decimal; numeric leading zeroes normalized',
                 'source': 'UCI data dictionary and analyst-declared public count caps; no training statistics',
                 'preprocessing_epsilon': 0.0}
        value['schema_sha256'] = hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()
        return value

    def encode(self, frame: pd.DataFrame) -> pd.DataFrame:
        if list(frame.columns) != list(self.cfg.data.columns):
            raise ValueError('DP input schema/order differs from configured analytic columns.')
        result = {}
        for c in self.cfg.data.columns:
            if c in self.bins:
                x = pd.to_numeric(frame[c], errors='raise').to_numpy(dtype=float)
                highs = np.array([hi for _, hi in self.bins[c]])
                codes = np.searchsorted(highs, np.rint(x), side='left').clip(0, len(highs)-1)
                codes[~np.isfinite(x)] = len(highs)
                result[c] = codes.astype(int)
            else:
                vals = frame[c].map(lambda x: '__MISSING__' if pd.isna(x) or str(x) in ('', '?') else str(x))
                if c == 'diag_1':
                    vals = vals.map(lambda s: str(int(s.split('.')[0])) if s.split('.')[0].isdigit()
                                    else s.split('.')[0].upper())
                if c == self.cfg.data.target:
                    vals = vals.replace({'0.0':'0', '1.0':'1'})
                    if not vals.isin(self.categories[c]).all():
                        raise ValueError('DP outcome must be binary and nonmissing.')
                lookup = {v:i for i,v in enumerate(self.categories[c])}
                result[c] = vals.map(lambda v: lookup.get(v, lookup.get('__OTHER__'))).astype(int)
        return pd.DataFrame(result, columns=self.cfg.data.columns)

    def decode(self, frame: pd.DataFrame, seed: int) -> pd.DataFrame:
        rng = np.random.default_rng(seed)
        result = {}
        for c, card in zip(self.cfg.data.columns, self.cards):
            codes = frame[c].to_numpy(dtype=int)
            if ((codes < 0) | (codes >= card)).any():
                raise ValueError('DP synthesizer returned an out-of-domain category.')
            if c in self.bins:
                intervals = self.bins[c] + [(self.bins[c][0][0], self.bins[c][0][0])]
                lo = np.array([intervals[i][0] for i in codes])
                hi = np.array([intervals[i][1] for i in codes])
                result[c] = rng.integers(lo, hi + 1)
            elif c == self.cfg.data.target:
                result[c] = np.array([int(self.categories[c][i]) for i in codes])
            else:
                result[c] = [np.nan if self.categories[c][i] == '__MISSING__' else self.categories[c][i] for i in codes]
        return pd.DataFrame(result, columns=self.cfg.data.columns)

    def transformer(self):
        # Fit labels ONLY on the complete public integer domain, never private rows.
        from snsynth.transform import LabelTransformer, TableTransformer
        transforms = []
        for card in self.cards:
            t = LabelTransformer(nullable=False)
            t.fit(list(range(card)))
            transforms.append(t)
        table = TableTransformer(transforms)
        assert table.fit_complete and not table.needs_epsilon and table.cardinality == self.cards
        return table


@contextlib.contextmanager
def private_fit_randomness():
    """Do not expose the upstream selector's random tape through public job seeds."""
    state = np.random.get_state()
    np.random.seed(np.frombuffer(os.urandom(128), dtype=np.uint32))
    try:
        yield
    finally:
        np.random.set_state(state)


class DPAIMGenerator(Generator):
    mechanism = 'AIM'

    def synthesizer_class(self):
        from snsynth.aim import AIMSynthesizer
        return AIMSynthesizer

    def fit(self, train: pd.DataFrame) -> DPAIMGenerator:
        import jax
        jax.config.update('jax_enable_x64', True)
        if importlib.metadata.version('smartnoise-synth') != '1.0.8':
            raise RuntimeError('SmartNoise version differs from the pinned adapter.')
        start = time.perf_counter()
        self.discretizer = PublicDiscretizer(self.cfg)
        discrete = self.discretizer.encode(train)
        public = self.discretizer.manifest()
        n = len(train)
        if n == 0:
            raise ValueError('Cannot fit DP generator to zero rows.')
        delta = self.cfg.generators.delta if self.cfg.generators.delta is not None else 1/(10*n)
        options = {'epsilon': self.cfg.generators.epsilon, 'delta': delta, 'verbose': False}
        if self.mechanism == 'AIM':
            options.update(rounds=self.cfg.generators.dp_smoke_rounds if (self.smoke or self.cfg.execution.smoke_mode)
                           else self.cfg.generators.dp_aim_rounds,
                           max_model_size=self.cfg.generators.dp_max_model_size_mb,
                           max_cells=self.cfg.generators.dp_max_cells, degree=2)
        self.model = self.synthesizer_class()(**options)
        with private_fit_randomness():
            self.model.fit(discrete, transformer=self.discretizer.transformer(), preprocessor_eps=0.0)
        if self.model._transformer.cardinality != self.discretizer.cards:
            raise RuntimeError('Private data altered the public domain cardinalities.')
        self.schema = list(train.columns)
        self.timing.update(fit_seconds=time.perf_counter()-start, device='CPU', parameters=options)
        self.provenance = {'public_preprocessing': public,
            'formal_privacy': {'status': 'NominalMechanismGuarantee', 'mechanism': self.mechanism,
                'epsilon': options['epsilon'], 'delta': delta, 'n_train_public': n,
                'delta_rule': 'configured' if self.cfg.generators.delta is not None else '1/(10*n_train)',
                'preprocessing_epsilon': 0.0, 'library': 'smartnoise-synth==1.0.8',
                'scope': 'One fitted synthesizer and its postprocessed samples; upstream unit-row accounting with public domain and sample size.',
                'randomness': 'Fresh unrecorded fit randomness and OpenDP Gaussian noise; public seed controls sampling only.',
                'exclusions': 'Raw inputs, hashes, exact cohort counts, all evaluator reports, other generators and joint multi-fit releases are not covered.',
                'assumptions': 'Public n and categorical schema; one record per patient; upstream implementation/accountant correctness. Not an independent DP certification.',
                'empirical_privacy_is_separate': True}}
        return self

    def sample(self, n: int, seed: int) -> pd.DataFrame:
        if self.model is None:
            raise RuntimeError('Fit before sampling.')
        start = time.perf_counter()
        state = np.random.get_state()
        try:
            np.random.seed(seed)
            discrete = pd.DataFrame(self.model.sample(n), columns=self.schema)
        finally:
            np.random.set_state(state)
        frame = self.discretizer.decode(discrete, seed)
        if len(frame) != n or list(frame.columns) != self.schema:
            raise ValueError('DP output schema or row count changed.')
        self.timing.update(sample_seconds=time.perf_counter()-start, sample_seed=seed)
        return frame

    def save(self, path: Path) -> None:
        import cloudpickle
        with Path(path).open('wb') as handle:
            cloudpickle.dump({'model':self.model, 'public_preprocessing':self.discretizer.manifest()}, handle)


class DPMSTGenerator(DPAIMGenerator):
    mechanism = 'MST'

    def synthesizer_class(self):
        from snsynth.mst import MSTSynthesizer
        return MSTSynthesizer


def self_test() -> None:
    schema = PublicDiscretizer(ExperimentConfig())
    assert schema.bins['time_in_hospital'] == [(i,i) for i in range(1,15)]
    assert schema.manifest()['preprocessing_epsilon'] == 0
''')

from syntrustbench_campaign import dp_generators

dp_generators.self_test()


### Optional isolated TabDDPM

**Inputs:** an enabled flag, Python 3.10 executable, and the embedded fully pinned/hash-checked requirements. **Outputs:** an isolated Synthcity 0.2.12 environment, CSV exchanges, fit/sample artifacts and environment/source hashes. **Success criterion:** the `ddpm` plugin runs without importing Synthcity into the SDV process, or environment setup becomes explicitly Unavailable with logs. The remaining campaign continues.

The first enabled run may build a large environment. Set `generators.tabddpm_python` if `python3.10` is not on PATH. CPU is used. A failed build is cached; after resolving it, `retry_failed=True` permits another attempt. Unexpected scientific errors after a successful environment probe are still logged and re-raised. The hash-locked requirements are compressed in the next cell solely to keep the embedded notebook manageable; the readable file is included in the bundle.

In [ ]:
import base64, zlib

_write_module('tabddpm-requirements.txt', zlib.decompress(base64.b64decode('...')).decode())

_write_module('tabddpm_worker.py', r'''"""Isolated Python 3.10 entry point. No imports from the main SDV environment."""
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import sys
import time


def provenance() -> dict:
    if sys.version_info[:2] != (3, 10):
        raise RuntimeError('TabDDPM requires isolated Python 3.10.')
    import synthcity
    from synthcity.plugins.generic.plugin_ddpm import TabDDPMPlugin
    assert TabDDPMPlugin.name() == 'ddpm'
    package = Path(synthcity.__file__).parent
    return {'python':platform.python_version(),
            'packages':{d.metadata['Name'].lower():d.version for d in importlib.metadata.distributions()},
            'synthcity_source_sha256':{str(p.relative_to(package)):hashlib.sha256(p.read_bytes()).hexdigest()
                                     for p in sorted(package.rglob('*.py'))}}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('operation', choices=('probe','fit','sample'))
    parser.add_argument('request')
    parser.add_argument('response')
    args = parser.parse_args()
    request = json.loads(Path(args.request).read_text())
    if args.operation == 'probe':
        result = provenance()
    else:
        import numpy as np
        import pandas as pd
        import torch
        from synthcity.plugins import Plugins
        from synthcity.plugins.core.dataloader import GenericDataLoader
        from synthcity.utils.serialization import load_from_file, save_to_file
        from synthcity.utils.reproducibility import enable_reproducible_results
        torch.set_num_threads(request['threads'])
        enable_reproducible_results(request['seed'])
        started = time.perf_counter()
        if args.operation == 'fit':
            data = pd.read_csv(request['input_csv'],keep_default_na=False,na_values=[''],
                               dtype={c:'object' for c in request['categorical']})
            # Optional plugin cannot consume missing numeric/categorical values.
            # Fixed imputation, unrelated to DP and confined to its own training copy.
            for c in request['categorical']:
                data[c] = data[c].fillna('__MISSING__').astype(str)
            for c in request['continuous']:
                data[c] = pd.to_numeric(data[c],errors='raise').fillna(0.)
            model = Plugins(categories=['generic']).get('ddpm', n_iter=request['epochs'],
                num_timesteps=request['timesteps'], batch_size=min(256,len(data)),
                random_state=request['seed'], device=torch.device('cpu'),
                is_classification=True, workspace=Path(request['workspace']))
            model.fit(GenericDataLoader(data,target_column=request['target']))
            save_to_file(request['model'],model)
            result = {'fit_seconds':time.perf_counter()-started,'device':'CPU'}
        else:
            model = load_from_file(request['model'])
            output = model.generate(request['n'],random_state=request['seed']).dataframe()
            if len(output)!=request['n'] or set(output.columns)!=set(request['columns']):
                raise ValueError('TabDDPM returned wrong schema or row count.')
            for c in request['categorical']:
                output[c] = output[c].replace('__MISSING__',np.nan)
            output[request['columns']].to_csv(request['output_csv'],index=False)
            result = {'sample_seconds':time.perf_counter()-started,'sample_seed':request['seed']}
    Path(args.response).write_text(json.dumps(result,indent=2)+'\n')


if __name__ == '__main__':
    main()
''')

from syntrustbench_campaign import tabddpm_worker

_write_module('tabddpm.py', r'''"""Optional Synthcity adapter: pinned subprocess environment, explicit unavailability."""
from __future__ import annotations

import fcntl
import os
from pathlib import Path
import shutil
import subprocess
import uuid

import pandas as pd

from .config import ExperimentConfig
from .core import UnavailableAnalysis, file_hash, read_json, save_frame, write_json
from .generators import Generator


class TabDDPMEnvironment:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.requirements = Path(__file__).with_name('tabddpm-requirements.txt')
        self.worker = Path(__file__).with_name('tabddpm_worker.py')
        self.digest = file_hash(self.requirements)
        self.root = Path(cfg.execution.output_root).resolve()/'optional_environments'/('tabddpm-'+self.digest[:16])
        self.python = self.root/'venv/bin/python'

    def run(self, operation: str, request: dict, folder: Path) -> dict:
        req = write_json(folder/f'{operation}_request.json',request)
        response = folder/f'{operation}_response.json'
        env = dict(os.environ, PYTHONHASHSEED=str(request.get('seed',0)),
                   OMP_NUM_THREADS=str(self.cfg.execution.threads_per_worker),
                   OPENBLAS_NUM_THREADS=str(self.cfg.execution.threads_per_worker))
        env.pop('PYTHONPATH',None)
        with (folder/f'{operation}_subprocess.log').open('w') as log:
            subprocess.run([str(self.python),'-I',str(self.worker),operation,req,str(response)],
                cwd=str(folder),env=env,stdout=log,stderr=subprocess.STDOUT,check=True,
                timeout=self.cfg.generators.tabddpm_timeout_seconds)
        return read_json(response)

    def ensure(self) -> dict:
        self.root.mkdir(parents=True,exist_ok=True)
        with (self.root/'build.lock').open('a') as lock:
            fcntl.flock(lock,fcntl.LOCK_EX)
            failure = self.root/'unavailable.json'
            if failure.exists() and not self.cfg.execution.retry_failed:
                raise UnavailableAnalysis(read_json(failure)['reason'])
            attempt = self.root/('check-'+uuid.uuid4().hex)
            attempt.mkdir()
            try:
                if not (self.root/'environment.json').exists():
                    if not self.cfg.generators.tabddpm_build_environment:
                        raise UnavailableAnalysis('TabDDPM environment absent and automatic build disabled.')
                    with (attempt/'build.log').open('w') as log:
                        if not self.python.exists():
                            subprocess.run([self.cfg.generators.tabddpm_python,'-m','venv',str(self.root/'venv')],
                                stdout=log,stderr=subprocess.STDOUT,check=True,timeout=120)
                        subprocess.run([str(self.python),'-c',
                            'import sys; assert sys.version_info[:2] == (3,10), "Python 3.10 required"'],
                            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=30)
                        # Every resolved dependency is version-pinned with allowed wheel/sdist hashes.
                        subprocess.run([str(self.python),'-m','pip','install','--require-hashes','--no-deps',
                            '-r',str(self.requirements)],stdout=log,stderr=subprocess.STDOUT,
                            check=True,timeout=self.cfg.generators.tabddpm_timeout_seconds)
                        subprocess.run([str(self.python),'-m','pip','check'],
                            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=60)
                current = self.run('probe',{},attempt)
            except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
                reason = f'TabDDPM Python 3.10 environment unavailable: {type(exc).__name__}: {exc}; logs: {attempt}'
                logs = sorted(attempt.glob('*.log'),key=lambda p:p.stat().st_mtime)
                if logs:
                    reason += '\nDiagnostic tail: ' + logs[-1].read_text(errors='replace')[-2000:]
                write_json(failure,{'status':'Unavailable','reason':reason})
                raise UnavailableAnalysis(reason) from exc
            snapshot = self.root/'environment.json'
            if snapshot.exists() and read_json(snapshot) != current:
                raise UnavailableAnalysis('TabDDPM environment/source drift; use a new isolated environment.')
            write_json(snapshot,current)
            if failure.exists():
                failure.unlink()
            return {'requirements_sha256':self.digest,'worker_sha256':file_hash(self.worker),
                    'manifest':current,'manifest_sha256':file_hash(snapshot)}


class TabDDPMGenerator(Generator):
    def synthesizer_class(self):
        return 'synthcity.plugins.generic.plugin_ddpm.TabDDPMPlugin (isolated)'

    def fit(self, train: pd.DataFrame) -> TabDDPMGenerator:
        self.environment = TabDDPMEnvironment(self.cfg)
        self.folder = Path(self.artifact_dir or self.cfg.root/('manual_tabddpm-'+uuid.uuid4().hex)).resolve()
        self.folder.mkdir(parents=True,exist_ok=True)
        try:
            provenance = self.environment.ensure()
        except UnavailableAnalysis:
            logs = {str(p):file_hash(p) for p in sorted(self.environment.root.rglob('*.log'))}
            write_json(self.folder/'optional_environment_unavailable.json',
                       {'requirements_sha256':self.environment.digest, 'build_log_hashes':logs})
            raise
        input_path = save_frame(train,self.folder/'tabddpm_train.csv')
        self.schema = list(train.columns)
        self.model_path = self.folder/'tabddpm_model.pkl'
        small = self.smoke or self.cfg.execution.smoke_mode
        self.request = {'input_csv':input_path,'model':str(self.model_path),
            'workspace':str(self.folder/'synthcity_workspace'),
            'categorical':list(self.cfg.data.categorical),'continuous':list(self.cfg.data.continuous),
            'columns':self.schema,'target':self.cfg.data.target,
            'threads':self.cfg.execution.threads_per_worker,
            'seed':self.seed,'epochs':1 if small else self.cfg.generators.tabddpm_epochs,
            'timesteps':10 if small else self.cfg.generators.tabddpm_timesteps}
        self.timing = self.environment.run('fit',self.request,self.folder)
        self.timing['parameters'] = {k:self.request[k] for k in ('epochs','timesteps','threads')}
        self.provenance.update(isolated_environment=provenance,
            preprocessing='Categorical missing token; numeric missing -> 0; no changes to other generators.')
        self.model = str(self.model_path)
        return self

    def sample(self, n: int, seed: int) -> pd.DataFrame:
        if self.model is None:
            raise RuntimeError('Fit before sampling.')
        output = self.folder/'tabddpm_sample.csv'
        self.timing.update(self.environment.run('sample',dict(self.request,n=n,seed=seed,output_csv=str(output)),self.folder))
        frame = pd.read_csv(output,keep_default_na=False,na_values=[''],dtype={c:'object' for c in self.cfg.data.categorical})
        if len(frame)!=n or list(frame.columns)!=self.schema:
            raise ValueError('TabDDPM CSV exchange violated its schema contract.')
        return frame

    def save(self, path: Path) -> None:
        shutil.copyfile(self.model_path,path)


def self_test() -> None:
    cfg = ExperimentConfig()
    assert not cfg.generators.enable_tabddpm and not cfg.generators.enable_dpmst
''')

from syntrustbench_campaign import tabddpm

tabddpm.self_test()


### Nested perturbations and controls

In [ ]:
_write_module('perturbations.py', r'''"""Nested perturbation strategies, adapted from the supplied v0.3 notebook."""
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass
import numpy as np
import pandas as pd

from .config import ExperimentConfig, seed_for
from .core import UnavailableAnalysis


def changed_cells(before: pd.DataFrame, after: pd.DataFrame) -> set:
    same = before.eq(after) | (before.isna() & after.isna())
    return set(map(tuple, np.argwhere(~same.to_numpy())))


@dataclass
class Perturbation(ABC):
    cfg: ExperimentConfig
    train: pd.DataFrame

    @abstractmethod
    def change(self, table: pd.DataFrame, dose: float, seed: int) -> tuple[pd.DataFrame, dict]:
        raise NotImplementedError

    def apply(self, table: pd.DataFrame, dose: float, seed: int) -> tuple[pd.DataFrame, dict]:
        if not 0 <= dose <= 1:
            raise ValueError('Dose must be in [0, 1].')
        if dose == 0:
            out, audit = table.copy(deep=True), {'realized_dose': 0., 'selected_rows': []}
        else:
            out, audit = self.change(table.copy(deep=True), dose, seed)
        if out.shape != table.shape or list(out) != list(table):
            raise ValueError('Perturbation changed the schema or row count.')
        cells = changed_cells(table, out)
        return out, {'requested_dose': dose, 'changed_cell_count': len(cells),
                     'changed_cells': sorted([list(p) for p in cells]), **audit}


class ExactCopy(Perturbation):
    def change(self, table, dose, seed):
        rng = np.random.default_rng(seed)
        order = rng.permutation(len(table))
        donors = rng.integers(0, len(self.train), len(table))
        rows = order[:int(np.ceil(dose * len(table)))]
        from syntrustbench.config import load_config
        from syntrustbench.validation import canonical_row_keys
        config = load_config(self.cfg.benchmark())
        keys = set(canonical_row_keys(self.train, config))
        before = np.isin(canonical_row_keys(table, config), list(keys))
        table.iloc[rows] = self.train.iloc[donors[:len(rows)]].to_numpy()
        after = np.isin(canonical_row_keys(table, config), list(keys))
        return table, {'realized_dose': float((after & ~before).mean()),
                       'total_copy_rate': float(after.mean()), 'baseline_copy_rate': float(before.mean()),
                       'selected_rows': rows.tolist(), 'donor_rows': donors[:len(rows)].tolist(),
                       'denominator': 'all rows; newly copied rows'}


class SubgroupSuppression(Perturbation):
    def change(self, table, dose, seed):
        s = self.cfg.injections
        mask = table[s.subgroup_column].eq(s.subgroup_value).to_numpy()
        members, others = np.flatnonzero(mask), np.flatnonzero(~mask)
        if not len(members) or not len(others):
            raise UnavailableAnalysis('Suppression needs target subgroup and replacement donors.')
        rng = np.random.default_rng(seed)
        order = rng.permutation(members)
        donors = rng.choice(others, len(members), replace=True)
        rows = order[:int(np.ceil(dose * len(members)))]
        table.iloc[rows] = table.iloc[donors[:len(rows)]].to_numpy()
        return table, {'realized_dose': len(rows)/len(members), 'selected_rows': rows.tolist(),
                       'donor_rows': donors[:len(rows)].tolist(), 'baseline_subgroup_n': len(members),
                       'remaining_subgroup_n': len(members)-len(rows), 'denominator': 'baseline target-group rows'}


class FeatureShuffle(Perturbation):
    def change(self, table, dose, seed):
        column = self.cfg.injections.shuffle_feature
        original = table[column].copy()
        order = np.random.default_rng(seed).permutation(len(table))
        previous = 0
        doses = dict(self.cfg.injections.doses)['feature_shuffle']
        if dose not in doses:
            raise ValueError('Feature-shuffle dose must be in the frozen nested grid.')
        for level in doses:
            end = int(np.ceil(level * len(table)))
            block = order[previous:end]
            if level <= dose and len(block) > 1:
                table.loc[table.index[block], column] = np.roll(original.iloc[block].to_numpy(), 1)
            previous = end
        same = original.eq(table[column]) | (original.isna() & table[column].isna())
        return table, {'realized_dose': float((~same).mean()),
                       'selected_rows': order[:int(np.ceil(dose*len(table)))].tolist(),
                       'denominator': 'all rows; fixed disjoint rotation blocks preserve marginals; ties remain'}


class MCARMissingness(Perturbation):
    def change(self, table, dose, seed):
        selected, total, count = {}, 0, 0
        for column in self.cfg.injections.missing_features:
            eligible = np.flatnonzero(table[column].notna().to_numpy())
            order = np.random.default_rng(seed_for('mcar', seed, column)).permutation(eligible)
            rows = order[:int(np.ceil(dose * len(eligible)))]
            table.loc[table.index[rows], column] = np.nan
            selected[column] = rows.tolist(); total += len(eligible); count += len(rows)
        if not total:
            raise UnavailableAnalysis('No observed cells available for MCAR injection.')
        return table, {'realized_dose': count/total, 'selected_cells_by_column': selected,
                       'denominator': 'initially observed cells in the four prespecified predictors'}


class TrainCopy(Perturbation):
    def change(self, table, dose, seed):
        if len(table) > len(self.train):
            raise UnavailableAnalysis('Training data smaller than comparator.')
        out = self.train.sample(n=len(table), random_state=seed).reset_index(drop=True)
        return out, {'realized_dose': 1., 'denominator': 'all rows copied from generator training set'}


class IndependentColumns(Perturbation):
    def change(self, table, dose, seed):
        out = pd.DataFrame({c: table[c].sample(frac=1, random_state=seed_for(seed, c)).to_numpy() for c in table})
        return out, {'realized_dose': 1., 'denominator': 'independently permuted columns; exact marginals retained'}


def strategy(name: str, cfg: ExperimentConfig, train: pd.DataFrame) -> Perturbation:
    classes = {'exact_copy': ExactCopy, 'subgroup_suppression': SubgroupSuppression,
               'feature_shuffle': FeatureShuffle, 'mcar': MCARMissingness,
               'copy': TrainCopy, 'negative': IndependentColumns}
    return classes[name](cfg, train)


def self_test() -> None:
    a = pd.DataFrame({'x': [1, np.nan]})
    assert not changed_cells(a, a.copy())
''')

from syntrustbench_campaign import perturbations

perturbations.self_test()


### Cached real-only privacy statistics from frozen helpers

In [ ]:
_write_module('cached_privacy.py', r'''"""Empirical privacy indicators under an explicit distance-based threat model."""

from __future__ import annotations

import numpy as np
from sklearn.metrics import roc_curve
from sklearn.neighbors import NearestNeighbors

from syntrustbench.models import Submission
from syntrustbench.stats import FeatureEncoder, aggregate_status, metric, percentile_interval, safe_auroc
from syntrustbench.validation import canonical_row_keys


def _bounded_rows(
    frame,
    maximum_rows: int,
    rng: np.random.Generator,
):
    """Return a reproducible row subset and its positional indices."""

    if len(frame) <= maximum_rows:
        positions = np.arange(len(frame))
    else:
        positions = np.sort(rng.choice(len(frame), size=maximum_rows, replace=False))
    return frame.iloc[positions], positions


def _tpr_at_fpr(
    labels: np.ndarray, scores: np.ndarray, target_fpr: float = 0.01
) -> float:
    false_positive_rate, true_positive_rate, _ = roc_curve(labels, scores)
    eligible = false_positive_rate <= target_fpr
    return float(true_positive_rate[eligible].max()) if eligible.any() else 0.0


def evaluate_privacy(submission: Submission, rng: np.random.Generator, real_cache: dict) -> dict:
    config = submission.config
    real = submission.real_train
    test = submission.real_test
    synthetic = submission.synthetic
    thresholds = config.thresholds
    iterations = config.bootstrap_iterations
    confidence = config.confidence_level
    metrics: dict[str, dict] = {}
    weaknesses: list[str] = []
    warnings: list[str] = []

    real_keys = real_cache["real_keys"]
    synthetic_keys = canonical_row_keys(synthetic, config)
    duplicate_mask = np.array([key in real_keys for key in synthetic_keys], dtype=bool)
    duplicate_rate = float(duplicate_mask.mean())
    duplicate_low, duplicate_high = percentile_interval(
        duplicate_rate,
        lambda rows: float(duplicate_mask[rows].mean()),
        len(synthetic),
        rng,
        iterations,
        confidence,
    )
    duplicate_flag = bool(duplicate_mask.any())
    metrics["exact_training_duplicate_rate"] = metric(
        duplicate_rate,
        ci_low=duplicate_low,
        ci_high=duplicate_high,
        interval_type="synthetic-row bootstrap",
        higher_is_worse=True,
        flag=duplicate_flag,
        threshold="Any exact training-record copy -> Fail",
        note=(
            "Share of synthetic rows matching a complete real-training row after "
            f"{config.exact_duplicate_decimals}-decimal canonicalization."
        ),
    )
    if duplicate_flag:
        weaknesses.append("The released synthetic table contains copied training rows.")

    privacy_continuous = config.continuous
    privacy_categorical = config.categorical + config.binary
    encoder = real_cache["encoder"]

    distance_real = real_cache["distance_real"]
    distance_synthetic, distance_synthetic_positions = _bounded_rows(
        synthetic, config.privacy_distance_max_rows, rng
    )
    real_matrix = real_cache["real_matrix"]
    synthetic_matrix = encoder.transform(distance_synthetic)
    if len(real_matrix) < 2:
        raise ValueError("At least two real training rows are required for privacy distances.")
    if len(distance_real) < len(real) or len(distance_synthetic) < len(synthetic):
        warnings.append(
            "Nearest-neighbor exposure was estimated from reproducible bounded samples "
            f"(real reference n={len(distance_real):,}; synthetic query "
            f"n={len(distance_synthetic):,}). Exact-copy analysis still used all rows."
        )

    neighbors = real_cache["neighbors"]
    synthetic_distances, _ = neighbors.kneighbors(synthetic_matrix)
    real_distances = real_cache["real_distances"]
    distance_to_closest = synthetic_distances[:, 0]
    full_synthetic_dcr = np.full(len(synthetic), np.nan)
    full_synthetic_dcr[distance_synthetic_positions] = distance_to_closest
    real_real_floor = float(np.percentile(real_distances[:, 1], 5))
    share_below_floor = float(np.mean(distance_to_closest < real_real_floor))

    median_distance = float(np.median(distance_to_closest))
    median_low, median_high = percentile_interval(
        median_distance,
        lambda rows: float(np.median(distance_to_closest[rows])),
        len(distance_to_closest),
        rng,
        iterations,
        confidence,
    )
    metrics["median_distance_to_closest_training_record"] = metric(
        median_distance,
        ci_low=median_low,
        ci_high=median_high,
        interval_type="synthetic-row bootstrap",
        higher_is_worse=False,
        note=(
            "Median standardized mixed-feature distance to the closest real training row; "
            f"estimated with {len(distance_synthetic):,} synthetic queries against "
            f"{len(distance_real):,} real references; descriptive only, not a privacy "
            "guarantee."
        ),
    )
    floor_low, floor_high = percentile_interval(
        share_below_floor,
        lambda rows: float(np.mean(distance_to_closest[rows] < real_real_floor)),
        len(distance_to_closest),
        rng,
        iterations,
        confidence,
    )
    floor_flag = share_below_floor > thresholds["privacy_dcr_share_conditional"]
    metrics["share_closer_than_real_real_floor"] = metric(
        share_below_floor,
        ci_low=floor_low,
        ci_high=floor_high,
        interval_type="synthetic-row bootstrap",
        higher_is_worse=True,
        flag=floor_flag,
        threshold=f"> {thresholds['privacy_dcr_share_conditional']:.2f} -> Conditional",
        note=(
            "Share of synthetic rows closer to training data than the fifth percentile "
            "of non-self real-to-real distances in the bounded distance sample."
        ),
    )
    distance_ratio = distance_to_closest / np.maximum(
        synthetic_distances[:, 1], 1e-12
    )
    ratio_p05 = float(np.percentile(distance_ratio, 5))
    metrics["nearest_neighbor_distance_ratio_p05"] = metric(
        ratio_p05,
        higher_is_worse=False,
        note=(
            "Fifth percentile of closest-to-second-closest distance ratio; "
            "low values identify unusually isolated matches."
        ),
    )

    attack_real, _ = _bounded_rows(real, config.privacy_attack_max_rows, rng)
    attack_test, _ = _bounded_rows(test, config.privacy_attack_max_rows, rng)
    attack_synthetic, _ = _bounded_rows(
        synthetic, config.privacy_attack_max_rows, rng
    )
    attack_available = (
        len(attack_real) >= 20
        and len(attack_test) >= 20
        and len(attack_synthetic) >= 20
    )
    attack_scores: np.ndarray | None = None
    attack_labels: np.ndarray | None = None
    if attack_available:
        released_matrix = encoder.transform(attack_synthetic)
        member_matrix = encoder.transform(attack_real)
        nonmember_matrix = encoder.transform(attack_test)
        released_neighbors = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(
            released_matrix
        )
        member_distance, _ = released_neighbors.kneighbors(member_matrix)
        nonmember_distance, _ = released_neighbors.kneighbors(nonmember_matrix)
        attack_scores = np.concatenate(
            [-member_distance[:, 0], -nonmember_distance[:, 0]]
        )
        attack_labels = np.concatenate(
            [np.ones(len(member_distance)), np.zeros(len(nonmember_distance))]
        )
        attack_auc = safe_auroc(attack_labels, attack_scores)
        attack_low, attack_high = percentile_interval(
            attack_auc,
            lambda rows: safe_auroc(attack_labels[rows], attack_scores[rows]),
            len(attack_labels),
            rng,
            iterations,
            0.95,
        )
        attack_tpr = _tpr_at_fpr(attack_labels, attack_scores, 0.01)
        attack_flag = (
            attack_low is not None
            and attack_low > thresholds["privacy_attack_auc_fail"]
        )
        if (
            attack_low is None or attack_high is None
            or not np.isfinite(attack_low) or not np.isfinite(attack_high)
        ):
            attack_status = "NotEvaluated"
        elif attack_flag:
            attack_status = "Fail"
        elif attack_high <= thresholds["privacy_attack_auc_fail"]:
            attack_status = "Pass"
        else:
            attack_status = "Conditional"
        metrics["distance_membership_attack_auc"] = metric(
            attack_auc,
            ci_low=attack_low,
            ci_high=attack_high,
            interval_type="candidate-record bootstrap (95%)",
            higher_is_worse=True,
            flag=attack_flag,
            status=attack_status,
            threshold=(
                f"Provisional: lower 95% bound > {thresholds['privacy_attack_auc_fail']:.2f} -> Fail; "
                f"upper 95% bound <= {thresholds['privacy_attack_auc_fail']:.2f} -> Pass; "
                "otherwise Conditional"
            ),
            note=(
                "Black-box distance attack: candidate closeness to released synthetic "
                "data predicts membership in real training versus held-out real data; "
                f"members n={len(attack_real):,}, nonmembers n={len(attack_test):,}, "
                f"released rows n={len(attack_synthetic):,}."
            ),
        )
        metrics["distance_membership_attack_tpr_at_1pct_fpr"] = metric(
            attack_tpr,
            higher_is_worse=True,
            note=(
                "Exploratory TPR at 1% FPR; not decision-driving because sufficiently "
                "reliable low-FPR uncertainty is not currently implemented."
            ),
        )
    else:
        warnings.append(
            "Distance membership attack was not run because at least one input had fewer "
            "than 20 rows."
        )
        metrics["distance_membership_attack_auc"] = metric(
            None,
            higher_is_worse=True,
            status="NotEvaluated",
            flag=False,
            note="Insufficient sample size for the configured attack.",
        )
        metrics["distance_membership_attack_tpr_at_1pct_fpr"] = metric(
            None,
            higher_is_worse=True,
            status="NotEvaluated",
            flag=False,
            note=(
                "Insufficient sample size for the configured attack. Exploratory TPR at 1% FPR; "
                "not decision-driving because sufficiently reliable low-FPR uncertainty "
                "is not currently implemented."
            ),
        )

    metrics["exact_training_duplicate_rate"]["status"] = "Fail" if duplicate_flag else "Pass"
    metrics["share_closer_than_real_real_floor"]["status"] = "Conditional" if floor_flag else "Pass"
    required = (
        "exact_training_duplicate_rate", "share_closer_than_real_real_floor",
        "distance_membership_attack_auc",
    )
    for name in required:
        metrics[name]["required"] = True
    metrics["distance_membership_attack_tpr_at_1pct_fpr"]["required"] = False
    attack_failure = metrics["distance_membership_attack_auc"]["status"] == "Fail"
    status = aggregate_status(metrics[name]["status"] for name in required)

    if floor_flag:
        weaknesses.append(
            "An elevated share of synthetic rows is unusually close to training records."
        )
    if attack_failure:
        weaknesses.append(
            "The tested membership attacker extracted signal above the provisional limit."
        )
    return {
        "dimension": "privacy",
        "status": status,
        "metrics": metrics,
        "failure_flags": [
            f"privacy:{name}" for name, value in metrics.items() if value["flag"]
        ],
        "warnings": warnings,
        "weaknesses": weaknesses,
        "threat_model": {
            "attacker_knowledge": "released synthetic table and a candidate record",
            "attacker_access": "black-box access to released rows; no generator access",
            "attacker_goal": "infer whether the candidate was in real_train",
            "members": "real_train",
            "nonmembers": "held-out real_test",
            "attack": "distance to closest released synthetic row",
            "interpretation": (
                "Empirical risk under this attacker only; not a formal privacy guarantee."
            ),
        },
        "details": {
            "real_real_distance_floor_p05": real_real_floor,
            "attack_available": attack_available,
            "distance_reference_rows": len(distance_real),
            "distance_query_rows": len(distance_synthetic),
            "attack_member_rows": len(attack_real),
            "attack_nonmember_rows": len(attack_test),
            "attack_released_rows": len(attack_synthetic),
        },
        "_internal": {
            "duplicate_mask": duplicate_mask,
            "synthetic_dcr": full_synthetic_dcr,
            "real_real_floor": real_real_floor,
            "synthetic_keys": synthetic_keys,
            "attack_scores": attack_scores,
            "attack_labels": attack_labels,
        },
    }


def build_real_cache(real, config, seed):
    """Compute all expensive real-only distance quantities once per role set.

    This file is adapted from frozen v0.3 privacy.py: only real-only caching
    is added here. The v0.4 wrapper replaces the exact-copy decision and adds
    the 0.1% TPR report. The official installed helper is never patched.
    """
    encoder = FeatureEncoder(config.continuous, config.categorical + config.binary).fit(real)
    distance_real, _ = _bounded_rows(real, config.privacy_distance_max_rows, np.random.default_rng(seed))
    matrix = encoder.transform(distance_real)
    neighbors = NearestNeighbors(n_neighbors=2, n_jobs=-1).fit(matrix)
    distances, _ = neighbors.kneighbors(matrix)
    return {"encoder": encoder, "distance_real": distance_real, "real_matrix": matrix,
            "neighbors": neighbors, "real_distances": distances,
            "real_keys": set(canonical_row_keys(real, config).tolist()), "seed": seed}


def self_test():
    assert _tpr_at_fpr(np.array([0, 0, 1, 1]), np.array([0., .1, .8, .9]), .01) == 1.
''')

from syntrustbench_campaign import cached_privacy

cached_privacy.self_test()


### Official adapter and v0.4 Privacy, Equity, Robustness (equity and robustness are non-gating diagnostics)

In [ ]:
_write_module('evaluation.py', r'''"""Additive v0.4 fixes; supplied reference statistics remain unchanged."""
from __future__ import annotations

import copy
from dataclasses import dataclass
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from syntrustbench.privacy import _tpr_at_fpr
from syntrustbench.stats import categorical_values, metric, percentile_interval, safe_auroc
from syntrustbench.validation import canonical_row_keys, _coerce_frame, _map_target
from syntrustbench.config import load_config
from syntrustbench.models import Submission

from .config import ExperimentConfig, DIMENSIONS, GATE_DIMENSIONS, seed_for
from .core import Gate, available_number, read_json, write_json
from .data import read_analytic
from .reference import ReferenceContext, ReferenceFidelity, ReferenceUtility, _retention, _retention_verdict
from .results import DimensionEvaluator, DimensionResult
from .cached_privacy import evaluate_privacy, build_real_cache


def result(dimension: str, metrics: dict, statuses: list[str], **details) -> DimensionResult:
    return DimensionResult.from_dict({'dimension': dimension, 'status': Gate.aggregate(statuses),
                                     'metrics': metrics, 'weaknesses': [], 'warnings': [], **details})


def signal(value, *, higher: bool = True, reason: str = '') -> dict:
    return {'estimate': float(value) if available_number(value) else None,
            'higher_is_worse': higher, 'status': 'Available' if available_number(value) else 'NotEvaluated',
            'note': reason if not available_number(value) else ''}


@dataclass
class PrivacyV04(DimensionEvaluator):
    cfg: ExperimentConfig
    train: pd.DataFrame
    test: pd.DataFrame
    seed: int
    smoke: bool = False
    real_cache: dict | None = None

    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        config = load_config(self.cfg.benchmark(smoke=self.smoke))
        frames, warnings = [], []
        for label, frame, require_two in [('real_train', self.train, True), ('real_test', self.test, True), ('synthetic', synthetic, False)]:
            value, notes = _coerce_frame(frame, config, label)
            value, targets = _map_target(value, config, label, require_two_classes=require_two)
            frames.append(value); warnings.extend(notes + targets)
        # RoleAssigner already proved patient disjointness. Analytic-row coincidences
        # must remain observable for this test, unlike frozen v0.3's overlap check.
        submission = Submission(*frames, config=config, input_hashes={}, warnings=warnings)
        cache = self.real_cache if self.real_cache is not None else build_real_cache(submission.real_train, config, seed_for(self.seed, 'real_floor'))
        raw = evaluate_privacy(submission, np.random.default_rng(self.seed), cache)
        internal = raw.pop('_internal')
        metrics = raw['metrics']
        keys = set(canonical_row_keys(submission.real_train, submission.config))
        coincidences = np.isin(canonical_row_keys(submission.real_test, submission.config), list(keys))
        copies = internal['duplicate_mask']
        p0 = float(coincidences.mean())
        pvalue = float(binomtest(int(copies.sum()), len(copies), p=p0, alternative='greater').pvalue)
        fail = pvalue < self.cfg.privacy_binomial_alpha
        duplicate = metrics['exact_training_duplicate_rate']
        duplicate.update(flag=fail, status='Fail' if fail else 'Pass',
                         threshold=f'one-sided binomial p < {self.cfg.privacy_binomial_alpha}',
                         note=f'Exact-match count against plug-in E-in-T coincidence rate {p0:.8g}; all T rows used.')
        metrics['exact_match_binomial_pvalue'] = metric(pvalue, higher_is_worse=False,
                                                       flag=fail, note='Plug-in null; uncertainty in p0 is not integrated.')
        metrics['real_real_coincidence_rate'] = metric(p0, higher_is_worse=True,
                                                      note=f'{int(coincidences.sum())}/{len(coincidences)} held-out rows found in T.')
        metrics['exact_match_excess'] = metric(float(copies.mean())-p0, higher_is_worse=True)
        labels, scores = internal['attack_labels'], internal['attack_scores']
        for rate, key, minimum in [(.001, 'distance_membership_attack_tpr_at_0_1pct_fpr', 1000),
                                    (.01, 'distance_membership_attack_tpr_at_1pct_fpr', 100)]:
            usable = labels is not None and scores is not None and int(np.sum(labels == 0)) >= minimum
            value = _tpr_at_fpr(labels, scores, rate) if usable else None
            metrics[key] = metric(value, higher_is_worse=True, status='Available' if usable else 'NotEvaluated',
                                  note=f'Exploratory weak lower-bound attack; requires >= {minimum} nonmembers for FPR resolution; no low-FPR CI.')
            metrics[key]['required'] = False
        raw['weaknesses'] = [s for s in raw['weaknesses'] if 'copied training rows' not in s]
        if fail:
            raw['weaknesses'].append('Exact matches exceed the empirical real–real coincidence null.')
        required = ('exact_training_duplicate_rate', 'share_closer_than_real_real_floor', 'distance_membership_attack_auc')
        raw['status'] = Gate.aggregate([metrics[k]['status'] for k in required])
        raw['failure_flags'] = [f'privacy:{k}' for k, v in metrics.items() if v.get('flag')]
        raw['signals'] = {k: signal(metrics[k]['estimate'], reason=metrics[k].get('note', '')) for k in
                          ('exact_match_excess', 'share_closer_than_real_real_floor', 'distance_membership_attack_auc')}
        return DimensionResult.from_dict(raw)


@dataclass
class EquityV04(DimensionEvaluator):
    cfg: ExperimentConfig
    context: ReferenceContext
    seed: int

    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        ctx, rng = self.context, np.random.default_rng(self.seed)
        pred = ctx._predict('random_forest', synthetic)
        base, y = ctx.trtr['random_forest'], ctx.y_test
        groups, statuses, retentions, gaps = [], [], [], []
        for attribute in self.cfg.data.protected:
            test_values = categorical_values(ctx.test[attribute])
            ref_mass = categorical_values(ctx.reference[attribute]).value_counts(normalize=True)
            syn_mass = categorical_values(synthetic[attribute]).value_counts(normalize=True)
            levels = sorted(set(ref_mass.index) | set(syn_mass.index))
            gaps.extend(abs(float(ref_mass.get(level, 0))-float(syn_mass.get(level, 0))) for level in levels)
            for level in sorted(test_values.unique()):
                rows = np.flatnonzero((test_values == level).to_numpy())
                positives, negatives = int(y[rows].sum()), int(len(rows)-y[rows].sum())
                eligible = len(rows) >= self.cfg.min_subgroup_n and min(positives, negatives) >= self.cfg.min_subgroup_outcomes
                record = {'attribute': attribute, 'level': level, 'n': len(rows), 'positive': positives,
                          'negative': negatives, 'eligible': eligible, 'status': 'NotEvaluated',
                          'estimate': None, 'ci_low': None, 'ci_high': None,
                          'note': 'Below prespecified subgroup support; excluded from decision-eligible group set.'}
                if eligible:
                    point = _retention(y[rows], base[rows], pred[rows])
                    base_ci = percentile_interval(safe_auroc(y[rows], base[rows]),
                        lambda r: safe_auroc(y[rows][r], base[rows][r]), len(rows), rng, ctx.iterations, .95)
                    low, high = percentile_interval(point,
                        lambda r: _retention(y[rows][r], base[rows][r], pred[rows][r]), len(rows), rng, ctx.iterations, .95)
                    verdict = _retention_verdict(point, low, high, base_ci[0])
                    record.update(estimate=point, ci_low=low, ci_high=high, status=verdict,
                                  note='Paired subgroup E-row bootstrap; chance-corrected against subgroup TRTR.',
                                  subgroup_trtr_auc=safe_auroc(y[rows], base[rows]), subgroup_trtr_ci=base_ci)
                    statuses.append(verdict)
                    retentions.append(point if verdict != 'NotEvaluated' else None)
                groups.append(record)
        gap = max(gaps) if gaps else None
        worst = min(retentions) if retentions and all(available_number(v) for v in retentions) else None
        metrics = {
            'maximum_representation_gap': metric(gap, higher_is_worse=True, flag=available_number(gap) and gap > .1,
                                                  threshold='> 0.10 -> Conditional'),
            'worst_group_retention': metric(worst, higher_is_worse=False,
                                            status='Available' if worst is not None else 'NotEvaluated',
                                            note='Minimum over decision-eligible groups; unavailable if any eligible group has weak TRTR.'),
        }
        if not statuses:
            statuses = ['NotEvaluated']
        statuses.append('NotEvaluated' if gap is None else ('Conditional' if gap > .1 else 'Pass'))
        return result('equity', metrics, statuses, subgroup_results=groups,
                      eligible_groups=sum(g['eligible'] for g in groups), total_groups=len(groups),
                      signals={'maximum_representation_gap': signal(gap),
                               'worst_group_retention': signal(worst, higher=False, reason='No fully evaluable eligible subgroup set.')})


@dataclass
class RobustnessV04(DimensionEvaluator):
    cfg: ExperimentConfig
    context: ReferenceContext
    utility: dict
    utility_rng_state: dict
    seed: int
    smoke: bool = False

    @staticmethod
    def flips(baseline: str, stressed: str) -> bool:
        return baseline != stressed and baseline != 'NotEvaluated' and stressed != 'NotEvaluated'

    def _utility(self, table: pd.DataFrame, model_seed: int) -> tuple[dict, float]:
        ctx = copy.copy(self.context)
        ctx.rng = np.random.default_rng(); ctx.rng.bit_generator.state = copy.deepcopy(self.utility_rng_state)
        ctx.seed = model_seed
        value = ReferenceUtility(ctx).evaluate(table).to_dict()
        from .calibration import utility_signals
        missing = [k for k, v in utility_signals(value['metrics']).items() if not available_number(v['estimate'])]
        value['raw_status'] = value['status']
        value['missing_required_evidence'] = missing
        if missing:
            value['status'] = Gate.aggregate([value['status'], 'NotEvaluated'])
        auc = safe_auroc(ctx.y_test, ctx._predict('random_forest', table))
        return value, auc

    def evaluate(self, synthetic: pd.DataFrame) -> DimensionResult:
        ctx, baseline = self.context, self.utility['status']
        pred = ctx._predict('random_forest', synthetic)
        if baseline == 'NotEvaluated' or np.allclose(pred, pred[0]):
            reason = 'Utility NotEvaluated or constant TSTR random-forest predictor.'
            names = ('downstream_model_seed_auroc_cv', 'tstr_bootstrap_relative_ci_width',
                     'tstr_auroc_drop_at_maximum_missingness', 'training_size_auroc_spread')
            return result('robustness', {k: metric(None, higher_is_worse=True, status='NotEvaluated', note=reason) for k in names},
                          ['NotEvaluated'], stressors=[], constant_predictor=bool(np.allclose(pred, pred[0])),
                          signals={k: signal(None, reason=reason) for k in names}, verdict_flip=False)
        small = self.smoke or self.cfg.execution.smoke_mode
        seeds = self.cfg.model_seeds[:2] if small else self.cfg.model_seeds
        records, seed_aucs, missing_aucs, size_aucs = [], [], {}, []

        def score(label, frame, model_seed):
            value, auc = self._utility(frame, model_seed)
            records.append({'stressor': label, 'utility_status': value['status'], 'tstr_auroc': auc,
                            'verdict_flip': self.flips(baseline, value['status']), 'utility': value})
            return auc

        for model_seed in seeds:
            seed_aucs.append(score(f'model_seed/{model_seed}', synthetic, model_seed))
        repeats = 2 if small else self.cfg.missingness_repeats
        for rate in self.cfg.missingness_rates:
            missing_aucs[rate] = []
            for repeat in range(repeats):
                frame = synthetic.copy(deep=True)
                for column in ctx.config.feature_columns:
                    rng = np.random.default_rng(seed_for(self.seed, 'stress_missing', rate, repeat, column))
                    frame.loc[rng.random(len(frame)) < rate, column] = np.nan
                missing_aucs[rate].append(score(f'missingness/{rate}/{repeat}', frame, ctx.seed))
        for fraction in self.cfg.training_fractions:
            frame = synthetic if fraction == 1 else synthetic.sample(frac=fraction, random_state=seed_for(self.seed, 'training_size'))
            size_aucs.append(score(f'training_size/{fraction}', frame, ctx.seed))
        auc = safe_auroc(ctx.y_test, pred)
        ci = percentile_interval(auc, lambda r: safe_auroc(ctx.y_test[r], pred[r]), len(pred),
                                 np.random.default_rng(seed_for(self.seed, 'auc_ci')), ctx.iterations, .95)
        finite = lambda values: all(available_number(v) for v in values)
        cv = float(np.std(seed_aucs, ddof=1)/abs(np.mean(seed_aucs))) if finite(seed_aucs) and np.mean(seed_aucs) != 0 else None
        width = (ci[1]-ci[0])/auc if all(available_number(v) for v in ci) and auc > 0 else None
        maximum = missing_aucs[max(missing_aucs)]
        drop = auc-float(np.mean(maximum)) if finite(maximum) else None
        spread = max(size_aucs)-min(size_aucs) if finite(size_aucs) else None
        values = {'downstream_model_seed_auroc_cv': (cv, .05), 'tstr_bootstrap_relative_ci_width': (width, .20),
                  'tstr_auroc_drop_at_maximum_missingness': (drop, .10), 'training_size_auroc_spread': (spread, .10)}
        metrics, statuses = {}, []
        for name, (v, threshold) in values.items():
            status = 'NotEvaluated' if v is None else ('Conditional' if v > threshold else 'Pass')
            metrics[name] = metric(v, higher_is_worse=True, status=status, flag=status == 'Conditional', threshold=f'> {threshold} -> Conditional')
            statuses.append(status)
        flip = any(r['verdict_flip'] for r in records)
        statuses.extend('NotEvaluated' for r in records if r['utility_status'] == 'NotEvaluated')
        statuses.append('Fail' if flip else 'Pass')
        metrics['any_utility_verdict_flip'] = metric(int(flip), higher_is_worse=True, flag=flip,
                                                    threshold='Any evaluable verdict change in either direction -> Fail')
        return result('robustness', metrics, statuses, stressors=records, verdict_flip=flip, constant_predictor=False,
                      baseline_utility_status=baseline, signals={k: signal(v[0], reason='Stressor evidence unavailable.') for k, v in values.items()})


class OfficialProfiler:
    CANONICAL = ('summary.json', 'metrics.csv', 'subgroup_results.csv', 'benchmark_card.yaml',
                 'report.md', 'run_manifest.json', 'execution_log.txt', 'details.json')

    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg

    def evaluate(self, roles: dict, synthetic: pd.DataFrame, folder: Path, smoke=False) -> dict:
        import syntrustbench
        # Same n-sized floor and candidate as reference utility; full T remains the v0.4 privacy member set.
        train, test = [read_analytic(roles['paths'][k], self.cfg) for k in ('floor', 'test')]
        try:
            syntrustbench.evaluate(train, test, synthetic, self.cfg.benchmark(smoke=smoke), folder)
        except ValueError as exc:
            # Frozen v0.3 refuses to evaluate when one held-out real row exactly matches a
            # training row. Roles here are patient-disjoint by hashed ID, so this is a
            # real-real coincidence. Record v0.3's refusal as an explicit NotEvaluated
            # outcome (v0.3 itself is not modified); every other error is re-raised.
            if not str(exc).startswith('Real train/test overlap detected'):
                raise
            folder.mkdir(parents=True, exist_ok=True)
            stub = {'version': '0.3.0', 'benchmark_gate': 'NotEvaluated',
                    'dimension_status': {d: 'NotEvaluated' for d in DIMENSIONS},
                    'dimensions': {d: {'status': 'NotEvaluated', 'metrics': {}} for d in DIMENSIONS},
                    'official_stop': str(exc),
                    'official_stop_policy': 'v0.3 exact-row stop on patient-disjoint roles; recorded, not overridden.'}
            write_json(folder / 'official_stop.json', stub)
            return stub
        if not all((folder / p).is_file() for p in self.CANONICAL):
            raise RuntimeError('Official v0.3 did not write all eight canonical files.')
        summary = read_json(folder / 'summary.json')
        if summary['version'] != '0.3.0':
            raise RuntimeError('Unexpected official protocol version.')
        return summary


def build_context(cfg: ExperimentConfig, roles: dict, folder: Path, smoke=False) -> dict:
    frames = [read_analytic(roles['paths'][k], cfg) for k in ('floor', 'reference', 'test')]
    ctx = ReferenceContext(*frames, cfg.benchmark(smoke=smoke))
    members = read_analytic(roles['paths']['train'], cfg)
    members, _ = _coerce_frame(members, ctx.config, 'real_train')
    members, _ = _map_target(members, ctx.config, 'real_train', require_two_classes=True)
    ctx.privacy_real_cache = build_real_cache(members, ctx.config, seed_for(roles['key'], 'privacy_real_floor'))
    path = folder / 'reference_context.pkl'
    with path.open('wb') as handle:
        pickle.dump(ctx, handle, protocol=5)
    write_json(folder / 'reference_context_summary.json', {
        'floor': {k: v for k, v in ctx.floor.items() if k != 'coverage_rows'},
        'reference_calibration': ctx.reference_calibration, 'trtr_auroc_ci': ctx.trtr_auroc_ci,
        'rng_policy': 'Private copy of frozen post-construction state per table; paired utility state for stressors.',
        'rare_categories': 'Top 250 levels per column, others pooled as __OTHER__',
        'confidence_limit': 'Detection bootstrap omits classifier-refit variance.',
        'exact_row_coincidences_with_floor': ctx.exact_row_coincidences,
        'exact_row_policy': 'Recorded, not stopped: roles are patient-disjoint by hashed ID.'})
    return {'context_path': str(path), 'roles': roles}


def reference_signals(f: dict, u: dict, ctx: ReferenceContext) -> dict:
    fm, um = f['metrics'], u['metrics']
    get = lambda m, k, field='estimate': m[k].get(field)
    fs = {k: signal(get(fm, k)) for k in ('mass_weighted_coverage_deficit',
        'mean_normalized_wasserstein_excess', 'mean_jensen_shannon_divergence_excess',
        'association_matrix_error_excess', 'clinical_constraint_violation_rate')}
    fs['detection_auc_excess'] = signal(get(fm, 'detection_auc')-ctx.floor['detection_auc'])
    us = {}
    for probe in ctx.PROBES:
        key = f'synthetic_retention_{probe}'
        m = um[key]
        us[key] = signal(m['ci_high'] if m['status'] != 'NotEvaluated' else None, higher=False,
                         reason='Retention upper CI unavailable or weak real-trained baseline.')
    for key in ('relative_calibration_in_the_large', 'relative_calibration_slope'):
        us[key + '_high'] = signal(get(um, key))
        us[key + '_low'] = signal(get(um, key), higher=False)
    return {'fidelity': fs, 'utility': us}


def evaluate_v04(cfg: ExperimentConfig, context_path: str, roles: dict,
                 synthetic: pd.DataFrame, key: str, smoke=False) -> dict:
    # Only trusted, ledger-hash-verified local context artifacts are loaded.
    with open(context_path, 'rb') as handle:
        ctx = pickle.load(handle)
    f = ReferenceFidelity(ctx).evaluate(synthetic).to_dict()
    utility_rng = copy.deepcopy(ctx.rng.bit_generator.state)
    u = ReferenceUtility(ctx).evaluate(synthetic).to_dict()
    raw_reference = {'fidelity': copy.deepcopy(f), 'utility': copy.deepcopy(u)}
    signals = reference_signals(f, u, ctx)
    # Guard missing evidence without changing any statistic in the supplied evaluators.
    for dimension, value in [('fidelity', f), ('utility', u)]:
        unavailable = [k for k, v in signals[dimension].items() if v['status'] == 'NotEvaluated']
        if dimension == 'fidelity':
            unavailable += [f'column:{c}' for c in cfg.data.continuous
                            if any(not pd.to_numeric(frame[c], errors='coerce').notna().any()
                                   for frame in (synthetic, ctx.reference, ctx.train))]
        if unavailable:
            value['status'] = Gate.aggregate([value['status'], 'NotEvaluated'])
        value['missing_required_evidence'] = unavailable
        for name, m in value['metrics'].items():
            estimate = m.get('estimate')
            if estimate is None or isinstance(estimate, (int,float,np.number)) and not available_number(estimate):
                m['status'] = 'NotEvaluated'
                m['availability_reason'] = 'The preserved reference calculation returned no finite estimate.'
    members, test = [read_analytic(roles['paths'][k], cfg) for k in ('train', 'test')]
    p = PrivacyV04(cfg, members, test, seed_for(key, 'privacy'), smoke,
                   getattr(ctx, 'privacy_real_cache', None)).evaluate(synthetic).to_dict()
    e = EquityV04(cfg, ctx, seed_for(key, 'equity')).evaluate(synthetic).to_dict()
    r = RobustnessV04(cfg, ctx, u, utility_rng, seed_for(key, 'robustness'), smoke).evaluate(synthetic).to_dict()
    dimensions = dict(zip(DIMENSIONS, (f, u, p, e, r)))
    for dimension in ('privacy', 'equity', 'robustness'):
        signals[dimension] = dimensions[dimension].pop('signals')
    hard_fail = {'fidelity': False, 'utility': synthetic[cfg.data.target].nunique() < 2,
                 'privacy': False, 'equity': False, 'robustness': r.get('verdict_flip', False)}
    return {'dimensions': dimensions, 'raw_reference': raw_reference, 'signals': signals,
            'hard_fail': hard_fail, 'gate_dimensions': list(GATE_DIMENSIONS),
            'gate': Gate.aggregate(dimensions[d]['status'] for d in GATE_DIMENSIONS),
            'scope': ('provisional v0.4.1; gate over fidelity, utility and privacy; equity and robustness '
                      'are non-gating diagnostics; calibration is a separate saved-evidence wrapper')}


def self_test() -> None:
    assert RobustnessV04.flips('Pass', 'Fail') and RobustnessV04.flips('Fail', 'Pass')
    assert RobustnessV04.flips('Pass', 'Conditional')
    assert not RobustnessV04.flips('Pass', 'NotEvaluated')
''')

from syntrustbench_campaign import evaluation

evaluation.self_test()


### Null calibration (full conformal, gate and diagnostic families) and calibrated wrappers

In [ ]:
_write_module('calibration.py', r'''"""Direction-aware full-conformal maxT calibration, with explicit evidence gaps.

Two families are calibrated separately: the gate family (fidelity, utility,
privacy), whose decision is the verdict, and the diagnostic family (equity,
robustness), which is reported but never decides the verdict.

Decision rule (full conformal, per family). Each signal is oriented so that
larger is worse. A new table is pooled with the n fit runs; every signal is
standardized by the pooled median and SD; each run's statistic is its maximum
standardized signal. With r = floor(alpha (n + 1)), the table fails when its
statistic exceeds the r-th largest fit-run statistic. Because the rule is a
symmetric function of the n + 1 pooled runs, an honest table exchangeable with
the fit runs is rejected with probability at most r / (n + 1) <= alpha, for any
signal distribution (Vovk et al. 2005; Lei et al. 2018). A dimension fails when
its own maximum exceeds that same family threshold, so the gate fails exactly
when any gating dimension fails.
"""
from __future__ import annotations

from dataclasses import dataclass
import copy
import math
import numpy as np

from .config import CalibrationConfig, DIMENSIONS, GATE_DIMENSIONS, DIAGNOSTIC_DIMENSIONS
from .core import Gate, UnavailableAnalysis, available_number

FAMILIES = {'gate': GATE_DIMENSIONS, 'diagnostic': DIAGNOSTIC_DIMENSIONS}


def family_of(dimension: str) -> str:
    return 'gate' if dimension in GATE_DIMENSIONS else 'diagnostic'


def rejection_rank(n: int, alpha: float) -> int:
    """r = floor(alpha (n+1)); r = 0 means no table can be rejected at this alpha."""
    return math.floor(alpha * (n + 1) + 1e-12)


def standardize(matrix: np.ndarray, floor: float) -> np.ndarray:
    center = np.median(matrix, axis=0)
    scale = np.maximum(np.std(matrix, axis=0, ddof=1), floor)
    return (matrix - center) / scale


def conformal_decision(fit: np.ndarray, candidate: np.ndarray, alpha: float, floor: float) -> dict:
    """Full-conformal max-statistic decision for one candidate row of oriented signals."""
    n = fit.shape[0]
    z = standardize(np.vstack([fit, candidate]), floor)
    maxima = z.max(axis=1)
    r = rejection_rank(n, alpha)
    threshold = float(np.sort(maxima[:n])[::-1][r - 1]) if r >= 1 else None
    pvalue = float((1 + np.sum(maxima[:n] >= maxima[-1])) / (n + 1))
    return {'candidate_z': z[-1].tolist(), 'threshold': threshold, 'candidate_max': float(maxima[-1]),
            'pvalue': pvalue, 'rejection_rank': r, 'nominal_level': r / (n + 1),
            'finite_sample_valid': r >= 1}


def binomial_admission(failures: int, runs: int, target: float, level: float) -> tuple[float, bool]:
    """One-sided exact test of excess honest-data non-acceptance over target."""
    from scipy.stats import binom
    pvalue = float(binom.sf(failures - 1, runs, target)) if failures > 0 else 1.
    return pvalue, pvalue >= level


def utility_signals(metrics: dict) -> dict:
    """Transform existing estimates only; no refits or new bootstrap calculations."""
    output = {}
    for probe in ('random_forest', 'logistic_regression', 'gradient_boosting'):
        key = f'synthetic_retention_{probe}'
        value = metrics[key]
        output[key] = {'estimate': value['ci_high'] if value['status'] != 'NotEvaluated' else None,
                       'higher_is_worse': False}
    for key in ('relative_calibration_in_the_large', 'relative_calibration_slope'):
        output[key+'_high'] = {'estimate': metrics[key]['estimate'], 'higher_is_worse': True}
        output[key+'_low'] = {'estimate': metrics[key]['estimate'], 'higher_is_worse': False}
    return output


@dataclass
class NullCalibrator:
    cfg: CalibrationConfig

    def counts(self, smoke: bool = False) -> tuple[int, int]:
        return ((self.cfg.smoke_repetitions, self.cfg.smoke_fit_runs) if smoke
                else (self.cfg.repetitions, self.cfg.fit_runs))

    def assignment(self, smoke: bool = False) -> tuple[list[int], list[int]]:
        count, fit = self.counts(smoke)
        order = np.random.default_rng(self.cfg.assignment_seed).permutation(count).tolist()
        return order[:fit], order[fit:]

    def fit(self, profiles: dict[int, dict], smoke: bool = False, sdmetrics_scores: dict | None = None) -> dict:
        """Freeze the oriented fit-run signals; audit runs are never read."""
        fit_ids, audit_ids = self.assignment(smoke)
        missing = set(fit_ids)-set(profiles)
        if missing:
            raise UnavailableAnalysis(f'Missing fit null profiles: {sorted(missing)}')
        # Menu comes from the frozen evaluator definitions, not generator outcomes.
        menu = profiles[fit_ids[0]]['signals']
        columns, excluded, blocked = [], [], set()
        for dimension in DIMENSIONS:
            for name in menu[dimension]:
                items = [profiles[i]['signals'][dimension][name] for i in fit_ids]
                values = [v['estimate'] for v in items]
                directions = [bool(v['higher_is_worse']) for v in items]
                if len(set(directions)) != 1:
                    raise ValueError('Metric direction changed between null repetitions.')
                if not all(available_number(v) for v in values):
                    excluded.append({'dimension': dimension, 'metric': name, 'family': family_of(dimension),
                                     'missing_run_ids': [i for i, v in zip(fit_ids, values) if not available_number(v)],
                                     'reason': 'Required signal missing; calibrated dimension cannot Pass.'})
                    blocked.add(dimension); continue
                direction = 1 if directions[0] else -1
                harm = direction * np.asarray(values, dtype=float)
                sd = float(np.std(harm, ddof=1))
                columns.append({'dimension': dimension, 'family': family_of(dimension), 'metric': name,
                                'direction': direction, 'fit_harm': harm.tolist(),
                                # Fixed fit-run location/scale: used only for the frozen
                                # compensatory score mapping and descriptive thresholds.
                                'center': float(np.median(harm)), 'scale': max(sd, self.cfg.scale_floor),
                                'observed_sd': sd, 'degenerate': sd <= self.cfg.scale_floor,
                                'degenerate_note': ('Constant on honest fit runs; any move in the harmful '
                                                    'direction ranks above every fit run.' if sd <= self.cfg.scale_floor else '')})
        if not any(c['family'] == 'gate' for c in columns):
            raise UnavailableAnalysis('No calibratable gate signals; nothing can be frozen.')
        n = len(fit_ids)
        levels = {str(a): {'rejection_rank': rejection_rank(n, a), 'nominal_level': rejection_rank(n, a)/(n+1),
                           'finite_sample_valid': rejection_rank(n, a) >= 1} for a in self.cfg.target_rates}
        # Descriptive per-signal thresholds: the signal value that alone would reach the
        # family threshold computed among fit runs. Decisions never use these numbers.
        descriptive = []
        for family in FAMILIES:
            cols = [c for c in columns if c['family'] == family]
            if not cols: continue
            z = standardize(np.array([c['fit_harm'] for c in cols]).T, self.cfg.scale_floor)
            maxima = np.sort(z.max(axis=1))[::-1]
            for a, level in levels.items():
                r = level['rejection_rank']
                cut = float(maxima[r-1]) if r >= 1 else None
                descriptive += [{**{k: c[k] for k in ('dimension', 'family', 'metric', 'direction', 'center', 'scale')},
                                 'alpha': float(a), 'family_threshold_z_among_fit_runs': cut,
                                 'raw_cutoff_descriptive': None if cut is None else (c['center'] + cut*c['scale'])/c['direction'],
                                 'comparison': '>' if c['direction'] == 1 else '<',
                                 'note': 'Descriptive only; decisions use full-conformal ranking with the candidate pooled.'}
                                for c in cols]
        return {'method': ('full-conformal studentized global max statistic per family (gate; diagnostic); '
                           'candidate pooled with fit runs; reject when the candidate statistic exceeds the '
                           'floor(alpha(n+1))-th largest fit-run statistic; strict exceedance'),
                'families': {f: list(d) for f, d in FAMILIES.items()}, 'scale_floor': self.cfg.scale_floor,
                'fit_ids': fit_ids, 'audit_ids': audit_ids, 'columns': columns, 'levels': levels,
                'raw_cutoffs': descriptive, 'excluded': excluded,
                'sdmetrics_fit_scores': [(sdmetrics_scores or {}).get(i) for i in fit_ids],
                'blocked_dimensions': sorted(blocked), 'smoke_only': smoke,
                'limitations': [f'Exact marginal level r/(n+1) <= alpha if honest tables are exchangeable with the {n} fit runs; '
                                'the held-out audit tests that assumption, and main-campaign roles are larger than null roles',
                                'audit run IDs held out; development patients reused across runs',
                                'with few fit runs some alpha levels cannot reject (finite_sample_valid false)']}


class CalibratedEvaluator:
    """Frozen-evidence wrapper: raw ReferenceFidelity/Utility dictionaries are untouched."""
    def __init__(self, model: dict):
        self.model = model

    def z_values(self, dimension: str, signals: dict) -> tuple[list[float], list[str]]:
        """Fixed fit-run standardization, used only by the frozen compensatory score mapping."""
        zs, unavailable = [], []
        if dimension in self.model['blocked_dimensions']:
            unavailable.append('Calibration lacks one or more required signals.')
        for c in self.model['columns']:
            if c['dimension'] != dimension:
                continue
            value = signals.get(c['metric'], {}).get('estimate')
            if not available_number(value):
                unavailable.append(c['metric']); continue
            zs.append((c['direction']*value-c['center'])/c['scale'])
        if not zs:
            unavailable.append('No standardized evidence.')
        return zs, unavailable

    def family_decision(self, family: str, all_signals: dict, alpha: float) -> dict:
        """Pool the candidate with the fit runs over every available signal of the family."""
        cols = [c for c in self.model['columns'] if c['family'] == family
                and available_number(all_signals.get(c['dimension'], {}).get(c['metric'], {}).get('estimate'))]
        if not cols:
            return {'threshold': None, 'columns': [], 'finite_sample_valid': False, 'candidate_z': [],
                    'pvalue': None, 'candidate_max': None}
        fit = np.array([c['fit_harm'] for c in cols]).T
        candidate = np.array([c['direction']*all_signals[c['dimension']][c['metric']]['estimate'] for c in cols])
        result = conformal_decision(fit, candidate, alpha, self.model['scale_floor'])
        result['columns'] = [(c['dimension'], c['metric']) for c in cols]
        return result

    def status(self, dimension: str, signals: dict, alpha: float, *, missing=False, hard_fail=False,
               all_signals: dict | None = None, decision: dict | None = None) -> tuple[str, dict]:
        """Dimension verdict under its family's conformal threshold."""
        _, unavailable = self.z_values(dimension, signals)
        pooled = dict(all_signals or {}); pooled[dimension] = signals
        decision = decision or self.family_decision(family_of(dimension), pooled, alpha)
        own = [z for (d, _), z in zip(decision['columns'], decision['candidate_z']) if d == dimension]
        threshold = decision['threshold']
        if not decision['finite_sample_valid']:
            unavailable.append('Too few fit runs to reject at this alpha.')
        exceeds = bool(own and threshold is not None and max(own) > threshold)
        status = Gate.aggregate(['Fail' if hard_fail or exceeds else 'Pass',
                                 'NotEvaluated' if missing or unavailable else 'Pass'])
        return status, {'max_z_pooled': max(own) if own else None, 'family_threshold_z': threshold,
                        'family': family_of(dimension), 'family_pvalue': decision['pvalue'], 'alpha': alpha,
                        'unavailable': unavailable, 'strict_exceedance': exceeds}

    def apply(self, evidence: dict, alpha: float) -> dict:
        signals_all = evidence['signals']
        decisions = {f: self.family_decision(f, signals_all, alpha) for f in FAMILIES}
        dimensions = {}
        for d in DIMENSIONS:
            source = evidence['dimensions'][d]
            hard = evidence['hard_fail'][d]
            missing = source['status'] == 'NotEvaluated' or bool(source.get('missing_required_evidence'))
            # Diagnostic robustness: any utility verdict flip under this variant's utility rule.
            if d == 'robustness' and source.get('stressors'):
                baseline = dimensions['utility']['status']
                stress = []
                for row in source['stressors']:
                    value = row['utility']
                    collapsed = any('collapsed' in s for s in value.get('weaknesses', []))
                    stressed = {**signals_all, 'utility': utility_signals(value['metrics'])}
                    status, _ = self.status('utility', stressed['utility'], alpha,
                                            missing=value['status'] == 'NotEvaluated', hard_fail=collapsed,
                                            all_signals=stressed)
                    stress.append(status)
                hard = any(s != baseline and 'NotEvaluated' not in (s, baseline) for s in stress)
                missing = missing or baseline == 'NotEvaluated' or 'NotEvaluated' in stress
            decision = decisions[family_of(d)]
            status, audit = self.status(d, signals_all[d], alpha, missing=missing, hard_fail=hard,
                                        all_signals=signals_all, decision=decision)
            if d == 'robustness' and (dimensions['utility']['status'] == 'NotEvaluated'
                                       or source.get('constant_predictor', False)):
                status, hard = 'NotEvaluated', False
                audit['eligibility_reason'] = 'Utility NotEvaluated or constant primary predictor.'
            calibrated_signals = {}
            for (dim, metric), z in zip(decision['columns'], decision['candidate_z']):
                if dim != d: continue
                flag = bool(decision['threshold'] is not None and z > decision['threshold'])
                calibrated_signals[metric] = {'estimate': signals_all[d][metric]['estimate'], 'z_pooled': z,
                                              'family_threshold_z': decision['threshold'], 'flag': flag,
                                              'status': 'Fail' if flag else 'Pass'}
            for c in self.model['columns']:
                if c['dimension'] == d and c['metric'] not in calibrated_signals:
                    calibrated_signals[c['metric']] = {'estimate': None, 'status': 'NotEvaluated', 'flag': False}
            dimensions[d] = {'dimension': d, 'status': status, 'calibration': audit,
                             'role': 'gate' if d in GATE_DIMENSIONS else 'diagnostic',
                             'hard_failure': hard, 'metrics': copy.deepcopy(source['metrics']),
                             'metric_rule_scope': 'Original metric rules retained for provenance; calibrated decisions use calibrated_signals.',
                             'calibrated_signals': calibrated_signals}
        return {'dimensions': dimensions, 'gate_dimensions': list(GATE_DIMENSIONS),
                'gate_family_pvalue': decisions['gate']['pvalue'],
                'gate': Gate.aggregate(dimensions[d]['status'] for d in GATE_DIMENSIONS)}


def self_test() -> None:
    fit, audit = NullCalibrator(CalibrationConfig()).assignment()
    assert len(fit) == len(audit) == 50 and not set(fit) & set(audit)
    assert rejection_rank(50, .05) == 2 and rejection_rank(50, .025) == 1 and rejection_rank(50, .10) == 5
    assert rejection_rank(3, .05) == 0
    assert binomial_admission(5, 50, .05, .05)[1] and not binomial_admission(6, 50, .05, .05)[1]
    assert binomial_admission(0, 50, .05, .05) == (1., True)
    rng = np.random.default_rng(1)
    fit_rows = rng.normal(size=(50, 3))
    assert not conformal_decision(fit_rows, np.zeros(3), .05, 1e-8)['candidate_max'] > 10
    assert conformal_decision(fit_rows, np.array([0, 0, 50.]), .05, 1e-8)['pvalue'] == 1/51
''')

from syntrustbench_campaign import calibration

calibration.self_test()


### The gate and four compensatory judges, with fixed score mapping

In [ ]:
_write_module('judges.py', r'''"""Non-compensatory and compensatory judges on identical saved evidence."""
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass
import math
import numpy as np
import pandas as pd

from .calibration import CalibratedEvaluator, rejection_rank, standardize
from .config import GATE_DIMENSIONS, ExperimentConfig, JudgeConfig
from .core import available_number


COMPENSATORY = ('Mean', 'Geometric', 'Harmonic', 'SDMetrics')
JUDGE_NAMES = ('Gate',) + COMPENSATORY
# What each judge stands for in published practice; cited in the paper's judge table.
JUDGE_PROVENANCE = {
    'Gate': 'Non-compensatory: Pass only if fidelity, utility and privacy each pass (this work; '
            'a fixed-threshold version appears in prior work cited in the paper).',
    'Mean': 'Equal-weight average of the three dimension scores: TabSynDex (Chundawat et al. 2022) and the '
            'default equal weights of the weighted rank sum in Yan et al. 2022.',
    'Geometric': 'Geometric mean, argued to act as an "and" operation: trust index of Belgodere et al. '
                 '(equal weights).',
    'Harmonic': 'Harmonic mean: the most low-score-sensitive of the classical means; strongest compensatory rival.',
    'SDMetrics': 'Native SDMetrics QualityReport overall score (average of property scores), run on each table; '
                 'fidelity-only, as shipped.'}


@dataclass(frozen=True)
class JudgeScore:
    judge: str
    status: str
    score: float | None
    reason: str = ''


class Judge(ABC):
    @abstractmethod
    def score(self, evidence: dict) -> JudgeScore:
        raise NotImplementedError


class GateJudge(Judge):
    def score(self, evidence):
        status = evidence['gate']
        return JudgeScore('Gate', status, None, 'Pass only when every required dimension passes; no ranking score.')


@dataclass
class MeanJudge(Judge):
    name: str = 'Mean'

    def combine(self, values):
        return float(np.mean(values))

    def score(self, evidence):
        values = [evidence['dimension_scores'].get(d) for d in GATE_DIMENSIONS]
        if not all(available_number(v) for v in values):
            return JudgeScore(self.name, 'NotEvaluated', None, 'All three gating dimension scores are required; no missing-value omission.')
        return JudgeScore(self.name, 'Available', self.combine(values))


class GeometricJudge(MeanJudge):
    def __init__(self): super().__init__('Geometric')
    def combine(self, values):
        return 0. if min(values) == 0 else float(math.exp(sum(math.log(v) for v in values)/len(values)))


class HarmonicJudge(MeanJudge):
    def __init__(self): super().__init__('Harmonic')
    def combine(self, values):
        return 0. if min(values) == 0 else float(len(values)/sum(1/v for v in values))


class SDMetricsJudge(Judge):
    def score(self, evidence):
        item = evidence['sdmetrics']
        if not available_number(item.get('score')):
            return JudgeScore('SDMetrics', 'NotEvaluated', None, item.get('reason', 'Quality Report unavailable.'))
        return JudgeScore('SDMetrics', 'Available', item['score'], 'External fidelity-only headline, as shipped; not a multi-dimension safety score.')


def quality_report(reference: pd.DataFrame, synthetic: pd.DataFrame, cfg: ExperimentConfig) -> dict:
    from sdmetrics.reports.single_table import QualityReport
    metadata = {'columns': {c: {'sdtype': 'numerical' if c in cfg.data.continuous else 'categorical'}
                            for c in cfg.data.columns}}
    report = QualityReport()
    report.generate(reference, synthetic, metadata, verbose=False)
    properties = report.get_properties().to_dict(orient='records')
    omitted = []
    for item in properties:
        valid = available_number(item.get('Score'))
        item['status'] = 'Available' if valid else 'NotEvaluated'
        if not valid:
            item['reason'] = 'Native SDMetrics property is nonfinite; overall headline may omit it.'
            omitted.append(item.get('Property', 'unnamed property'))
    score = float(report.get_score())
    return {'score': score if np.isfinite(score) else None, 'properties': properties, 'omitted_properties': omitted,
            'reason': '' if np.isfinite(score) else 'SDMetrics returned a nonfinite score.',
            'native_missingness_policy': 'Native SDMetrics headline retained; omitted property scores recorded explicitly.'}


def dimension_scores(evidence: dict, model: dict, cfg: JudgeConfig) -> tuple[dict, dict]:
    calibrated = CalibratedEvaluator(model)
    scores, notes = {}, {}
    for d in GATE_DIMENSIONS:
        zs, missing = calibrated.z_values(d, evidence['signals'][d])
        if evidence['dimensions'][d]['status'] == 'NotEvaluated':
            missing.append('Required dimension evidence unavailable.')
        if evidence['dimensions'][d].get('missing_required_evidence'):
            missing.extend(evidence['dimensions'][d]['missing_required_evidence'])
        scores[d] = None if missing else float(np.exp(-max(0., max(zs))/cfg.temperature))
        notes[d] = missing
    return scores, notes


def _gate_family(model: dict, evidence: dict):
    """Fit-run and candidate oriented gate signals, or None if any gate signal is missing."""
    cols = [c for c in model['columns'] if c['family'] == 'gate']
    values = [evidence['signals'][c['dimension']].get(c['metric'], {}).get('estimate') for c in cols]
    if not cols or not all(available_number(v) for v in values) or set(model['blocked_dimensions']) & set(GATE_DIMENSIONS):
        return None
    fit = np.array([c['fit_harm'] for c in cols]).T
    return cols, fit, np.array([c['direction']*v for c, v in zip(cols, values)])


def matched_acceptance(judge: Judge, evidence: dict, sdmetrics: dict, model: dict, cfg: ExperimentConfig,
                       alpha: float) -> tuple[bool | None, str]:
    """Accept/reject a table with a compensatory judge calibrated to the gate's honest-data rate.

    Same 50 fit runs, same full-conformal pooling: each run's three dimension scores and
    combined score are computed after pooling the candidate with the fit runs; the table is
    rejected when its combined score is below the floor(alpha(n+1))-th lowest fit-run score.
    SDMetrics scores do not depend on the fit runs, so its threshold is the same order
    statistic of the fit runs' native SDMetrics scores (split conformal).
    """
    if isinstance(judge, GateJudge):
        return None, 'Gate decisions come from the calibrated evaluator.'
    if isinstance(judge, SDMetricsJudge):
        fit = model.get('sdmetrics_fit_scores') or []
        candidate = sdmetrics.get('score')
        if not fit or not all(available_number(v) for v in fit) or not available_number(candidate):
            return None, 'SDMetrics fit-run or candidate score unavailable.'
        fit_scores, cand = np.asarray(fit, dtype=float), float(candidate)
    else:
        if any(evidence['dimensions'][d]['status'] == 'NotEvaluated' or evidence['dimensions'][d].get('missing_required_evidence')
               for d in GATE_DIMENSIONS):
            return None, 'A gating dimension is NotEvaluated.'
        family = _gate_family(model, evidence)
        if family is None:
            return None, 'A gate-family signal is unavailable.'
        cols, fit, candidate = family
        z = standardize(np.vstack([fit, candidate]), model['scale_floor'])
        per_run = np.column_stack([np.exp(-np.maximum(0., z[:, [j for j, c in enumerate(cols) if c['dimension'] == d]].max(axis=1))
                                          / cfg.judges.temperature) for d in GATE_DIMENSIONS])
        combined = np.array([judge.combine(list(row)) for row in per_run])
        fit_scores, cand = combined[:-1], float(combined[-1])
    r = rejection_rank(len(fit_scores), alpha)
    if r < 1:
        return None, 'Too few fit runs to reject at this alpha.'
    threshold = float(np.sort(fit_scores)[r - 1])
    return bool(cand >= threshold), f'matched threshold {threshold:.6g} (rank {r} of {len(fit_scores)})'


def all_judges(cfg: JudgeConfig) -> tuple[Judge, ...]:
    judges = GateJudge(), MeanJudge(), GeometricJudge(), HarmonicJudge(), SDMetricsJudge()
    assert tuple(j.name if hasattr(j, 'name') else type(j).__name__.replace('Judge', '') for j in judges) == JUDGE_NAMES
    return judges


def score_table(profile: dict, model: dict, cfg: ExperimentConfig) -> list[dict]:
    evidence = profile['v04']
    scores, missing = dimension_scores(evidence, model, cfg.judges)
    rows = []
    for alpha in cfg.calibration.target_rates:
        calibrated = CalibratedEvaluator(model).apply(evidence, alpha)
        for variant, evaluated in [('provisional', evidence), ('calibrated', calibrated)]:
            shared = {'gate': evaluated['gate'], 'dimension_scores': scores, 'sdmetrics': profile['sdmetrics']}
            for judge in all_judges(cfg.judges):
                result = asdict(judge.score(shared))
                if variant == 'calibrated':
                    if result['judge'] == 'Gate':
                        matched, why = (None, 'Gate NotEvaluated') if result['status'] == 'NotEvaluated' else (result['status'] == 'Pass', 'calibrated gate')
                    else:
                        matched, why = matched_acceptance(judge, evidence, profile['sdmetrics'], model, cfg, alpha)
                else:
                    matched, why = None, 'Matched-rate decisions are defined for the calibrated variant only.'
                rows.append({**profile['labels'], 'alpha': alpha, 'variant': variant, **result,
                             'accepted_matched': matched, 'matched_note': why,
                             'dimension_scores': scores, 'score_missing': missing})
    return rows


def self_test() -> None:
    e = {'dimension_scores': dict(zip(GATE_DIMENSIONS, [1, 1, 0]))}
    assert abs(MeanJudge().score(e).score - 2/3) < 1e-12
    assert HarmonicJudge().score(e).score == GeometricJudge().score(e).score == 0
    e['dimension_scores'] = dict(zip(GATE_DIMENSIONS, [1, .5, 1]))
    assert MeanJudge().score(e).score > GeometricJudge().score(e).score > HarmonicJudge().score(e).score > .5
    e['dimension_scores']['privacy'] = None
    assert MeanJudge().score(e).status == 'NotEvaluated'
    assert len(all_judges(JudgeConfig())) == len(JUDGE_NAMES) == len(JUDGE_PROVENANCE)
''')

from syntrustbench_campaign import judges

judges.self_test()


### Saved-evidence analysis and report figures

In [ ]:
_write_module('reporting.py', r'''"""Tables and publication figures derived only from saved, verified evidence."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from scipy.stats import norm

from .calibration import CalibratedEvaluator, binomial_admission
from .config import DIMENSIONS, GATE_DIMENSIONS, DIAGNOSTIC_DIMENSIONS, PROTOCOL, ExperimentConfig
from .core import JobLedger, UnavailableAnalysis, available_number, read_json, save_frame, write_json
from .judges import COMPENSATORY, JUDGE_NAMES, JUDGE_PROVENANCE, score_table


def wilson(k: int, n: int) -> tuple[float | None, float | None]:
    if n == 0:
        return None, None
    z = norm.ppf(.975); p = k/n; den = 1 + z*z/n
    mid = (p + z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return float(max(0., mid-half)), float(min(1., mid+half))


def known_bad_group(row: dict) -> str | None:
    """Which known-bad family a table belongs to; None for tables expected to pass."""
    if row.get('kind') == 'control':
        return None if row.get('control') == 'positive' else f"{row['control']}_control"
    if row.get('kind') == 'injection' and row.get('dose', 0) > 0:
        return 'real_with_one_injected_flaw' if row.get('baseline_kind') == 'positive' else 'generator_with_injected_flaw'
    return None


# The headline known-bad set: real patient data with exactly one serious, known flaw.
PRIMARY_BAD = ('real_with_one_injected_flaw', 'copy_control')
BAD_GROUPS = PRIMARY_BAD + ('negative_control', 'generator_with_injected_flaw')


def collect_profiles(ledger: JobLedger, descriptors: list[dict]) -> list[dict]:
    records = []
    for descriptor in descriptors:
        key = descriptor['profile_key']
        state = ledger.state(key)
        if state['state'] == 'completed':
            record = ledger.result(key)
            records.append(read_json(record['profile_path']))
        else:
            records.append({'labels': descriptor['labels'], 'availability': 'Unavailable',
                            'reason': f'{key}: {state["state"]}; {state.get("reason", "missing completed evidence")}'})
    return records


def judge_artifact(cfg: ExperimentConfig, profiles: list[dict], model: dict, folder: Path) -> dict:
    """Score every planned table and every held-out null audit run with every judge."""
    rows = []
    for profile in profiles:
        if profile.get('availability') == 'Unavailable':
            for alpha in cfg.calibration.target_rates:
                for variant in ('provisional', 'calibrated'):
                    for judge in JUDGE_NAMES:
                        rows.append({**profile['labels'], 'alpha': alpha, 'variant': variant, 'judge': judge,
                                     'score': None, 'status': 'NotEvaluated', 'reason': profile['reason']})
        else:
            rows.extend(score_table(profile, model, cfg))
    expected_reps = 1 if cfg.execution.smoke_mode else len(cfg.positive_replicates)
    thresholds = []
    for judge in COMPENSATORY:
        positives = [r for r in rows if r['kind'] == 'control' and r.get('control') == 'positive'
                     and r['judge'] == judge and r['alpha'] == cfg.calibration.primary_rate and r['variant'] == 'calibrated']
        values = [r['score'] for r in positives]
        valid = len(values) == expected_reps and all(available_number(v) for v in values)
        tau = min(values) if valid else None
        thresholds.append({'judge': judge, 'tau_star': tau, 'positive_replicates_expected': expected_reps,
                           'positive_replicates_available': sum(available_number(v) for v in values),
                           'status': 'Available' if valid else 'Unavailable',
                           'reason': '' if valid else 'Every planned positive-control score is required.'})
    for row in rows:
        row['accepted_by_gate'] = row['status'] == 'Pass' if row['judge'] == 'Gate' and row['status'] != 'NotEvaluated' else None
    payload = {'rows': rows, 'thresholds': thresholds,
               'tau_definition': ('minimum compensatory score over the positive-control replicates (real held-out '
                                  'patients), i.e. the most lenient threshold that still accepts all real data; '
                                  'also applied at offsets -0.05 and -0.10'),
               'positive_replicates_note': 'Fixed P pool reused; seeded subsampling/order, not independent patient replicates.'}
    path = write_json(folder / 'judge_evidence.json', payload)
    save_frame(pd.DataFrame(thresholds), folder / 'positive_control_thresholds.csv')
    return {'judge_path': path}


@dataclass
class Analysis:
    cfg: ExperimentConfig
    profiles: list[dict]
    null_profiles: list[dict]
    model: dict
    judges: dict

    def evaluated(self, p: dict, variant: str, alpha: float | None = None) -> tuple[dict, str]:
        """Statuses and gate of one profile under one variant; missing evidence never passes."""
        if p.get('availability') == 'Unavailable':
            return {d: 'NotEvaluated' for d in DIMENSIONS}, 'NotEvaluated'
        if variant == 'official_v03':
            return dict(p['official']['dimension_status']), p['official']['benchmark_gate']
        evidence = p['v04'] if variant == 'provisional' else CalibratedEvaluator(self.model).apply(
            p['v04'], self.cfg.calibration.primary_rate if alpha is None else alpha)
        return {d: evidence['dimensions'][d]['status'] for d in DIMENSIONS}, evidence['gate']

    def status_rows(self) -> list[dict]:
        rows = []
        for p in self.profiles:
            for variant in ('official_v03', 'provisional', 'calibrated'):
                for alpha in (self.cfg.calibration.target_rates if variant == 'calibrated' else (self.cfg.calibration.primary_rate,)):
                    statuses, gate = self.evaluated(p, variant, alpha)
                    rows.append({**p['labels'], 'variant': variant, 'alpha': alpha, **statuses, 'gate': gate,
                                 'gate_rule': ('frozen v0.3: all five dimensions' if variant == 'official_v03'
                                               else 'v0.4.1: fidelity, utility, privacy; equity and robustness diagnostic'),
                                 'availability': p.get('availability', 'Available'), 'reason': p.get('reason', '')})
        return rows

    def control_rows(self) -> list[dict]:
        thresholds = {r['judge']: r['tau_star'] for r in self.judges['thresholds']}
        rows = []
        for r in self.judges['rows']:
            if r['kind'] != 'control': continue
            expected = r['control'] == 'positive'
            for offset in self.cfg.judges.threshold_offsets:
                tau = thresholds.get(r['judge'])
                accepted = r.get('accepted_by_gate') if r['judge'] == 'Gate' else (
                    r['score'] >= tau+offset if available_number(r['score']) and tau is not None else None)
                rows.append({**r, 'threshold_offset': offset, 'threshold': tau+offset if tau is not None else None,
                             'expected_overall_acceptance': expected, 'accepted': accepted,
                             'matches_truth': accepted == expected if accepted is not None else None,
                             'truth_note': 'Copy should pass fidelity/utility but fail privacy.' if r['control'] == 'copy' else r['truth']})
        return rows

    def injections(self) -> tuple[list, list, list]:
        lookup = {p['labels']['table']: p for p in self.profiles}
        by_baseline = {}
        for p in self.profiles:
            label = p['labels']
            if label['kind'] == 'injection':
                by_baseline.setdefault((label['baseline'], label['family']), []).append(p)
        minimum, responses, selectivity = [], [], []
        judge_lookup = {r['table']: r.get('dimension_scores') for r in self.judges['rows']
                        if r['judge'] == 'Mean' and r['variant'] == 'calibrated' and r['alpha'] == self.cfg.calibration.primary_rate}
        intended_map = dict(self.cfg.injections.intended)
        for (baseline, family), members in by_baseline.items():
            base = lookup[baseline]
            members = sorted(members, key=lambda p: p['labels']['dose'])
            intended = intended_map[family]
            for variant in ('provisional', 'calibrated'):
                base_statuses, base_gate = self.evaluated(base, variant)
                before = base_statuses | {'gate': base_gate}
                observed_all = [(p['labels']['dose'], self.evaluated(p, variant)) for p in members if p['labels']['dose'] > 0]
                for d in DIMENSIONS + ('gate',):
                    observed = [(dose, statuses[d] if d != 'gate' else gate) for dose, (statuses, gate) in observed_all]
                    missing = before[d] == 'NotEvaluated' or any(s == 'NotEvaluated' for _, s in observed)
                    detected = [dose for dose, status in observed if status == 'Fail']
                    reason = 'Baseline already fails; incident failure not identifiable.' if before[d] == 'Fail' else (
                        'Required dose/baseline evidence missing.' if missing else ('No failure in tested doses.' if not detected else ''))
                    minimum.append({'baseline': baseline, 'family': family, 'variant': variant, 'dimension': d,
                                    'role': 'gate' if d in GATE_DIMENSIONS + ('gate',) else 'diagnostic',
                                    'intended': d in intended, 'baseline_status': before[d],
                                    'minimum_detected_dose': None if missing or before[d] == 'Fail' else min(detected, default=None),
                                    'minimum_observed_absolute_failure_dose': 0. if before[d] == 'Fail' else min(detected, default=None),
                                    'status': 'NotEvaluated' if missing or before[d] == 'Fail' else 'Available', 'reason': reason})
            base_scores = judge_lookup.get(baseline) or {}
            for p in members:
                label = p['labels']
                if label['dose'] == 0: continue
                scores = judge_lookup.get(label['table']) or {}
                delta = {d: base_scores[d]-scores[d] if available_number(base_scores.get(d)) and available_number(scores.get(d)) else None
                         for d in GATE_DIMENSIONS}
                responses.extend({**label, 'dimension': d, 'response': delta[d], 'intended': d in intended,
                                  'status': 'Available' if delta[d] is not None else 'NotEvaluated',
                                  'reason': '' if delta[d] is not None else 'Baseline or injected dimension score unavailable.'}
                                 for d in GATE_DIMENSIONS)
                off_target = [d for d in GATE_DIMENSIONS if d not in intended]
                available = all(v is not None for v in delta.values())
                score = (max(delta[d] for d in intended)-(max(delta[d] for d in off_target) if off_target else 0.)) if available else None
                selectivity.append({**label, 'selectivity': score, 'status': 'Available' if available else 'NotEvaluated',
                                    'definition': 'maximum intended score decrease minus maximum off-target score decrease (gating dimensions)'})
        return minimum, responses, selectivity

    def audit_runs(self) -> list[dict]:
        audit_ids = set(self.model['audit_ids'])
        return [p for p in self.null_profiles if p['labels']['repeat'] in audit_ids]

    def false_alarms(self) -> list[dict]:
        """Honest-data rejection on held-out audit runs, per dimension and for each gate."""
        rows = []
        audits = self.audit_runs(); expected = len(self.model['audit_ids'])
        plans = [('official_v03', self.cfg.calibration.primary_rate), ('provisional', self.cfg.calibration.primary_rate)]
        plans += [('calibrated', a) for a in self.cfg.calibration.target_rates]
        for variant, alpha in plans:
            evaluated = [self.evaluated(p, variant, alpha) for p in audits]
            for d in DIMENSIONS + ('gate',):
                statuses = [gate if d == 'gate' else s[d] for s, gate in evaluated]
                evaluable = [s for s in statuses if s != 'NotEvaluated']
                k, n = evaluable.count('Fail'), len(evaluable)
                low, high = wilson(k, n)
                complete = n == expected
                rows.append({'variant': variant, 'alpha': alpha, 'dimension': d,
                             'role': 'gate' if d in GATE_DIMENSIONS + ('gate',) else 'diagnostic',
                             'audit_runs_expected': expected, 'evaluable_runs': n,
                             'missing_runs': expected-n, 'failures': k,
                             'false_alarm_rate': k/n if complete and n else None,
                             'observed_rate_among_evaluable': k/n if n else None, 'ci_low': low, 'ci_high': high,
                             'not_accepted_rate': sum(s != 'Pass' for s in statuses)/len(statuses) if statuses else None,
                             'status': 'Available' if complete else 'Unavailable',
                             'reason': '' if complete else 'Some audit runs were not evaluable; subset rate explicitly labeled.'})
        return rows

    def admission(self) -> list[dict]:
        """Prespecified rule: a dimension may gate only if, on every held-out honest audit run,
        it is evaluable and its non-acceptance shows no significant excess over the target rate."""
        c = self.cfg.calibration
        audits = self.audit_runs(); expected = len(self.model['audit_ids'])
        rows = []
        for variant in ('official_v03', 'provisional', 'calibrated'):
            evaluated = [self.evaluated(p, variant, c.primary_rate) for p in audits]
            for d in DIMENSIONS + ('gate',):
                statuses = [gate if d == 'gate' else s[d] for s, gate in evaluated]
                n = len(statuses)
                not_evaluable = sum(s == 'NotEvaluated' for s in statuses)
                not_accepted = sum(s != 'Pass' for s in statuses)
                pvalue, no_excess = binomial_admission(not_accepted, n, c.primary_rate, c.admission_test_alpha) if n else (None, False)
                low, high = wilson(not_accepted, n)
                complete = n == expected and n > 0
                meets = bool(complete and not_evaluable == 0 and no_excess)
                rows.append({'variant': variant, 'dimension': d,
                             'declared_role': ('gate' if d in GATE_DIMENSIONS + ('gate',) else 'diagnostic') if variant != 'official_v03' else 'gate (v0.3)',
                             'audit_runs': n, 'audit_runs_expected': expected, 'not_evaluable_runs': not_evaluable,
                             'honest_runs_not_accepted': not_accepted,
                             'not_accepted_rate': not_accepted/n if n else None, 'ci_low': low, 'ci_high': high,
                             'target_rate': c.primary_rate, 'binomial_excess_pvalue': pvalue,
                             'meets_admission_rule': meets,
                             'rule': (f'every audit run evaluable and one-sided exact binomial p >= {c.admission_test_alpha} '
                                      f'for excess non-acceptance over {c.primary_rate}')})
        return rows

    def comparison(self) -> tuple[list, list, list, list]:
        """Known-bad and honest-data acceptance per judge under two prespecified schemes.

        matched (primary): every judge is calibrated on the same fit null runs to the same
        honest-data rejection level, so known-bad acceptance is compared at equal cost.
        tau_star (secondary): compensatory threshold = lowest positive-control score, at offsets.
        """
        thresholds = {r['judge']: r['tau_star'] for r in self.judges['thresholds']}
        primary_alpha = self.cfg.calibration.primary_rate

        def decide(r, scheme, offset):
            if scheme == 'matched':
                return r.get('accepted_matched')
            if r['judge'] == 'Gate':
                return r.get('accepted_by_gate')
            tau = thresholds.get(r['judge'])
            return r['score'] >= tau+offset if available_number(r['score']) and tau is not None else None

        def summary(selected, judge, scheme, alpha, offset, label):
            values = [decide(r, scheme, offset) for r in selected]
            usable = [v for v in values if v is not None]
            complete = len(usable) == len(values) and bool(values)
            low, high = wilson(sum(usable), len(usable))
            return {'scheme': scheme, 'alpha': alpha, 'offset': offset, 'judge': judge, 'set': label,
                    'tau_star': thresholds.get(judge) if scheme == 'tau_star' else None,
                    'tables_expected': len(values), 'tables_evaluable': len(usable), 'accepted': sum(usable),
                    'acceptance_rate': sum(usable)/len(usable) if complete else None, 'ci_low': low, 'ci_high': high,
                    'status': 'Available' if complete else 'NotEvaluated',
                    'reason': '' if complete else 'Missing decision/threshold or no tables in this set.'}

        plans = [('matched', a, 0.) for a in self.cfg.calibration.target_rates]
        plans += [('tau_star', primary_alpha, o) for o in self.cfg.judges.threshold_offsets]
        acceptance, honest, by_family, rankings = [], [], [], []
        for scheme, alpha, offset in plans:
            rows = [r for r in self.judges['rows'] if r['variant'] == 'calibrated' and r['alpha'] == alpha]
            for judge in JUDGE_NAMES:
                selected = [r for r in rows if r['judge'] == judge]
                bad = {g: [r for r in selected if known_bad_group(r) == g] for g in BAD_GROUPS}
                primary = [r for g in PRIMARY_BAD for r in bad[g]]
                every_bad = [r for g in BAD_GROUPS for r in bad[g]]
                acceptance.append(summary(primary, judge, scheme, alpha, offset,
                                          'primary: real data with one serious flaw (injections on real + copy control)'))
                for g in BAD_GROUPS:
                    acceptance.append(summary(bad[g], judge, scheme, alpha, offset, g))
                acceptance.append(summary(every_bad, judge, scheme, alpha, offset, 'all known-bad tables'))
                honest.append(summary([r for r in selected if r['kind'] == 'null'], judge, scheme, alpha, offset,
                                      'held-out honest real-vs-real audit runs'))
                honest.append(summary([r for r in selected if r['kind'] == 'control' and r.get('control') == 'positive'],
                                      judge, scheme, alpha, offset, 'positive controls'))
                for family, doses in self.cfg.injections.doses:
                    for dose in doses:
                        members = [r for r in bad['real_with_one_injected_flaw'] if r['family'] == family and r['dose'] == dose]
                        by_family.append({**summary(members, judge, scheme, alpha, offset, 'real_with_one_injected_flaw'),
                                          'family': family, 'dose': dose})
        rows = [r for r in self.judges['rows'] if r['variant'] == 'calibrated' and r['alpha'] == primary_alpha]
        for judge in JUDGE_NAMES:
            selected = [r for r in rows if r['judge'] == judge]
            every_bad = [r for r in selected if known_bad_group(r) is not None]
            clean = [r for r in selected if r['kind'] == 'clean']
            pairs = [(a['score'], b['score']) for a in every_bad for b in clean]
            valid = [(a, b) for a, b in pairs if available_number(a) and available_number(b)]
            complete = len(valid) == len(pairs) and bool(pairs) and judge != 'Gate'
            inversions = sum(a > b for a, b in valid)
            rankings.append({'judge': judge, 'pairs_expected': len(pairs), 'pairs_evaluable': len(valid),
                             'inversions': inversions, 'inversion_fraction': inversions/len(valid) if complete else None,
                             'definition': 'known-bad table scored strictly above a clean generator table',
                             'status': 'Available' if complete else 'NotEvaluated',
                             'reason': 'Gate has no scalar ranking.' if judge == 'Gate' else '' if complete else 'Missing pair evidence; no omission.'})
        return acceptance, honest, by_family, rankings

    def sensitivity(self) -> list[dict]:
        rows = []
        for p in self.profiles:
            evaluated = [self.evaluated(p, 'calibrated', a) for a in self.cfg.calibration.target_rates]
            for d in DIMENSIONS + ('gate',):
                values = [gate if d == 'gate' else s[d] for s, gate in evaluated]
                baseline = values[list(self.cfg.calibration.target_rates).index(self.cfg.calibration.primary_rate)]
                available = 'NotEvaluated' not in values
                rows.append({**p['labels'], 'dimension': d, 'statuses_by_alpha': dict(zip(self.cfg.calibration.target_rates, values)),
                             'fraction_changed_from_primary': sum(v != baseline for v in values)/len(values) if available else None,
                             'any_change': len(set(values)) > 1 if available else None,
                             'status': 'Available' if available else 'NotEvaluated'})
        return rows

    def build(self, cohort: dict, folder: Path, roles: dict | None = None) -> dict:
        minimum, responses, selectivity = self.injections()
        acceptance, honest, by_family, rankings = self.comparison()
        status = self.status_rows()
        formal = []
        for profile in self.profiles:
            if profile['labels']['kind'] != 'clean':
                continue
            privacy = profile.get('generator_provenance', {}).get('formal_privacy',
                {'status':'Unavailable', 'reason':profile.get('reason','Generator provenance missing.')})
            formal.append({**profile['labels'], **privacy})
        tables = {'gate_admission': self.admission(),
                  'judge_acceptance': acceptance, 'honest_data_acceptance': honest,
                  'acceptance_by_injection': by_family, 'ranking_inversions': rankings,
                  'judge_provenance': [{'judge': j, 'represents': JUDGE_PROVENANCE[j]} for j in JUDGE_NAMES],
                  'generator_formal_privacy': formal,
                  'dimension_status': status, 'clean_profiles': [r for r in status if r['kind'] == 'clean'],
                  'controls_by_judge': self.control_rows(), 'injection_minimum_dose': minimum,
                  'injection_responses': responses, 'injection_selectivity': selectivity,
                  'audit_false_positive_rates': self.false_alarms(), 'threshold_sensitivity': self.sensitivity()}
        metrics = []
        for profile in self.profiles:
            if profile.get('availability') == 'Unavailable': continue
            for variant, dimensions in [('official_v03',profile['official']['dimensions']),('provisional',profile['v04']['dimensions'])]:
                for d, value in dimensions.items():
                    metrics.extend({**profile['labels'], 'variant':variant, 'dimension':d, 'metric':name, **m}
                                   for name,m in value['metrics'].items())
        tables['metric_evidence'] = metrics
        tables['calibrated_metric_thresholds'] = self.model['raw_cutoffs']
        tables['null_role_sizes'] = [{**p['labels'], **p['rows']} for p in self.null_profiles if p.get('availability') != 'Unavailable']
        paths = {name: save_frame(pd.DataFrame(rows), folder / f'{name}.csv') for name, rows in tables.items()}
        for name in ('cohort_split_summary', 'missingness', 'subgroup_counts'):
            source = Path(cohort['folder']) / f'{name}.csv'
            shutil.copyfile(source, folder / source.name); paths[name] = str(folder / source.name)
        if roles is not None:
            cohort_table = pd.read_csv(paths['cohort_split_summary'])
            extra = pd.DataFrame([{'section':'v04_role', 'label':k, 'rows':v}
                                  for k,v in roles['role_counts'].items()])
            save_frame(pd.concat([cohort_table,extra], ignore_index=True), paths['cohort_split_summary'])
        evidence = write_json(folder / 'analysis_tables.json', tables)
        return {'tables': paths, 'evidence': evidence, 'source': 'verified saved profiles only; no fitting in analysis'}


@dataclass
class Reporter:
    cfg: ExperimentConfig

    @staticmethod
    def _pct(value) -> str:
        return 'NA' if not available_number(value) else f'{100*value:.1f}%'

    def headline(self, tables: dict) -> list[str]:
        """Plain-language results computed from the saved tables; no hand-written numbers."""
        c = self.cfg.calibration
        lines = ['## Headline results (computed from the tables below)', '']
        lines += ['### 1. Which dimensions may gate (prespecified admission rule)', '',
                  '| Variant | Dimension | Declared role | Honest audit runs not accepted | 95% CI | Not evaluable | Meets rule |',
                  '|---|---|---|---|---|---|---|']
        for r in tables['gate_admission']:
            lines.append(f"| {r['variant']} | {r['dimension']} | {r['declared_role']} | {r['honest_runs_not_accepted']}/{r['audit_runs']} "
                         f"({self._pct(r['not_accepted_rate'])}) | {self._pct(r['ci_low'])}–{self._pct(r['ci_high'])} | "
                         f"{r['not_evaluable_runs']} | {'yes' if r['meets_admission_rule'] else 'NO'} |")
        declared = [r for r in tables['gate_admission'] if r['variant'] == 'calibrated' and r['declared_role'] == 'gate']
        ok = bool(declared) and all(r['meets_admission_rule'] for r in declared)
        lines += ['', ('All declared gating dimensions and the calibrated gate meet the admission rule.' if ok else
                       '**WARNING: at least one declared gating dimension or the calibrated gate fails the admission rule. '
                       'The declared three-dimension gate is not supported as specified; report this, do not re-tune.**'), '']
        get = lambda rows, judge, scheme, alpha, offset, prefix: next((r for r in rows if r['judge'] == judge and r['scheme'] == scheme
                                                and r['alpha'] == alpha and r['offset'] == offset and r['set'].startswith(prefix)), None)
        def cell(r):
            return 'NA' if r is None or r['acceptance_rate'] is None else f"{r['accepted']}/{r['tables_evaluable']} ({self._pct(r['acceptance_rate'])})"
        for scheme, offset, title, note in (
                ('matched', 0., f'### 2. PRIMARY: every judge calibrated to the same honest-data rate (alpha {c.primary_rate}, same fit runs)',
                 'All judges share the same full-conformal calibration on the same fit runs, so honest-data rejection is matched and known-bad acceptance is compared at equal cost.'),
                ('tau_star', 0., '### 3. SECONDARY: compensatory threshold tau* = lowest positive-control score',
                 'tau* accepts all positive controls by construction; it leaves each judge at a different honest-data rejection rate, so this comparison is not at matched cost.')):
            lines += [title, '', '| Judge | Real data with one serious flaw accepted | All known-bad accepted | Honest audit runs accepted | Positive controls accepted |',
                      '|---|---|---|---|---|']
            for judge in JUDGE_NAMES:
                cells = [cell(get(tables['judge_acceptance'], judge, scheme, c.primary_rate, offset, 'primary')),
                         cell(get(tables['judge_acceptance'], judge, scheme, c.primary_rate, offset, 'all known-bad')),
                         cell(get(tables['honest_data_acceptance'], judge, scheme, c.primary_rate, offset, 'held-out')),
                         cell(get(tables['honest_data_acceptance'], judge, scheme, c.primary_rate, offset, 'positive'))]
                lines.append(f'| {judge} | ' + ' | '.join(cells) + ' |')
            lines += ['', note, '']
        return lines

    def build(self, analysis: dict, folder: Path) -> dict:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        from matplotlib.colors import ListedColormap, BoundaryNorm
        from matplotlib.patches import Patch, Rectangle
        tables = read_json(analysis['evidence'])
        figures = {}
        columns = list(GATE_DIMENSIONS) + ['gate'] + list(DIAGNOSTIC_DIMENSIONS)
        labels_x = list(GATE_DIMENSIONS) + ['GATE'] + [f'{d}\n(diagnostic)' for d in DIAGNOSTIC_DIMENSIONS]
        statuses = [r for r in tables['dimension_status'] if r['kind'] in ('clean', 'control')
                    and r['variant'] == 'calibrated' and r['alpha'] == self.cfg.calibration.primary_rate]
        if statuses:
            codes = {'NotEvaluated': -1, 'Pass': 0, 'Conditional': 1, 'Fail': 2}
            matrix = np.array([[codes[r[d]] for d in columns] for r in statuses])
            fig, ax = plt.subplots(figsize=(10, max(4, .35*len(statuses)+1.5)))
            cmap = ListedColormap(['#d8dce2', '#78baa3', '#efc56c', '#d97979'])
            ax.imshow(matrix, cmap=cmap, norm=BoundaryNorm([-1.5,-.5,.5,1.5,2.5], cmap.N), aspect='auto')
            ax.set_xticks(range(len(columns)), labels_x); ax.set_yticks(range(len(statuses)), [r['table'] for r in statuses])
            ax.axvline(len(GATE_DIMENSIONS)+.5, color='black', linewidth=1.5)
            for i in range(len(statuses)):
                for j, d in enumerate(columns): ax.text(j, i, statuses[i][d], ha='center', va='center', fontsize=8)
            ax.set_title('Calibrated status. The gate uses fidelity, utility and privacy;\n'
                         'diagnostics (right of the line) are reported only. NA never counts as Pass.')
            figures['status_heatmap'] = self._save(fig, folder, 'status_heatmap'); plt.close(fig)
        else:
            figures['status_heatmap'] = {'status': 'Unavailable', 'reason': 'No planned clean/control status rows.'}
        response = pd.DataFrame(tables['injection_responses'])
        if not response.empty:
            grouped = []
            for keys, group in response.groupby(['baseline_kind', 'family', 'dose', 'dimension'], dropna=False):
                values = group.response.tolist()
                complete = all(available_number(v) for v in values)
                grouped.append(dict(zip(['baseline_kind','family','dose','dimension'], keys)) | {'response': float(np.mean(values)) if complete else None})
            g = pd.DataFrame(grouped)
            g['label'] = g.apply(lambda r: f'{r.baseline_kind} / {r.family} / {r.dose:g}', axis=1)
            labels = list(dict.fromkeys(g.label))
            matrix = np.array([[next(r['response'] for r in grouped if f"{r['baseline_kind']} / {r['family']} / {r['dose']:g}" == label and r['dimension'] == d)
                                for d in GATE_DIMENSIONS] for label in labels], dtype=float)
            fig, ax = plt.subplots(figsize=(8, max(4, .35*len(labels)+1.7)))
            cmap = plt.get_cmap('RdBu_r').copy(); cmap.set_bad('#d8dce2')
            im = ax.imshow(np.ma.masked_invalid(matrix), cmap=cmap, vmin=-1, vmax=1, aspect='auto')
            ax.set_xticks(range(len(GATE_DIMENSIONS)), GATE_DIMENSIONS); ax.set_yticks(range(len(labels)), labels)
            for i, label in enumerate(labels):
                family = label.split(' / ')[1]
                for j, d in enumerate(GATE_DIMENSIONS):
                    ax.text(j, i, f'{matrix[i,j]:.2f}' if np.isfinite(matrix[i,j]) else 'NA', ha='center', va='center', fontsize=8)
                    if d in dict(self.cfg.injections.intended)[family]:
                        ax.add_patch(Rectangle((j-.49,i-.49), .98,.98, fill=False, edgecolor='black', linewidth=1.5))
            ax.set_title('Injection response: baseline score − injected score\nIntended dimensions outlined; missing comparisons gray')
            fig.colorbar(im, ax=ax, label='Score decrease (higher = more harm)')
            figures['injection_selectivity_heatmap'] = self._save(fig, folder, 'injection_selectivity_heatmap'); plt.close(fig)
        else:
            figures['injection_selectivity_heatmap'] = {'status': 'Unavailable', 'reason': 'No injection evidence.'}
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
        alphas = list(self.cfg.calibration.target_rates)
        colors = ('#317a95', '#b07126', '#884c8c')
        pick = lambda rows, judge, alpha, prefix: next((r for r in rows if r['scheme'] == 'matched' and r['judge'] == judge
                                                        and r['alpha'] == alpha and r['set'].startswith(prefix)), None)
        panels = [(tables['judge_acceptance'], 'primary', 'Known-bad accepted (real data, one serious flaw)\nlower is better'),
                  (tables['honest_data_acceptance'], 'held-out', 'Honest data accepted (held-out audit runs)\nhigher is better')]
        for ax, (rows, prefix, title) in zip(axes[:2], panels):
            for j, alpha in enumerate(alphas):
                xs = np.arange(len(JUDGE_NAMES)) + (j-1)*.24
                for x, judge in zip(xs, JUDGE_NAMES):
                    row = pick(rows, judge, alpha, prefix)
                    value = None if row is None else row['acceptance_rate']
                    if available_number(value):
                        ax.bar(x, value, width=.22, color=colors[j])
                    else:
                        ax.text(x, .03, 'NA', rotation=90, ha='center', fontsize=8, color=colors[j])
            ax.set_xticks(range(len(JUDGE_NAMES)), JUDGE_NAMES); ax.set_ylim(0, 1); ax.set_title(title, fontsize=10)
        axes[0].set_ylabel('Fraction accepted')
        axes[1].legend(handles=[Patch(facecolor=colors[j], label=f'matched alpha {a:g}') for j, a in enumerate(alphas)], fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.1), ncol=3, frameon=False)
        markers = dict(zip(JUDGE_NAMES, ('o', 's', '^', 'v', 'D')))
        for judge in JUDGE_NAMES:
            points = []
            for alpha in alphas:
                bad, ok = pick(tables['judge_acceptance'], judge, alpha, 'primary'), pick(tables['honest_data_acceptance'], judge, alpha, 'held-out')
                if bad and ok and available_number(bad['acceptance_rate']) and available_number(ok['acceptance_rate']):
                    points.append((1-ok['acceptance_rate'], bad['acceptance_rate']))
            if points:
                axes[2].plot(*zip(*points), marker=markers[judge], label=judge)
        axes[2].set_xlabel('Honest data rejected (audit)'); axes[2].set_ylabel('Known-bad accepted (primary set)')
        axes[2].set_xlim(0, 1); axes[2].set_ylim(0, 1); axes[2].legend(fontsize=8)
        axes[2].set_title('Error trade-off across matched alphas\n(bottom-left is best)', fontsize=10)
        figures['judge_comparison'] = self._save(fig, folder, 'judge_comparison'); plt.close(fig)
        lines = [f'# SynTrustBench {PROTOCOL} campaign report', '',
                 '**Smoke execution only; not paper evidence.**' if self.cfg.execution.smoke_mode else 'Full campaign evidence; scientific acceptance requires reviewing the controls and audit.', '']
        lines += self.headline(tables)
        lines += ['## Tables', '']
        lines += [f'- [{name}]({path})' for name, path in analysis['tables'].items()]
        lines += ['', '## Figures', '']
        for name, value in figures.items():
            lines.append(f'![{name}]({value["png"]})' if value['status'] == 'Available' else f'{name}: Unavailable — {value["reason"]}')
        lines += ['', '## Interpretation boundaries', '',
                  'The gate decides on fidelity, utility and privacy only. Equity and robustness are computed and reported as diagnostics; they never change the verdict.',
                  'Frozen v0.3 is reported unmodified as the uncalibrated five-dimension baseline; its exact-row refusals are recorded as NotEvaluated.',
                  'Generator epsilon/delta accounting is in generator_formal_privacy. It is separate from empirical Privacy verdicts and does not cover this benchmark report or the joint release of multiple fits.',
                  'No clinical certification or formal privacy guarantee. Utility is the declared readmission prediction task on one public cohort.',
                  'Reference detection CIs omit refit variance. Native categorical encoders pool rare levels beyond 250.',
                  'Null repetitions reuse development patients and use smaller roles than the main campaign; the calibration guarantee assumes exchangeability, which the held-out audit tests.',
                  'Compensatory score normalization and positive-control thresholds are prespecified design choices, not externally validated utility functions.',
                  'No result is inferred from execution coverage. Missing values remain unavailable; inspect denominators and reasons.']
        (folder / 'report.md').write_text('\n'.join(lines) + '\n')
        return {'report': str(folder / 'report.md'), 'figures': figures, 'tables': analysis['tables']}

    @staticmethod
    def _save(fig, folder, name) -> dict:
        fig.tight_layout()
        paths = {ext: str(folder / f'{name}.{ext}') for ext in ('png', 'pdf')}
        for ext, path in paths.items(): fig.savefig(path, dpi=160, bbox_inches='tight')
        return {'status': 'Available', **paths}


def self_test() -> None:
    low, high = wilson(0, 20)
    assert low <= 1e-12 and 0 < high < 1
    assert known_bad_group({'kind': 'control', 'control': 'positive'}) is None
    assert known_bad_group({'kind': 'control', 'control': 'copy'}) == 'copy_control'
    assert known_bad_group({'kind': 'injection', 'dose': .1, 'baseline_kind': 'positive'}) == 'real_with_one_injected_flaw'
    assert known_bad_group({'kind': 'injection', 'dose': 0., 'baseline_kind': 'positive'}) is None
    assert known_bad_group({'kind': 'null'}) is None
''')

from syntrustbench_campaign import reporting

reporting.self_test()


### Instrument checks and original reference regression fixture

In [ ]:
_write_module('original_reference.py.txt', r'''"""Reference-calibrated Fidelity and Utility (experimental v0.4 add-on).

This module does not modify the frozen v0.3 dimensions. It adds three things
that v0.3 lacks, each tied to an established method:

1. A held-out real reference.  Fidelity is measured against real rows the
   generator never saw (as Synthcity's Benchmarks do with ``X.test()``), and
   every fidelity statistic is paired with the same statistic computed for an
   independent *real* sample of the same population (the "real-vs-real floor").
   A synthetic table is judged by its excess over what sampling alone produces.
2. Mass-weighted category coverage.  v0.3 counts category *levels*, so a level
   seen once weighs as much as a common one and a second real sample fails the
   0.95 rule.  Here coverage is the share of reference *patients* whose level
   appears in the synthetic table.
3. A joint-distribution test.  v0.3 checks marginals and pairwise associations
   only.  A cross-fitted classifier two-sample test (Lopez-Paz & Oquab, 2017;
   Synthcity ``detection_*``) and the propensity-score MSE (Snoke et al., 2018)
   are added, again relative to the real-vs-real floor.

Utility keeps the v0.3 primary rule (random-forest chance-corrected retention)
so official verdicts stay comparable, and adds: a real-reference retention
(positive control inside the dimension), a probe family (random forest,
logistic regression, gradient boosting, as in Synthcity's performance metrics),
and calibration (calibration-in-the-large and slope; Van Calster et al., 2019),
each relative to a model trained on real data.

All margins below are provisional, are recorded in the output, and are meant
to be replaced by empirically calibrated cut-offs from repeated real-vs-real
runs. Detection can raise Conditional but never Fail until that calibration
exists, because with tens of thousands of rows almost any difference is
statistically detectable.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from .constraints import constraint_violations
from .fidelity import _association_error, _jensen_shannon, _normalized_wasserstein
from .stats import (
    FeatureEncoder,
    aggregate_status,
    categorical_values,
    metric,
    percentile_interval,
    safe_auroc,
)
from .validation import load_submission

PROVISIONAL = {
    # Fidelity
    "coverage_margin": 0.05,          # tolerated loss of reference patient mass beyond the floor
    "detection_margin": 0.05,         # AUC excess over the floor that raises Conditional
    "wasserstein_excess": 0.20,       # v0.3 thresholds, now applied to excess over floor
    "jsd_excess": 0.10,
    "association_excess": 0.15,
    "constraint_rate_fail": 0.01,     # unchanged from v0.3
    # Utility
    "retention_fail": 0.80,           # unchanged from v0.3
    "retention_conditional": 0.90,
    "calibration_ratio_band": 1.25,   # relative O/E and slope must stay within [1/1.25, 1.25]
}
MAX_TREE_LEVELS = 250  # HistGradientBoosting native categoricals allow at most 255 levels


# --------------------------------------------------------------------------- encoders
@dataclass
class _TreeEncoder:
    """Integer codes for native-categorical gradient boosting.

    Levels outside the ``MAX_TREE_LEVELS`` most frequent levels of the fitting
    table are pooled into one ``__OTHER__`` code (stated limitation: rare-level
    detail is invisible to this encoder; mass-weighted coverage covers it).
    """

    continuous: list[str]
    categorical: list[str]
    levels: dict[str, list[str]] = field(default_factory=dict)

    def fit(self, frame: pd.DataFrame) -> "_TreeEncoder":
        self.levels = {
            c: categorical_values(frame[c]).value_counts().index[:MAX_TREE_LEVELS].tolist()
            for c in self.categorical
        }
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        out = frame[self.continuous].apply(pd.to_numeric, errors="coerce").copy()
        for c in self.categorical:
            values = categorical_values(frame[c])
            values = values.where(values.isin(self.levels[c]), "__OTHER__")
            out[c] = pd.Categorical(values, categories=self.levels[c] + ["__OTHER__"]).codes
        return out.reset_index(drop=True)

    @property
    def categorical_mask(self) -> list[bool]:
        return [False] * len(self.continuous) + [True] * len(self.categorical)


def _hgb(seed: int, mask: list[bool]) -> HistGradientBoostingClassifier:
    return HistGradientBoostingClassifier(
        max_iter=200, categorical_features=mask, random_state=seed
    )


# --------------------------------------------------------------------------- fidelity pieces
def _mass_coverage(reference: pd.Series, candidate: pd.Series) -> np.ndarray:
    """Per-reference-row indicator: is this patient's level present in the candidate table?"""
    present = set(categorical_values(candidate))
    return categorical_values(reference).isin(present).to_numpy(dtype=float)


def _two_sample_detection(
    reference: pd.DataFrame,
    candidate: pd.DataFrame,
    encoder: _TreeEncoder,
    seed: int,
    folds: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """Cross-fitted classifier two-sample test; returns labels and out-of-fold scores."""
    features = pd.concat(
        [encoder.transform(reference), encoder.transform(candidate)], ignore_index=True
    )
    labels = np.r_[np.zeros(len(reference)), np.ones(len(candidate))]
    scores = np.zeros(len(labels))
    splitter = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    for train_rows, test_rows in splitter.split(features, labels):
        model = _hgb(seed, encoder.categorical_mask).fit(
            features.iloc[train_rows], labels[train_rows]
        )
        scores[test_rows] = model.predict_proba(features.iloc[test_rows])[:, 1]
    return labels, scores


def _pmse(labels: np.ndarray, scores: np.ndarray) -> float:
    share = float(labels.mean())
    return float(np.mean((scores - share) ** 2))


def _distance_block(reference, candidate, continuous, categorical) -> dict[str, float]:
    wass = [_normalized_wasserstein(reference[c], candidate[c]) for c in continuous]
    wass = [w for w in wass if np.isfinite(w)]
    jsd = [_jensen_shannon(reference[c], candidate[c])[0] for c in categorical]
    per_column_wass = {
        c: _normalized_wasserstein(reference[c], candidate[c]) for c in continuous
    }
    return {
        "mean_normalized_wasserstein": float(np.mean(wass)) if wass else float("nan"),
        "worst_column_normalized_wasserstein": float(np.nanmax(list(per_column_wass.values())))
        if per_column_wass else float("nan"),
        "worst_column": max(per_column_wass, key=lambda c: np.nan_to_num(per_column_wass[c], nan=-1))
        if per_column_wass else None,
        "mean_jensen_shannon_divergence": float(np.mean(jsd)) if jsd else float("nan"),
        "association_matrix_error": _association_error(
            reference, candidate, continuous, categorical
        ),
    }


# --------------------------------------------------------------------------- utility pieces
def _fit_predict(probe: str, train: pd.DataFrame, test: pd.DataFrame, target: str,
                 onehot: FeatureEncoder, tree: _TreeEncoder, seed: int) -> np.ndarray:
    y = train[target].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return np.full(len(test), float(y.mean()))
    if probe == "random_forest":  # identical to the v0.3 primary probe
        model = RandomForestClassifier(
            n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=seed
        )
        return model.fit(onehot.transform(train), y).predict_proba(onehot.transform(test))[:, 1]
    if probe == "logistic_regression":
        model = LogisticRegression(C=1.0, max_iter=3000)
        return model.fit(onehot.transform(train), y).predict_proba(onehot.transform(test))[:, 1]
    if probe == "gradient_boosting":
        model = _hgb(seed, tree.categorical_mask)
        return model.fit(tree.transform(train), y).predict_proba(tree.transform(test))[:, 1]
    raise ValueError(probe)


def _calibration(y: np.ndarray, p: np.ndarray) -> tuple[float, float]:
    """Calibration-in-the-large (observed/expected) and calibration slope."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    if len(np.unique(y)) < 2 or np.allclose(p, p[0]):
        return float("nan"), float("nan")
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    slope = LogisticRegression(C=1e6, max_iter=2000).fit(logit, y).coef_[0][0]
    return float(y.mean() / p.mean()), float(slope)


def _retention(y, base, cand):
    real, syn = safe_auroc(y, base), safe_auroc(y, cand)
    return (syn - 0.5) / (real - 0.5) if np.isfinite(real) and real > 0.5 else float("nan")


def _retention_verdict(point, low, high, base_low) -> str:
    if base_low is None or not np.isfinite(base_low) or base_low <= 0.5:
        return "NotEvaluated"
    if high is not None and high < PROVISIONAL["retention_fail"]:
        return "Fail"
    if low is None or high is None:
        return "NotEvaluated"
    return "Pass" if low >= PROVISIONAL["retention_conditional"] else "Conditional"


# --------------------------------------------------------------------------- main context
class ReferenceContext:
    """Everything that depends only on real data, computed once per cohort.

    ``real_train`` is the generator's training data, ``real_reference`` an
    independent real sample of the same population and size (e.g. the
    calibration pool), ``real_test`` the untouched utility test set.
    """

    PROBES = ("random_forest", "logistic_regression", "gradient_boosting")

    def __init__(self, real_train, real_reference, real_test, config, *,
                 iterations: int | None = None, seed: int | None = None):
        # Reuse frozen v0.3 validation: types, target mapping, split-overlap checks.
        base = load_submission(real_train, real_test, real_reference, config)
        ref_check = load_submission(real_train, real_reference, real_test, config)
        self.config = base.config
        self.train = base.real_train.reset_index(drop=True)
        self.test = base.real_test.reset_index(drop=True)
        self.reference = ref_check.real_test.reset_index(drop=True)
        self.iterations = iterations or self.config.bootstrap_iterations
        self.seed = self.config.random_seed if seed is None else seed
        self.rng = np.random.default_rng(self.seed)
        cfg = self.config
        self.continuous = cfg.continuous
        self.categorical = cfg.categorical + cfg.binary
        self.target = cfg.target
        self.tree_fidelity = _TreeEncoder(self.continuous, self.categorical).fit(self.reference)
        self.onehot = FeatureEncoder(cfg.continuous_features, cfg.categorical_features).fit(self.train)
        self.tree_utility = _TreeEncoder(cfg.continuous_features, cfg.categorical_features).fit(self.train)
        self.y_test = self.test[self.target].astype(int).to_numpy()

        # Fidelity floor: the generator's own training data versus the reference.
        self.floor = self._fidelity_statistics(self.train)
        # Utility baselines: train on real training data and on the real reference.
        self.trtr = {p: self._predict(p, self.train) for p in self.PROBES}
        self.reference_pred = {p: self._predict(p, self.reference) for p in self.PROBES}
        self.trtr_auroc_ci = {
            p: percentile_interval(
                safe_auroc(self.y_test, self.trtr[p]),
                lambda rows, p=p: safe_auroc(self.y_test[rows], self.trtr[p][rows]),
                len(self.y_test), self.rng, self.iterations, 0.95)
            for p in self.PROBES
        }
        lr = "logistic_regression"
        self.reference_calibration = _calibration(self.y_test, self.trtr[lr])

    # ---------------------------------------------------------------- helpers
    def _predict(self, probe, train_frame):
        return _fit_predict(probe, train_frame, self.test, self.target,
                            self.onehot, self.tree_utility, self.seed)

    def _fidelity_statistics(self, candidate: pd.DataFrame) -> dict:
        coverage_rows = {c: _mass_coverage(self.reference[c], candidate[c]) for c in self.categorical}
        coverage = {c: float(v.mean()) for c, v in coverage_rows.items()}
        labels, scores = _two_sample_detection(self.reference, candidate, self.tree_fidelity, self.seed)
        auc = safe_auroc(labels, scores)
        auc_ci = percentile_interval(
            auc, lambda rows: safe_auroc(labels[rows], scores[rows]),
            len(labels), self.rng, self.iterations, 0.95)
        stats = _distance_block(self.reference, candidate, self.continuous, self.categorical)
        stats.update({
            "coverage": coverage, "coverage_rows": coverage_rows,
            "detection_auc": auc, "detection_auc_ci": auc_ci,
            "pmse": _pmse(labels, scores),
        })
        return stats

    # ---------------------------------------------------------------- fidelity
    def fidelity(self, synthetic: pd.DataFrame) -> dict:
        P = PROVISIONAL
        cand = self._fidelity_statistics(synthetic)
        floor = self.floor
        metrics: dict[str, dict] = {}
        weaknesses: list[str] = []

        # Mass-weighted coverage, judged as loss relative to the real floor.
        deficits = {}
        for c in self.categorical:
            rows_syn, rows_floor = cand["coverage_rows"][c], floor["coverage_rows"][c]
            point = float(rows_floor.mean() - rows_syn.mean())
            low, high = percentile_interval(
                point, lambda r, a=rows_floor, b=rows_syn: float(a[r].mean() - b[r].mean()),
                len(rows_syn), self.rng, self.iterations, 0.95)
            deficits[c] = (point, low, high)
        worst = max(deficits, key=lambda c: deficits[c][0])
        point, low, high = deficits[worst]
        coverage_fail = low is not None and low > P["coverage_margin"]
        metrics["mass_weighted_coverage_deficit"] = metric(
            point, ci_low=low, ci_high=high, interval_type="paired reference-row bootstrap (95%)",
            higher_is_worse=True, flag=coverage_fail,
            threshold=f"lower 95% bound > {P['coverage_margin']:.2f} -> Fail",
            note=(f"Worst column '{worst}': share of reference patients whose category is "
                  f"absent from the synthetic table, minus the same share for the real floor "
                  f"(synthetic {cand['coverage'][worst]:.4f}; floor {floor['coverage'][worst]:.4f})."))
        if coverage_fail:
            weaknesses.append(f"{worst}: synthetic omits categories held by "
                              f"{point:.1%} more reference patients than a real sample does")

        # Joint-distribution test (cannot Fail before null calibration).
        auc, (alow, ahigh) = cand["detection_auc"], cand["detection_auc_ci"]
        fhigh = floor["detection_auc_ci"][1]
        detected = (alow is not None and fhigh is not None
                    and alow > fhigh + P["detection_margin"])
        metrics["detection_auc"] = metric(
            auc, ci_low=alow, ci_high=ahigh,
            interval_type="bootstrap over cross-fitted predictions (95%)",
            higher_is_worse=True, flag=detected,
            threshold=(f"lower bound > floor upper bound + {P['detection_margin']:.2f} "
                       "-> Conditional (Fail requires null calibration)"),
            note=(f"Cross-fitted gradient-boosting classifier two-sample test against the "
                  f"held-out real reference; real-vs-real floor AUC {floor['detection_auc']:.3f}."))
        floor_pmse = floor["pmse"]
        metrics["pmse_ratio_to_real_floor"] = metric(
            cand["pmse"] / floor_pmse if floor_pmse > 0 else float("nan"), higher_is_worse=True,
            note="Propensity-score MSE divided by the real-vs-real pMSE (empirical null; 1 = like real).")

        # Low-order distances, now as excess over the floor (v0.3 thresholds).
        for name, key in [("mean_normalized_wasserstein", "wasserstein_excess"),
                          ("mean_jensen_shannon_divergence", "jsd_excess"),
                          ("association_matrix_error", "association_excess")]:
            excess = cand[name] - floor[name]
            metrics[f"{name}_excess"] = metric(
                excess, higher_is_worse=True, flag=bool(excess > P[key]),
                threshold=f"> {P[key]:.2f} -> Conditional",
                note=f"Synthetic {cand[name]:.4f} minus real floor {floor[name]:.4f}.")
        metrics["worst_column_normalized_wasserstein"] = metric(
            cand["worst_column_normalized_wasserstein"], higher_is_worse=True,
            note=(f"Worst continuous column '{cand['worst_column']}' (reported so a column "
                  "average cannot hide one broken variable); floor "
                  f"{floor['worst_column_normalized_wasserstein']:.4f}."))

        # Clinical constraints: unchanged v0.3 rule.
        rate, per_rule, _ = constraint_violations(synthetic, self.config.clinical_constraints)
        constraint_fail = rate is not None and rate > P["constraint_rate_fail"]
        metrics["clinical_constraint_violation_rate"] = metric(
            rate, higher_is_worse=True, flag=constraint_fail,
            status="Available" if rate is not None else "NotEvaluated",
            threshold=f"> {P['constraint_rate_fail']:.2f} -> Fail", note=str(per_rule))

        conditional = detected or any(
            metrics[f"{n}_excess"]["flag"] for n in
            ["mean_normalized_wasserstein", "mean_jensen_shannon_divergence", "association_matrix_error"])
        status = aggregate_status([
            "Fail" if (coverage_fail or constraint_fail) else "Pass",
            "NotEvaluated" if rate is None else "Pass",
            "Conditional" if conditional else "Pass",
        ])
        return {"dimension": "fidelity_reference_calibrated", "status": status,
                "metrics": metrics, "weaknesses": weaknesses,
                "provisional_margins": {k: v for k, v in P.items() if not k.startswith(("retention", "calibration"))},
                "per_column_coverage": cand["coverage"],
                "floor_per_column_coverage": floor["coverage"]}

    # ---------------------------------------------------------------- utility
    def utility(self, synthetic: pd.DataFrame) -> dict:
        P = PROVISIONAL
        y = self.y_test
        metrics: dict[str, dict] = {}
        weaknesses: list[str] = []
        outcome_levels = synthetic[self.target].astype(int).nunique()
        verdicts = {}
        for probe in self.PROBES:
            tstr = self._predict(probe, synthetic)
            base_low = self.trtr_auroc_ci[probe][0]
            for label, cand in [("synthetic", tstr), ("real_reference", self.reference_pred[probe])]:
                point = _retention(y, self.trtr[probe], cand)
                low, high = (None, None)
                if np.isfinite(point):
                    low, high = percentile_interval(
                        point, lambda r, c=cand: _retention(y[r], self.trtr[probe][r], c[r]),
                        len(y), self.rng, self.iterations, 0.95)
                verdict = _retention_verdict(point, low, high, base_low)
                if label == "synthetic":
                    verdicts[probe] = verdict
                    if probe == "logistic_regression":
                        oe, slope = _calibration(y, cand)
                metrics[f"{label}_retention_{probe}"] = metric(
                    point if np.isfinite(point) else None, ci_low=low, ci_high=high,
                    interval_type="paired held-out real-test bootstrap (95%)",
                    higher_is_worse=False, status=verdict if label == "synthetic" else "Available",
                    note=("Chance-corrected AUROC retention versus the same probe trained on "
                          "real training data." + (" Positive control: an independent real "
                          "sample; should be near 1." if label == "real_reference" else "")))
        primary = verdicts["random_forest"]
        disagree = sorted(p for p, v in verdicts.items() if v != primary)
        # Only a confident failure elsewhere (upper bound below the Fail line) downgrades a
        # primary Pass; a merely wider interval does not (it would flag real data, as the
        # half-size positive control showed for logistic regression).
        probe_conditional = primary == "Pass" and any(verdicts[p] == "Fail" for p in disagree)
        metrics["probe_agreement"] = metric(
            ", ".join(f"{p}={v}" for p, v in verdicts.items()), higher_is_worse=False,
            flag=bool(disagree),
            note="Primary verdict is the v0.3 random forest; a primary Pass becomes "
                 "Conditional only if another probe confidently Fails (probe-dependent utility).")

        ref_oe, ref_slope = self.reference_calibration
        rel_oe = oe / ref_oe if np.isfinite(oe) and ref_oe else float("nan")
        rel_slope = slope / ref_slope if np.isfinite(slope) and ref_slope else float("nan")
        band = P["calibration_ratio_band"]
        outside = lambda v: bool(np.isfinite(v) and not (1 / band <= v <= band))
        miscalibrated = outside(rel_oe) or outside(rel_slope)
        metrics["relative_calibration_in_the_large"] = metric(
            rel_oe if np.isfinite(rel_oe) else None, higher_is_worse=False, flag=outside(rel_oe),
            threshold=f"outside [{1/band:.2f}, {band:.2f}] -> Conditional",
            note=f"Observed/expected events of the synthetic-trained logistic probe on real test data, "
                 f"divided by the real-trained value ({ref_oe:.3f}).")
        metrics["relative_calibration_slope"] = metric(
            rel_slope if np.isfinite(rel_slope) else None, higher_is_worse=False,
            flag=outside(rel_slope),
            threshold=f"outside [{1/band:.2f}, {band:.2f}] -> Conditional",
            note=f"Calibration slope of the synthetic-trained logistic probe divided by the "
                 f"real-trained slope ({ref_slope:.3f}).")
        if outcome_levels < 2:
            weaknesses.append("Synthetic outcome collapsed to a single class; no model can be trained.")
        status = aggregate_status([
            "Fail" if outcome_levels < 2 else "Pass",
            primary,
            "Conditional" if (probe_conditional or miscalibrated) else "Pass",
        ])
        return {"dimension": "utility_reference_calibrated", "status": status,
                "metrics": metrics, "weaknesses": weaknesses,
                "probe_verdicts": verdicts,
                "provisional_margins": {k: v for k, v in P.items() if k.startswith(("retention", "calibration"))}}''')

_write_module('verification.py', r'''"""Small assertion-based instrument and engineering checks; no paper results."""
from __future__ import annotations

import ast
import copy
from dataclasses import replace
import hashlib
import importlib.util
from pathlib import Path
import sys
import types

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from .config import ExperimentConfig, DIMENSIONS, GATE_DIMENSIONS, DIAGNOSTIC_DIMENSIONS, seed_for
from .core import Gate, JobLedger, JobRunner, UnavailableAnalysis, write_json
from .reference import ReferenceContext, ReferenceFidelity, ReferenceUtility
from .evaluation import EquityV04, PrivacyV04, RobustnessV04
from .perturbations import strategy, changed_cells
from .calibration import CalibratedEvaluator, NullCalibrator, conformal_decision, rejection_rank
from .judges import MeanJudge, HarmonicJudge, GeometricJudge, SDMetricsJudge, matched_acceptance


def toy_tables(n=160) -> tuple:
    """Balanced, strongly predictive artificial population; role rows do not match."""
    cfg = ExperimentConfig(bootstrap_iterations=20)
    frames = []
    for split in range(4):
        rng = np.random.default_rng(seed_for('unit_toy', split))
        y = np.tile([0, 1], n//2)
        values = {c: 2 + rng.uniform(0, .01, n) for c in cfg.data.continuous}
        values['number_inpatient'] = 1 + 6*y + rng.uniform(0, .001, n)
        frame = pd.DataFrame(values)
        for c in cfg.data.categorical:
            frame[c] = np.tile(['A','A','B','B'], n//4)
        frame['race'] = np.tile(['AfricanAmerican','AfricanAmerican','Other','Other'], n//4)
        frame['gender'] = np.tile(['Female','Male','Male','Female'], n//4)
        frame[cfg.data.target] = y
        frames.append(frame.sample(frac=1, random_state=split).reset_index(drop=True))
    return cfg, frames


def assert_same(a, b):
    if isinstance(a, dict):
        assert a.keys() == b.keys()
        for k in a: assert_same(a[k], b[k])
    elif isinstance(a, (list, tuple)):
        assert len(a) == len(b)
        for x,y in zip(a,b): assert_same(x,y)
    elif isinstance(a, (np.ndarray, float, np.floating)):
        np.testing.assert_allclose(a,b,rtol=0,atol=1e-12,equal_nan=True)
    else: assert a == b


def run_checks(folder: Path) -> dict:
    cfg, frames = toy_tables()
    train, reference, test, positive = frames
    checks = []
    with threadpool_limits(limits=1):
        ctx = ReferenceContext(train, reference, test, cfg.benchmark())
        initial = copy.deepcopy(ctx)
        f = ReferenceFidelity(ctx).evaluate(positive).to_dict()
        utility_state = copy.deepcopy(ctx.rng.bit_generator.state)
        u = ReferenceUtility(ctx).evaluate(positive).to_dict()
        assert f['status'] == 'Pass', f['status']
        assert u['status'] == 'Pass', u['status']
        checks.append('independent artificial positive control passes reference fidelity and utility')
        collapse = positive.assign(**{cfg.data.target: 0})
        collapsed = ReferenceUtility(copy.deepcopy(initial)).evaluate(collapse).to_dict()
        assert collapsed['status'] == 'Fail'
        robust = RobustnessV04(cfg, initial, collapsed, utility_state, 7).evaluate(collapse)
        assert robust.status == 'NotEvaluated'
        checks.append('collapsed synthetic outcome fails utility; constant predictor robustness NotEvaluated')
        assert RobustnessV04.flips('Fail','Pass') and RobustnessV04.flips('Conditional','Fail')
        assert not RobustnessV04.flips('NotEvaluated','Pass')
        checks.append('robustness flips detected in both directions; unavailable is not a flip')
        small_group_cfg = replace(cfg, min_subgroup_n=10000)
        assert EquityV04(small_group_cfg, initial, 9).evaluate(positive).status == 'NotEvaluated'
        checks.append('no eligible subgroups yields equity NotEvaluated')
        # Nonzero real–real coincidences must be accepted for the new privacy null.
        coincident_test = pd.concat([train.iloc[:40], test.iloc[40:]], ignore_index=True)
        privacy = PrivacyV04(cfg, train, coincident_test, 10).evaluate(train.copy()).to_dict()
        assert privacy['metrics']['real_real_coincidence_rate']['estimate'] == .25
        assert privacy['metrics']['exact_match_binomial_pvalue']['estimate'] < .05
        assert privacy['status'] == 'Fail'
        checks.append('binomial privacy correction handles nonzero coincidence floor and detects copied training data')
        source = Path(__file__).with_name('original_reference.py.txt').read_text()
        if hashlib.sha256(source.encode()).hexdigest() != '3ebc21f477459ac08f9d8c6d0dfdc4ae72e40530b9d62e9311e177a65cf6b33f':
            raise RuntimeError('Supplied reference regression fixture changed.')
        module = types.ModuleType('syntrustbench._v04_original_test')
        module.__package__ = 'syntrustbench'; sys.modules[module.__name__] = module
        exec(compile(source, '<supplied_reference>', 'exec'), module.__dict__)
        old = module.ReferenceContext(train, reference, test, cfg.benchmark())
        assert_same(old.floor, initial.floor)
        assert_same(old.fidelity(positive), f)
        assert_same(old.utility(positive), u)
        assert_same(old.rng.bit_generator.state, ctx.rng.bit_generator.state)
        checks.append('original reference estimates, intervals, flags, notes and RNG state match (1e-12 absolute tolerance)')
    for family, doses in cfg.injections.doses:
        transform = strategy(family, cfg, train)
        before = set()
        for dose in (0.,)+doses:
            candidate, audit = transform.apply(positive, dose, seed_for('unit',family))
            changed = changed_cells(positive, candidate)
            assert before <= changed
            before = changed
            if dose == 0:
                assert positive.to_csv(index=False) == candidate.to_csv(index=False)
            if family == 'feature_shuffle':
                pd.testing.assert_series_equal(positive[cfg.injections.shuffle_feature].value_counts(dropna=False).sort_index(),
                                               candidate[cfg.injections.shuffle_feature].value_counts(dropna=False).sort_index())
        checks.append(f'{family}: dose-zero identity and nested changed cells')
    assert Gate.aggregate(['Pass','NotEvaluated','Fail']) == 'Fail'
    assert Gate.aggregate(['Pass','Conditional','NotEvaluated']) == 'NotEvaluated'
    evidence = {'dimension_scores': dict.fromkeys(GATE_DIMENSIONS, 1.)}
    evidence['dimension_scores']['privacy'] = None
    assert MeanJudge().score(evidence).score is None
    evidence['dimension_scores']['privacy'] = 0.
    assert HarmonicJudge().score(evidence).score == GeometricJudge().score(evidence).score == 0.
    checks.append('gate precedence and missing-value propagation in compensatory judges')
    calibrator = NullCalibrator(cfg.calibration)
    synthetic_nulls = {i: {'signals': {d: {'x': {'estimate': float(i), 'higher_is_worse': True}} for d in DIMENSIONS}}
                       for i in range(calibrator.counts(smoke=True)[0])}
    fit, audit = calibrator.assignment(smoke=True)
    m1 = calibrator.fit(synthetic_nulls, smoke=True)
    for i in audit:
        for d in DIMENSIONS: synthetic_nulls[i]['signals'][d]['x']['estimate'] = 1e9
    assert m1 == calibrator.fit(synthetic_nulls, smoke=True)
    checks.append('held-out audit values cannot affect fitted calibration thresholds')
    fixture = {'dimensions':{d:{'status':'Pass','metrics':{}} for d in DIMENSIONS},
               'signals':copy.deepcopy(synthetic_nulls[fit[0]]['signals']),
               'hard_fail':dict.fromkeys(DIMENSIONS,False)}
    fixture['hard_fail']['utility'] = True
    fixture['signals']['fidelity']['x']['estimate'] = None
    wrapped = CalibratedEvaluator(m1).apply(fixture,.05)
    assert wrapped['dimensions']['utility']['status'] == 'Fail'
    assert wrapped['dimensions']['fidelity']['status'] == 'NotEvaluated'
    checks.append('calibration preserves structural failures and never promotes missing evidence to Pass')
    fixture['hard_fail']['utility'] = False
    fixture['signals']['utility']['x']['estimate'] = None
    fixture['signals']['robustness']['x']['estimate'] = 1e9
    assert CalibratedEvaluator(m1).apply(fixture,.05)['dimensions']['robustness']['status'] == 'NotEvaluated'
    fixture['signals']['utility']['x']['estimate'] = 0.
    fixture['dimensions']['robustness']['constant_predictor'] = True
    assert CalibratedEvaluator(m1).apply(fixture,.05)['dimensions']['robustness']['status'] == 'NotEvaluated'
    checks.append('calibrated robustness stays NotEvaluated for unavailable utility or constant predictors')
    checks.extend(gate_scope_checks(cfg))
    checks.append(conformal_size_check())
    checks.append(matched_judge_size_check(cfg))
    ledger = JobLedger(folder / 'ledger_test', 'fixture')
    runner = JobRunner(ledger)
    def artifact(where):
        path = write_json(where/'artifact.json', {'answer':1})
        return {'path':path}
    first = runner.run('test/artifact', artifact)
    assert runner.run('test/artifact', artifact) == first
    Path(first['path']).write_text('tampered')
    caught = False
    try: ledger.result('test/artifact')
    except RuntimeError: caught = True
    assert caught
    checks.append('resumption verifies artifact hashes and rejects tampering')
    def unavailable(where): raise UnavailableAnalysis('declared fixture unavailability')
    assert runner.run('test/unavailable', unavailable) is None
    assert ledger.state('test/unavailable')['state'] == 'unavailable'
    saved_unavailable = ledger.state('test/unavailable')
    assert runner.run('test/unavailable', unavailable) is None
    assert ledger.state('test/unavailable') == saved_unavailable
    def failure(where): raise ArithmeticError('deliberate failure-ledger fixture')
    caught = False
    try: runner.run('test/failure', failure)
    except ArithmeticError: caught = True
    assert caught and ledger.state('test/failure')['state'] == 'failed'
    assert 'ArithmeticError' in (folder/'ledger_test/failures.jsonl').read_text()
    checks.append('declared unavailability is explicit; unexpected failures are logged and re-raised')
    from .dp_generators import DPAIMGenerator, DPMSTGenerator, PublicDiscretizer
    from .tabddpm import TabDDPMEnvironment
    dp_cfg = replace(cfg, execution=replace(cfg.execution, smoke_mode=True))
    public = PublicDiscretizer(dp_cfg)
    before = public.manifest()
    extremes = train.copy()
    for col in dp_cfg.data.continuous:
        extremes[col] = np.linspace(-1e8,1e9,len(extremes))
    extremes['race'] = 'never_observed_publicly'
    public.encode(train)
    public.encode(extremes)
    assert before == public.manifest() == PublicDiscretizer(dp_cfg).manifest()
    assert public.transformer().cardinality == public.cards
    checks.append('DP bins and full categorical domains remain identical when private extremes/levels change')
    dp = DPAIMGenerator(dp_cfg,17).fit(train.iloc[:64])
    sample = dp.sample(64,1234)
    assert list(sample.columns) == list(train.columns) and len(sample) == 64
    assert all(pd.api.types.is_integer_dtype(sample[c]) and sample[c].ge(0).all() for c in dp_cfg.data.continuous)
    assert sample['time_in_hospital'].between(1,14).all()
    assert sample[dp_cfg.data.target].isin([0,1]).all()
    pd.testing.assert_frame_equal(sample,dp.sample(64,1234))
    assert dp.provenance['formal_privacy']['delta'] == 1/(10*64)
    assert dp.provenance['public_preprocessing']['preprocessing_epsilon'] == 0
    checks.append('DPAIM actual fit/sample: schema, row count, integer constraints, default delta and deterministic same-fit sampling')
    dp.save(folder/'dp_test_model.pkl')
    import cloudpickle
    with (folder/'dp_test_model.pkl').open('rb') as handle:
        dp.model = cloudpickle.load(handle)['model']
    pd.testing.assert_frame_equal(sample,dp.sample(64,1234))
    checks.append('DPAIM model persistence preserves deterministic sampling without replaying private fit noise')
    if cfg.generators.enable_dpmst:
        mst = DPMSTGenerator(dp_cfg,17).fit(train.iloc[:64])
        out = mst.sample(64,1234)
        assert len(out) == 64 and list(out.columns) == list(train.columns)
        pd.testing.assert_frame_equal(out,mst.sample(64,1234))
        checks.append('Optional DPMST actual fit/sample preserves schema and same-fit sampling seeds')
    absent_cfg = replace(cfg, generators=replace(cfg.generators,tabddpm_python='__missing_python310_test__'),
                         execution=replace(cfg.execution,output_root=str(folder/'optional_environment_test')))
    caught = False
    try:
        TabDDPMEnvironment(absent_cfg).ensure()
    except UnavailableAnalysis as exc:
        caught = 'FileNotFoundError' in str(exc)
    assert caught
    checks.append('Missing optional TabDDPM Python environment yields explicit Unavailable')
    return {'passed':len(checks), 'checks':checks, 'scope':'artificial test data only'}


def honest_signals(rng, n: int) -> np.ndarray:
    """Oriented honest signals: one spiky zero-inflated column, like coverage deficit."""
    spiky = np.where(rng.uniform(size=n) < .7, 0., rng.lognormal(-7, 1.2, n))
    return np.column_stack([spiky, rng.normal(size=n), rng.normal(size=n), rng.standard_t(3, n)])


def conformal_size_check() -> str:
    """Marginal honest-data rejection of the full-conformal rule stays at r/(n+1)."""
    rng = np.random.default_rng(seed_for('conformal_size_check'))
    rejections, trials = 0, 0
    for _ in range(40):
        fit = honest_signals(rng, 50)
        for row in honest_signals(rng, 100):
            decision = conformal_decision(fit, row, .05, 1e-8)
            rejections += decision['threshold'] is not None and decision['candidate_max'] > decision['threshold']
            trials += 1
    rate, nominal = rejections/trials, rejection_rank(50, .05)/51
    assert abs(rate - nominal) < .015, (rate, nominal)
    return f'full-conformal honest rejection {rate:.3f} vs nominal {nominal:.3f} with a spiky zero-inflated signal'


def matched_judge_size_check(cfg) -> str:
    """Compensatory judges calibrated by the matched rule reject honest tables at r/(n+1)."""
    rng = np.random.default_rng(seed_for('matched_judge_size_check'))
    calibrator = NullCalibrator(cfg.calibration)
    count, _ = calibrator.counts()
    def draw():
        return {'signals': {d: {f'{d}_{k}': {'estimate': float(rng.standard_t(4)), 'higher_is_worse': True} for k in range(2)}
                            for d in DIMENSIONS},
                'dimensions': {d: {'status': 'Pass', 'metrics': {}} for d in DIMENSIONS}}
    rates = {}
    for judge in (MeanJudge(), GeometricJudge(), HarmonicJudge(), SDMetricsJudge()):
        rejected = trials = 0
        for _ in range(12):
            nulls = {i: draw() for i in range(count)}
            sdm = {i: float(rng.uniform()) for i in range(count)}
            model = calibrator.fit(nulls, sdmetrics_scores=sdm)
            for _ in range(60):
                accepted, _ = matched_acceptance(judge, draw(), {'score': float(rng.uniform())}, model, cfg, .05)
                assert accepted is not None
                rejected += not accepted; trials += 1
        rates[type(judge).__name__] = rejected/trials
    nominal = rejection_rank(cfg.calibration.fit_runs, .05)/(cfg.calibration.fit_runs+1)
    assert all(abs(v - nominal) < .03 for v in rates.values()), (rates, nominal)
    return 'matched compensatory judges reject honest tables at the nominal rate: ' + ', '.join(f'{k} {v:.3f}' for k, v in rates.items())


def gate_scope_checks(cfg) -> list[str]:
    """The gate decides on the gating dimensions only; diagnostics never change it."""
    rng = np.random.default_rng(seed_for('gate_scope_checks'))
    calibrator = NullCalibrator(cfg.calibration)
    count, _ = calibrator.counts()
    nulls = {i: {'signals': {d: {f'{d}_signal': {'estimate': float(rng.normal()), 'higher_is_worse': True}}
                             for d in DIMENSIONS}} for i in range(count)}
    model = calibrator.fit(nulls)
    def table(**overrides):
        signals = {d: {f'{d}_signal': {'estimate': overrides.get(d, 0.), 'higher_is_worse': True}} for d in DIMENSIONS}
        return {'dimensions': {d: {'status': 'Pass', 'metrics': {}} for d in DIMENSIONS},
                'signals': signals, 'hard_fail': dict.fromkeys(DIMENSIONS, False)}
    evaluator = CalibratedEvaluator(model)
    honest = evaluator.apply(table(), .05)
    assert honest['gate'] == 'Pass' and all(honest['dimensions'][d]['status'] == 'Pass' for d in DIMENSIONS)
    for d in DIAGNOSTIC_DIMENSIONS:
        out = evaluator.apply(table(**{d: 1e6}), .05)
        assert out['dimensions'][d]['status'] == 'Fail' and out['gate'] == 'Pass', d
    for d in GATE_DIMENSIONS:
        out = evaluator.apply(table(**{d: 1e6}), .05)
        assert out['dimensions'][d]['status'] == 'Fail' and out['gate'] == 'Fail', d
        others = [x for x in GATE_DIMENSIONS if x != d]
        assert all(out['dimensions'][x]['status'] == 'Pass' for x in others), d
    missing = table(); missing['signals']['privacy']['privacy_signal']['estimate'] = None
    assert evaluator.apply(missing, .05)['gate'] == 'NotEvaluated'
    assert model['levels']['0.05']['rejection_rank'] == 2
    return ['diagnostic failures never change the gate; any gating failure fails it alone',
            'a missing gating signal makes the gate NotEvaluated, never Pass']


def self_test() -> None:
    assert Gate.aggregate(['Pass','Conditional']) == 'Conditional'
    print(conformal_size_check())
''')

from syntrustbench_campaign import verification

verification.self_test()


### Runtime provenance and environment lock

In [ ]:
_write_module('environment.py', r'''"""Pinned numerical environment and frozen official checkout verification."""
from __future__ import annotations

from dataclasses import asdict
import importlib
import importlib.metadata
import os
from pathlib import Path
import platform
import subprocess
import sys

from .config import COMMIT, ExperimentConfig
from .core import file_hash, read_json, utc, write_json

PINS = (('numpy', '2.3.4'), ('pandas', '2.3.3'), ('scipy', '1.16.3'),
        ('scikit-learn', '1.7.2'), ('torch', '2.7.1'), ('sdv', '1.37.4'),
        ('ctgan', '0.13.0'), ('rdt', '2.1.0'), ('copulas', '0.15.0'),
        ('sdmetrics', '0.33.0'), ('pyyaml', '6.0.2'), ('matplotlib', '3.10.3'),
        ('smartnoise-synth', '1.0.8'), ('mbi', '1.1.0'), ('jax', '0.6.2'),
        ('jaxlib', '0.6.2'), ('optax', '0.2.5'), ('chex', '0.1.90'),
        ('opendp', '0.14.2'), ('smartnoise-sql', '1.0.10'),
        ('networkx', '3.4.2'), ('disjoint-set', '0.9.0'), ('cloudpickle', '3.1.2'))

def _withheld() -> str:
    # The public repository of the frozen v0.3 release is withheld during double-blind review.
    # Set FROZEN_V030_REPOSITORY to its URL to rerun the full campaign; the analysis of saved
    # results uses third_party/syntrustbench_v030 and needs no network access.
    raise RuntimeError('Set FROZEN_V030_REPOSITORY to the public URL of the frozen v0.3 release.')


class Environment:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg

    def setup(self) -> dict:
        if sys.version_info < (3, 11):
            raise RuntimeError('Python >=3.11 required.')
        checkout = Path(self.cfg.execution.checkout_dir).resolve()
        if self.cfg.execution.install_dependencies:
            subprocess.run([sys.executable, '-m', 'pip', 'install', *[f'{k}=={v}' for k, v in PINS]], check=True)
            if not checkout.exists():
                checkout.parent.mkdir(parents=True, exist_ok=True)
                subprocess.run(['git', 'clone', os.environ.get('FROZEN_V030_REPOSITORY') or _withheld(), str(checkout)], check=True)
            subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', COMMIT], check=True)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', str(checkout)], check=True)
            importlib.invalidate_caches()
        if not (checkout / '.git').exists():
            raise RuntimeError('Frozen checkout missing. Enable install_dependencies in the configuration cell.')
        head = subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if head != COMMIT:
            raise RuntimeError('Frozen checkout commit mismatch.')
        if subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain', '--untracked-files=no'], text=True).strip():
            raise RuntimeError('Tracked files changed in the frozen checkout.')
        import syntrustbench
        installed = Path(syntrustbench.__file__).resolve().parent
        hashes = {}
        for path in sorted((checkout / 'src/syntrustbench').rglob('*.py')):
            relative = path.relative_to(checkout / 'src/syntrustbench')
            target = installed / relative
            if not target.exists() or file_hash(target) != file_hash(path):
                raise RuntimeError(f'Installed frozen source differs: {relative}; restart kernel after install.')
            hashes[str(relative)] = file_hash(path)
        versions = {name: importlib.metadata.version(name) for name, _ in PINS}
        if {k:v.split('+',1)[0] for k,v in versions.items()} != dict(PINS):
            raise RuntimeError(f'Numerical package pins differ: {versions}')
        root = self.cfg.root
        root.mkdir(parents=True, exist_ok=True)
        lock = root / 'numerical_environment.json'
        if lock.exists() and read_json(lock) != versions:
            raise RuntimeError('Numerical environment changed; refusing campaign resume.')
        write_json(lock, versions)
        source = {p.name: file_hash(p) for p in sorted(Path(__file__).parent.iterdir())
                  if p.is_file() and p.suffix in ('.py','.txt')}
        code_lock = root / 'implementation_hashes.json'
        if code_lock.exists() and read_json(code_lock) != source:
            raise RuntimeError('Campaign implementation changed; use a new output_root.')
        write_json(code_lock, source)
        import torch
        manifest = {'time': utc(), 'platform': platform.platform(), 'python': platform.python_version(),
                    'cpu_count': os.cpu_count(), 'cuda_available': torch.cuda.is_available(),
                    'device': 'CUDA' if self.cfg.generators.allow_cuda and torch.cuda.is_available() else 'CPU',
                    'packages': versions, 'git_commit': head, 'installed_source_hashes': hashes,
                    'scientific_hash': self.cfg.fingerprint, 'configuration': asdict(self.cfg),
                    'implementation_hashes': source}
        write_json(root / 'sessions' / (utc().replace(':', '-') + '.json'), manifest)
        write_json(root / 'configuration.json', asdict(self.cfg))
        return manifest


def self_test() -> None:
    assert dict(PINS)['sdv'] == '1.37.4' and len(dict(PINS)) == len(PINS)
''')

from syntrustbench_campaign import environment

environment.self_test()


### Job orchestration, parallel profiles and completion audit

In [ ]:
_write_module('campaign.py', r'''"""Resumable orchestration of every specified experiment stage."""
from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict, replace
import hashlib
import multiprocessing
import os
from pathlib import Path
import shutil
import time
import traceback

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from .config import ExperimentConfig, STAGES, DIMENSIONS, seed_for
from .core import JobLedger, JobRunner, UnavailableAnalysis, file_hash, read_json, save_frame, utc, write_json
from .data import CohortBuilder, RoleAssigner, read_analytic, split_three, stratified_indices
from .evaluation import OfficialProfiler, build_context, evaluate_v04
from .generators import generator, seed_all
from .perturbations import strategy
from .calibration import NullCalibrator, CalibratedEvaluator
from .judges import quality_report
from .reporting import Analysis, Reporter, collect_profiles, judge_artifact


def worker(cfg: ExperimentConfig, key: str, kind: str, args: dict,
           required: tuple[str, ...], watched: tuple[str, ...]) -> dict | None:
    """Importable spawn-safe worker. Writes only its job attempt/state plus append-only events."""
    ledger = JobLedger(cfg.root, cfg.fingerprint)
    runner = JobRunner(ledger, cfg.execution.retry_failed)

    def operation(folder):
        for dep in required: ledger.result(dep)
        os.environ['LOKY_MAX_CPU_COUNT'] = str(cfg.execution.threads_per_worker)
        seed_all(seed_for('job', key))
        import torch
        torch.set_num_threads(cfg.execution.threads_per_worker)
        with threadpool_limits(limits=cfg.execution.threads_per_worker):
            return Operations(cfg, ledger).execute(kind, args, folder)

    inputs = [ledger.state_path(k) for k in dict.fromkeys(required + watched)]
    inputs += [Path(p) for p in args.get('input_files', [])]
    return runner.run(key, operation, inputs)


class Operations:
    def __init__(self, cfg: ExperimentConfig, ledger: JobLedger):
        self.cfg, self.ledger = cfg, ledger

    def execute(self, kind: str, args: dict, folder: Path) -> dict:
        methods = {'cohort': self.cohort, 'roles': self.roles, 'null_roles': self.null_roles,
                   'context': self.context, 'generation': self.generation, 'control': self.control,
                   'injection': self.injection, 'profile': self.profile, 'alias': self.alias,
                   'calibrate': self.calibrate, 'judges': self.judges, 'analysis': self.analysis,
                   'report': self.report, 'smoke': self.smoke}
        return methods[kind](args, folder)

    def cohort(self, args, folder):
        return CohortBuilder(self.cfg).build(folder)

    def roles(self, args, folder):
        return RoleAssigner(self.cfg).main(self.ledger.result('prepare/cohort'), folder)

    def null_roles(self, args, folder):
        return RoleAssigner(self.cfg).null(self.ledger.result('prepare/cohort'), args['repeat'], folder)

    def context(self, args, folder):
        return build_context(self.cfg, self.ledger.result(args['roles_key']), folder)

    def generation(self, args, folder):
        roles = self.ledger.result('prepare/roles')
        train = read_analytic(roles['paths']['train'], self.cfg)
        model = generator(args['generator'], self.cfg, args['seed'])
        model.artifact_dir = folder
        full = model.fit(train).sample(len(train), seed_for(args['generator'], args['seed'], 'sampling'))
        full_path = save_frame(full, folder / 'synthetic_full.csv')
        # Use the same sampled candidate for all judges and official/v0.4 evaluations.
        rows = np.random.default_rng(seed_for(args['generator'], args['seed'], 'size_match')).choice(len(full), roles['n'], replace=False)
        candidate = save_frame(full.iloc[rows].reset_index(drop=True), folder / 'synthetic_matched.csv')
        read_analytic(candidate, self.cfg)
        model.save(folder / 'generator.pkl')
        write_json(folder / 'generator_provenance.json', model.provenance)
        return {'candidate_path': candidate, 'full_path': full_path, 'rows': len(full), 'matched_rows': roles['n'],
                'matched_positions': rows.tolist(), 'generator': args['generator'], 'seed': args['seed'],
                'train_sha256': file_hash(roles['paths']['train']), 'timing': model.timing,
                'generator_provenance': model.provenance}

    def control(self, args, folder):
        roles = self.ledger.result('prepare/roles')
        train = read_analytic(roles['paths']['train'], self.cfg)
        base = read_analytic(roles['paths']['positive' if args['control'] == 'positive' else 'floor'], self.cfg)
        seed = seed_for('control', args['control'], args['seed'])
        if args['control'] == 'positive':
            candidate = base.sample(n=roles['n'], random_state=seed).reset_index(drop=True)
            audit = {'seed': seed, 'reuses_fixed_P_pool': True, 'rows': len(candidate)}
        else:
            candidate, audit = strategy(args['control'], self.cfg, train).apply(base, 1., seed)
        return {'candidate_path': save_frame(candidate, folder / 'candidate.csv'), 'audit': audit}

    def injection(self, args, folder):
        baseline = self.ledger.result(args['baseline_data'])
        roles = self.ledger.result('prepare/roles')
        base = read_analytic(baseline['candidate_path'], self.cfg)
        train = read_analytic(roles['paths']['train'], self.cfg)
        seed = seed_for('injection', args['baseline'], args['family'])
        candidate, audit = strategy(args['family'], self.cfg, train).apply(base, args['dose'], seed)
        path = folder / 'candidate.csv'
        if args['dose'] == 0:
            shutil.copyfile(baseline['candidate_path'], path)
            if file_hash(path) != file_hash(baseline['candidate_path']):
                raise RuntimeError('Dose-zero bytes changed.')
        else:
            save_frame(candidate, path)
        previous = args.get('previous_data')
        if previous:
            prior = self.ledger.result(previous)['audit']
            before, after = {tuple(x) for x in prior['changed_cells']}, {tuple(x) for x in audit['changed_cells']}
            if not before <= after:
                raise RuntimeError('Injection changes are not nested.')
        write_json(folder / 'injection_audit.json', audit)
        return {'candidate_path': str(path), 'audit': audit, 'baseline': args['baseline'],
                'family': args['family'], 'dose': args['dose'], 'seed': seed,
                'known_bad': args['dose'] > 0 and audit['changed_cell_count'] > 0}

    def profile(self, args, folder):
        roles = self.ledger.result(args['roles_key'])
        context = self.ledger.result(args['context_key'])
        labels = dict(args['labels'])
        if args.get('data_key'):
            data = self.ledger.result(args['data_key'])
            candidate_path = data['candidate_path']
            if labels['kind'] == 'injection': labels['known_bad'] = data['known_bad']
        else:
            candidate_path = roles['paths']['positive']
        candidate = read_analytic(candidate_path, self.cfg)
        if len(candidate) != roles['n']:
            raise ValueError('Candidate is not size matched to reference.')
        started = time.perf_counter()
        official = OfficialProfiler(self.cfg).evaluate(roles, candidate, folder / 'official_v03')
        official_seconds = time.perf_counter()-started
        started = time.perf_counter()
        v04 = evaluate_v04(self.cfg, context['context_path'], roles, candidate, labels['table'])
        v04_seconds = time.perf_counter()-started
        reference = read_analytic(roles['paths']['reference'], self.cfg)
        external = quality_report(reference, candidate, self.cfg)
        payload = {'labels': labels, 'official': official, 'v04': v04, 'sdmetrics': external,
                   'official_folder': str(folder / 'official_v03'), 'availability': 'Available',
                   'input_hashes': {k: file_hash(p) for k, p in roles['paths'].items()} | {'candidate': file_hash(candidate_path)},
                   'rows': {'candidate': len(candidate), **roles['role_counts']},
                   'timing': {'official_seconds': official_seconds, 'v04_seconds': v04_seconds}}
        if args.get('data_key') and 'generator_provenance' in data:
            payload['generator_provenance'] = data['generator_provenance']
        if args.get('calibration_key'):
            calibration = self.ledger.result(args['calibration_key'])
            model = read_json(calibration['model_path'])
            if file_hash(calibration['model_path']) != calibration['sha256']:
                raise RuntimeError('Frozen calibration changed.')
            payload['calibration_sha256'] = calibration['sha256']
            payload['calibrated'] = {str(a): CalibratedEvaluator(model).apply(v04, a) for a in self.cfg.calibration.target_rates}
        return {'profile_path': write_json(folder / 'profile.json', payload)}

    def alias(self, args, folder):
        data = self.ledger.result(args['data_key'])
        original = self.ledger.result(args['baseline_profile'])
        profile = read_json(original['profile_path'])
        profile['labels'] = args['labels']
        profile['dose_zero_reuses_profile'] = {'path': original['profile_path'], 'sha256': file_hash(original['profile_path'])}
        return {'profile_path': write_json(folder / 'profile.json', profile)}

    def calibrate(self, args, folder):
        calibrator = NullCalibrator(self.cfg.calibration)
        fit, audit = calibrator.assignment(self.cfg.execution.smoke_mode)
        full = {r: read_json(self.ledger.result(f'null/profile/{r:02d}')['profile_path']) for r in fit}
        # Fit-run native SDMetrics scores calibrate the SDMetrics judge to the same honest-data rate.
        model = calibrator.fit({r: p['v04'] for r, p in full.items()}, self.cfg.execution.smoke_mode,
                               sdmetrics_scores={r: p['sdmetrics'].get('score') for r, p in full.items()})
        path = write_json(folder / 'empirical_maxT.json', model)
        save_frame(pd.DataFrame(model['raw_cutoffs']), folder / 'raw_cutoffs.csv')
        save_frame(pd.DataFrame(model['excluded']), folder / 'unavailable_calibration_metrics.csv')
        record = {'model_path': path, 'sha256': file_hash(path), 'frozen_at': utc(),
                  'fit_ids': fit, 'audit_ids': audit, 'blocked_dimensions': model['blocked_dimensions']}
        write_json(folder / 'freeze_record.json', record)
        return record

    def judges(self, args, folder):
        # Planned tables plus held-out honest audit runs: every judge is scored on both.
        profiles = collect_profiles(self.ledger, args['descriptors'] + args.get('audit_descriptors', []))
        model = read_json(self.ledger.result('calibrate/model')['model_path'])
        return judge_artifact(self.cfg, profiles, model, folder)

    def analysis(self, args, folder):
        profiles = collect_profiles(self.ledger, args['descriptors'])
        nulls = collect_profiles(self.ledger, args['null_descriptors'])
        model = read_json(self.ledger.result('calibrate/model')['model_path'])
        judges = read_json(self.ledger.result('judges/all')['judge_path'])
        return Analysis(self.cfg, profiles, nulls, model, judges).build(self.ledger.result('prepare/cohort'), folder,
                                                                      self.ledger.result('prepare/roles'))

    def report(self, args, folder):
        return Reporter(self.cfg).build(self.ledger.result('analysis/tables'), folder)

    def smoke(self, args, folder):
        from .verification import run_checks
        checks = run_checks(folder)
        if self.cfg.execution.smoke_mode:
            return {'checks': checks, 'scope': 'Small-campaign stage; remaining stages exercise all adapters.'}
        # Build a separate smoke campaign entirely from T+C. Never read main E.
        cohort = self.ledger.result('prepare/cohort')
        role = RoleAssigner(self.cfg).null(cohort, 'smoke_preflight', folder / 'development_roles')
        frames = {'real_train': read_analytic(role['paths']['train'], self.cfg),
                  'calibration_pool': pd.concat([read_analytic(role['paths'][k], self.cfg) for k in ('reference','positive')], ignore_index=True),
                  'real_test': read_analytic(role['paths']['test'], self.cfg)}
        audit = pd.read_csv(role['patient_audit'])
        ids = {'real_train': audit[audit.role == 'train'].patient_id_sha256.tolist(),
               'calibration_pool': sum([audit[audit.role == k].patient_id_sha256.tolist() for k in ('reference','positive')], []),
               'real_test': audit[audit.role == 'test'].patient_id_sha256.tolist()}
        small = folder / 'frozen_smoke'; small.mkdir()
        paths, patient_rows = {}, []
        for name, frame in frames.items():
            positions = stratified_indices(frame[self.cfg.data.target], min(len(frame), self.cfg.execution.smoke_rows), seed_for('smoke_cap',name))
            paths[name] = save_frame(frame.iloc[positions], small / f'{name}.csv')
            patient_rows.extend({'role': name, 'analytic_row': i, 'patient_id_sha256': ids[name][p]} for i,p in enumerate(positions))
        patient_path = save_frame(pd.DataFrame(patient_rows), small / 'patient_role_audit.csv')
        manifest_path = write_json(small / 'split_manifest.json', {'split_hashes': {n: file_hash(p) for n,p in paths.items()},
            'patient_audit': patient_path, 'patient_audit_sha256': file_hash(patient_path)})
        mini = replace(self.cfg, data=replace(self.cfg.data, frozen_dir=str(small), frozen_manifest=manifest_path),
                       execution=replace(self.cfg.execution, smoke_mode=True, dry_run=False, workers=1,
                                         stages=STAGES, output_root=str(folder / 'smoke_campaign')))
        from .environment import Environment
        mini = replace(mini, execution=replace(mini.execution, install_dependencies=False))
        Environment(mini).setup()
        campaign = Campaign(mini)
        for stage in STAGES: campaign.run_stage(stage)
        audit = campaign.completion_audit()
        optional_unavailable = [j['key'] for jobs in campaign.plan.values() for j in jobs
                                if j['key'].startswith('clean/TabDDPM/')
                                and campaign.ledger.state(j['key'])['state'] == 'unavailable']
        if audit['failed'] or audit['unavailable'] > len(optional_unavailable) or audit['planned']:
            raise UnavailableAnalysis('Smoke campaign incomplete; inspect its completion audit before full work.')
        return {'checks': checks, 'smoke_campaign': str(mini.root), 'optional_unavailable': optional_unavailable,
                'scope': 'All stage paths on development-only data; not paper evidence.'}


class Campaign:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.ledger = JobLedger(cfg.root, cfg.fingerprint)
        self.descriptors, self.null_descriptors, self.plan = self._plan()
        expected = {stage: [j['key'] for j in jobs] for stage, jobs in self.plan.items()}
        path = cfg.root / 'expected_jobs.json'
        if path.exists() and read_json(path) != expected:
            raise RuntimeError('Expected job plan changed; choose a new campaign directory.')
        write_json(path, expected)
        for jobs in self.plan.values():
            for j in jobs:
                if not self.ledger.state_path(j['key']).exists(): self.ledger.set(j['key'], 'planned')

    def _plan(self) -> tuple[list, list, dict]:
        cfg = self.cfg
        plan = {s: [] for s in STAGES}
        descriptors, null_descriptors = [], []

        def add(stage, key, kind, args=None, required=(), watched=()):
            plan[stage].append({'key': key, 'kind': kind, 'args': args or {},
                                'required': tuple(required), 'watched': tuple(watched)})

        def profile(stage, key, labels, data_key=None, roles='prepare/roles', context='clean/context', calibrated=True):
            required = [roles, context] + ([data_key] if data_key else []) + (['calibrate/model'] if calibrated else [])
            add(stage, key, 'profile', {'labels': labels, 'data_key': data_key, 'roles_key': roles,
                                      'context_key': context, 'calibration_key': 'calibrate/model' if calibrated else None}, required)
            descriptor = {'profile_key': key, 'labels': labels}
            (descriptors if calibrated else null_descriptors).append(descriptor)

        if cfg.data.frozen_dir:
            inputs = [cfg.data.frozen_manifest] + [str(Path(cfg.data.frozen_dir)/f'{n}.csv') for n in ('real_train','calibration_pool','real_test')]
            manifest = read_json(cfg.data.frozen_manifest) if Path(cfg.data.frozen_manifest).exists() else {}
            audit = manifest.get('patient_audit', str(Path(cfg.data.frozen_dir).resolve() / 'patient_role_audit.csv'))
            if not Path(audit).is_absolute(): audit = str(Path(cfg.data.frozen_manifest).resolve().parent / audit)
            inputs.append(audit)
        else: inputs = [cfg.data.raw_csv]
        add('prepare', 'prepare/cohort', 'cohort', {'input_files': inputs})
        add('prepare', 'prepare/roles', 'roles', required=['prepare/cohort'])
        add('smoke', 'smoke/all_paths', 'smoke', required=['prepare/cohort'])
        fit, audit = NullCalibrator(cfg.calibration).assignment(cfg.execution.smoke_mode)
        for r in sorted(fit+audit):
            role, context = f'null/data/{r:02d}', f'null/context/{r:02d}'
            add('null', role, 'null_roles', {'repeat': r}, ['prepare/cohort','smoke/all_paths'])
            add('null', context, 'context', {'roles_key': role}, [role])
            labels = {'table': f'null/{r:02d}', 'kind': 'null', 'repeat': r,
                      'partition': 'fit' if r in fit else 'audit', 'truth': 'good', 'known_bad': False}
            profile('null', f'null/profile/{r:02d}', labels, roles=role, context=context, calibrated=False)
        add('calibrate', 'calibrate/model', 'calibrate', required=[f'null/profile/{r:02d}' for r in fit])
        add('clean', 'clean/context', 'context', {'roles_key':'prepare/roles'}, ['prepare/roles','calibrate/model'])
        seeds = cfg.generators.seeds[:1] if cfg.execution.smoke_mode else cfg.generators.seeds
        reps = cfg.positive_replicates[:1] if cfg.execution.smoke_mode else cfg.positive_replicates
        baseline_data, baseline_profiles = {}, {}
        for g in cfg.generators.enabled_names:
            for s in seeds:
                key = f'clean/{g}/{s}'; data_key = key+'/data'
                labels = {'table': key, 'kind': 'clean', 'generator': g, 'seed': s, 'truth':'unknown', 'known_bad':False}
                add('clean', data_key, 'generation', {'generator':g,'seed':s}, ['prepare/roles','calibrate/model'])
                profile('clean', key+'/profile', labels, data_key)
                if g == 'GaussianCopula': baseline_data[key], baseline_profiles[key] = data_key, key+'/profile'
        for control in ('positive','copy','negative'):
            for s in reps:
                key = f'control/{control}/{s}'; data_key = key+'/data'
                labels = {'table':key, 'kind':'control', 'control':control, 'seed':s,
                          'truth':'good' if control == 'positive' else 'bad', 'known_bad':control != 'positive'}
                add('controls', data_key, 'control', {'control':control, 'seed':s}, ['prepare/roles','calibrate/model'])
                profile('controls', key+'/profile', labels, data_key)
                if control == 'positive': baseline_data[key], baseline_profiles[key] = data_key, key+'/profile'
        for baseline, data_key in baseline_data.items():
            for family, all_doses in cfg.injections.doses:
                doses = all_doses[-1:] if cfg.execution.smoke_mode else all_doses
                previous = None
                for dose in (0.,)+doses:
                    key = f'injection/{baseline}/{family}/{dose:g}'
                    labels = {'table': key, 'kind':'injection', 'baseline':baseline,
                              'baseline_kind':'positive' if baseline.startswith('control') else 'GaussianCopula',
                              'family':family, 'dose':dose, 'truth':'bad' if dose else 'baseline', 'known_bad':dose > 0}
                    required = [data_key,'prepare/roles','calibrate/model'] + ([previous] if previous else [])
                    add('injections', key+'/data', 'injection', {'baseline_data':data_key, 'baseline':baseline,
                        'family':family, 'dose':dose, 'previous_data':previous}, required)
                    if dose == 0:
                        add('injections', key+'/profile', 'alias', {'labels':labels,'data_key':key+'/data',
                            'baseline_profile':baseline_profiles[baseline]}, [key+'/data',baseline_profiles[baseline]])
                        descriptors.append({'profile_key':key+'/profile','labels':labels})
                    else: profile('injections', key+'/profile', labels, key+'/data')
                    previous = key+'/data'
        watched = [d['profile_key'] for d in descriptors]
        audit_descriptors = [d for d in null_descriptors if d['labels']['partition'] == 'audit']
        add('judges', 'judges/all', 'judges', {'descriptors':descriptors, 'audit_descriptors':audit_descriptors},
            ['calibrate/model'], watched + [d['profile_key'] for d in audit_descriptors])
        add('analysis', 'analysis/tables', 'analysis', {'descriptors':descriptors, 'null_descriptors':null_descriptors},
            ['judges/all','prepare/cohort','prepare/roles','calibrate/model'], watched+[d['profile_key'] for d in null_descriptors])
        add('report', 'report/artifacts', 'report', required=['analysis/tables'])
        return descriptors, null_descriptors, plan

    def _execute(self, job):
        return worker(self.cfg, job['key'], job['kind'], job['args'], job['required'], job['watched'])

    def run_stage(self, stage: str) -> dict:
        if stage not in STAGES: raise ValueError(stage)
        if self.cfg.execution.dry_run or stage not in self.cfg.execution.stages:
            return {'stage':stage, 'state':'planned', 'jobs':len(self.plan[stage]), 'reason':'dry_run or disabled'}
        self.ledger.set('stage/'+stage, 'running')
        try:
            # Dependency-bearing data/context jobs run first. Only independent profiles fan out.
            profiles = [j for j in self.plan[stage] if j['kind'] == 'profile']
            other = [j for j in self.plan[stage] if j['kind'] != 'profile']
            for job in other: self._execute(job)
            if self.cfg.execution.workers > 1 and len(profiles) > 1:
                with ProcessPoolExecutor(max_workers=self.cfg.execution.workers, mp_context=multiprocessing.get_context('spawn')) as pool:
                    futures = [pool.submit(worker, self.cfg, j['key'], j['kind'], j['args'], j['required'], j['watched']) for j in profiles]
                    for future in futures: future.result()
            else:
                for job in profiles: self._execute(job)
            incomplete = [j['key'] for j in self.plan[stage] if self.ledger.state(j['key'])['state'] != 'completed']
            state = 'unavailable' if incomplete else 'completed'
            self.ledger.set('stage/'+stage, state, incomplete_jobs=incomplete)
            return {'stage':stage, 'state':state, 'jobs':len(self.plan[stage]), 'incomplete':len(incomplete)}
        except BaseException:
            self.ledger.set('stage/'+stage, 'failed', traceback=traceback.format_exc())
            self.ledger.append('failures.jsonl', {'stage':stage, 'traceback':traceback.format_exc()})
            raise

    def completion_audit(self) -> dict:
        counts = {s:0 for s in ('planned','running','completed','failed','unavailable')}
        stages, missing, unavailable_analyses = {}, [], []
        for stage, jobs in self.plan.items():
            summary = {s:0 for s in counts}
            for j in jobs:
                record = self.ledger.state(j['key']); state = record['state']
                if state == 'completed':
                    result = self.ledger.result(j['key'])
                    if j['kind'] in ('profile','alias'):
                        profile = read_json(result['profile_path'])
                        for dimension, value in profile['v04']['dimensions'].items():
                            if value['status'] == 'NotEvaluated':
                                unavailable_analyses.append({'job':j['key'], 'dimension':dimension, 'metric':None,
                                                             'reason':'Dimension has missing required evidence.'})
                            for name, metric in value['metrics'].items():
                                if metric.get('status') == 'NotEvaluated':
                                    unavailable_analyses.append({'job':j['key'], 'dimension':dimension, 'metric':name,
                                                                 'required':metric.get('required'), 'reason':metric.get('note','')})
                            unavailable_analyses.extend({'job':j['key'], 'dimension':dimension, 'metric':name,
                                                        'required':True, 'reason':'Reference wrapper found missing evidence.'}
                                                       for name in value.get('missing_required_evidence', []))
                summary[state] += 1; counts[state] += 1
                if state != 'completed': missing.append({'key':j['key'], 'state':state, 'reason':record.get('reason','')})
            stages[stage] = {'expected':len(jobs), **summary}
        audit = {'time':utc(), 'scientific_hash':self.cfg.fingerprint, 'smoke_only':self.cfg.execution.smoke_mode,
                 'stages':stages, **counts, 'incomplete':missing, 'unavailable_analyses':unavailable_analyses,
                 'interpretation':'Execution coverage is not scientific acceptance; inspect NotEvaluated metrics inside completed profiles.'}
        write_json(self.cfg.root / 'audits' / (utc().replace(':','-')+'.json'), audit)
        return audit


def self_test() -> None:
    assert STAGES[0] == 'prepare' and STAGES[-1] == 'report'
''')

from syntrustbench_campaign import campaign

campaign.self_test()


## Freeze runtime provenance and instantiate the campaign

**Inputs:** the configuration and installed library. **Outputs:** `numerical_environment.json`, immutable implementation hashes, configuration, per-session manifest and expected job plan. **Success criterion:** installed v0.3 sources match the checkout byte for byte, numerical pins match, and any resumed campaign has unchanged numerical versions and implementation.

Material choices are recorded in [JUDGEMENT_CALLS.md](JUDGEMENT_CALLS.md). In particular: primary comparisons use a size-matched floor and candidate, while privacy members are the full generator-training T; positive replicates reuse the fixed P pool; null roles are smaller than the main experiment; raw provisional reference results are saved alongside the calibration wrappers. Missing calibrated signals block Pass. Outcome collapse remains a utility failure. Equity and robustness are reported as diagnostics only.

In [ ]:
from dataclasses import replace
from syntrustbench_campaign.config import from_mapping
from syntrustbench_campaign.environment import Environment
from syntrustbench_campaign.campaign import Campaign

CFG = from_mapping(SETTINGS)
# Installation already ran in the setup cell; verification here never reinstalls.
runtime_manifest = Environment(replace(CFG, execution=replace(CFG.execution, install_dependencies=False))).setup()
experiment = Campaign(CFG)
print({"campaign": str(CFG.root), "scientific_hash": CFG.fingerprint,
       "jobs": {stage: len(jobs) for stage, jobs in experiment.plan.items()},
       "dry_run": CFG.execution.dry_run, "smoke_mode": CFG.execution.smoke_mode})


## Stage: `prepare`

**Inputs:** Raw CSV or verified frozen splits and hashed patient audit.

**Outputs:** Cohort flow, missingness, subgroup counts, T/C/E splits, R/P/floor roles and disjointness audit.

**Success criterion:** Earliest encounter rule and stratification hold; no identifiers enter analytic tables.

In [ ]:
_stage_result = experiment.run_stage('prepare')
print(f"prepare: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `smoke`

**Inputs:** Prepared development data and the embedded artificial instrument fixture.

**Outputs:** Assertion results and, for full mode, a separate complete small campaign using T+C only.

**Success criterion:** Every small-campaign stage runs; expected evidence gaps are recorded. No main E rows enter preflight.

In [ ]:
_stage_result = experiment.run_stage('smoke')
print(f"smoke: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `null`

**Inputs:** Development T+C and a completed smoke preflight.

**Outputs:** Independent-within-run pseudo-roles, cached contexts and all 100 real-vs-real profiles (5 in smoke mode), drawn with fresh v0.4.1 seeds.

**Success criterion:** Every repetition is patient-disjoint within itself, size matched, and excludes main E.

In [ ]:
_stage_result = experiment.run_stage('null')
print(f"null: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `calibrate`

**Inputs:** Only the prespecified fit-run profiles (50 full; 3 smoke).

**Outputs:** Frozen oriented fit-run signals for the gate family (fidelity, utility, privacy) and the diagnostic family (equity, robustness), full-conformal levels at 0.025/0.05/0.10, descriptive per-signal thresholds, excluded-signal audit and SHA-256 freeze record.

**Success criterion:** Audit runs never affect the frozen model; missing calibration signals cannot permit Pass. (In smoke mode three fit runs cannot reject at these levels; that is recorded, not hidden.)

In [ ]:
_stage_result = experiment.run_stage('calibrate')
print(f"calibrate: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `clean`

**Inputs:** Frozen calibration, T for generation, R/floor/E for comparison.

**Outputs:** GC/CTGAN/TVAE/DPAIM × seeds 17/29/43 (optional DPMST/TabDDPM), matched candidates, eight official files per profile, v0.4 evidence and native SDMetrics scores.

**Success criterion:** Every scored table cites the frozen calibration hash; original models and data roles are recorded.

In [ ]:
_stage_result = experiment.run_stage('clean')
print(f"clean: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `controls`

**Inputs:** Frozen calibration, fixed P pool and full generator-training T.

**Outputs:** Positive, copy and independent-column controls, with expected truth stored separately from observed verdicts.

**Success criterion:** All requested replicates are scored. Expected passes/failures are audited, never forced.

In [ ]:
_stage_result = experiment.run_stage('controls')
print(f"controls: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `injections`

**Inputs:** Completed positive controls and Gaussian Copula baselines.

**Outputs:** Four nested injection families at dose zero and every specified nonzero dose; selection/change audits and all profiles.

**Success criterion:** Dose-zero bytes match and reuse baseline evidence; changed-cell sets are nested; unrealized changes are not labeled known-bad.

In [ ]:
_stage_result = experiment.run_stage('injections')
print(f"injections: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `judges`

**Inputs:** Identical saved evidence for every planned table and every held-out audit run, and the frozen score normalization.

**Outputs:** Gate, mean, geometric, harmonic and native SDMetrics results; matched-cost accept/reject decisions for every judge (same fit runs, same conformal level); positive-control τ* records.

**Success criterion:** Three-dimension scores never omit a missing dimension; native SDMetrics omission behavior is explicitly audited.

In [ ]:
_stage_result = experiment.run_stage('judges')
print(f"judges: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `analysis`

**Inputs:** Verified saved profiles, calibration model and judge evidence only.

**Outputs:** Gate admission table; known-bad acceptance by judge (headline set: real data with one injected flaw plus the copy control) and honest-data acceptance on held-out audit runs, under the primary matched-cost scheme and the secondary τ* scheme; acceptance by injection family and dose; ranking inversions; cohort/splits; clean profiles; controls; minimum detected doses; selectivity; audit Wilson intervals; threshold sensitivity; judge provenance.

**Success criterion:** No fitting occurs; denominators and unavailable comparisons remain explicit.

In [ ]:
_stage_result = experiment.run_stage('analysis')
print(f"analysis: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")


## Stage: `report`

**Inputs:** Saved analysis tables only.

**Outputs:** Status heatmap, injection response/selectivity heatmap, judge comparison (PNG/PDF), and linked Markdown report.

**Success criterion:** Missing cells/bars are labeled NA; intended injection dimensions are outlined; outputs trace to saved evidence.

In [ ]:
_stage_result = experiment.run_stage('report')
print(f"report: {_stage_result['state']} | planned jobs={_stage_result['jobs']}")
if experiment.ledger.state("report/artifacts")["state"] == "completed":
    from IPython.display import display, Image, Markdown
    report_result = experiment.ledger.result("report/artifacts")
    display(Markdown(f"Saved report: [{Path(report_result['report']).name}]({report_result['report']})"))
    for name, artifact in report_result["figures"].items():
        if artifact["status"] == "Available":
            display(Image(filename=artifact["png"]))
        else:
            print(name, "Unavailable:", artifact["reason"])


## Completion audit

**Inputs:** expected job plan, saved job states and artifact hashes. **Outputs:** dated JSON audit under the campaign's `audits/` directory, including job failures/unavailability and missing metric-level analyses within completed profiles. **Success criterion:** hashes verify, and expected/completed/failed/unavailable counts reconcile. Run this cell even after an interrupted stage; a changed artifact raises an error rather than being reused.

In [ ]:
completion = experiment.completion_audit()
print({key: completion[key] for key in ["completed", "planned", "running", "failed", "unavailable"]})
print("Recorded unavailable metric/dimension analyses:", len(completion["unavailable_analyses"]))


## Interpretation boundary

Execution coverage is not scientific acceptance. Read the report's headline section first: (1) the gate admission table, which must show that every declared gating dimension and the gate itself meet the prespecified rule on held-out honest runs; then (2) known-bad acceptance against honest-data acceptance for every judge. If (1) fails, report it as a finding and do not re-tune thresholds. Smoke-mode runs are engineering checks only.

The calibration guarantee is exact and marginal only if honest tables are exchangeable with the fit runs; null roles are smaller than main roles and reuse development patients, so the held-out audit is the empirical check of that assumption. Reference detection uncertainty omits classifier-refit variation. Native categorical models pool levels beyond 250. Positive replicates reuse the fixed P pool. Privacy is a weak empirical lower bound, not a privacy guarantee. No clinical certification is implied. The evidence comes from one public clinical cohort and one prediction task.

Equity and robustness are reported as diagnostics with their v0.4 rules and their own calibration family; they never change the gate. Frozen v0.3 is reported unmodified as the uncalibrated five-dimension gate; its exact-row refusals are recorded as NotEvaluated. Native SDMetrics is a fidelity-only headline, run as shipped, with its omitted properties reported.

The generator formal-privacy table records per-fit nominal ε/δ accounting and its assumptions. It is distinct from the empirical privacy dimension. Multiple DP fits compose; no ε = 1 guarantee is claimed for releasing the whole multi-seed campaign.